# [개별기업] 뉴스 수집 + ESG 분석 파이프라인

**셀을 위에서 아래로 순서대로 실행하면 자동으로 완료됩니다.**

| 섹션 | 내용 |
|---|---|
| 1 | 설정 (이 셀만 수정) |
| 2 | 설치 + 초기화 |
| 3 | DB 초기화 (처음 or 재수집 시) |
| 4 | 헤드라인 수집 + GPT 필터링 → 시각화 |
| 5 | 전체 본문 수집 → 시각화 |
| 6 | 키워드 추출 + 임베딩 → 시각화 |
| 7 | 감성 분석 → 시각화 |
| 8 | ESG 이슈 분류 → 시각화 |
| 9 | ESG 감성지수 + 주가 회귀분석 |
| 10 | 주제분석 + 네트워크 + HTML 스토리북 |

> ⚠️ **실행 전 선행 작업 (섹션9 필요)**
> 섹션9(회귀분석)는 주가·재무 데이터를 사용합니다. 이 노트북을 실행하기 전에 아래 두 노트북을 먼저 돌려 `cha_stock` DB에 데이터가 채워져 있어야 합니다.
> - **Company_01_Stock** → `tbl_stock_price` 테이블 생성/채움
> - **Company_02_Financial** → `tbl_financial` 테이블 생성/채움
>
> 이 세 노트북(01/02/03)은 모두 같은 `cha_stock` DB를 공유합니다.
>
> | 용도 | DB | 테이블 | 출처 |
> |---|---|---|---|
> | 주가 | `cha_stock` | `tbl_stock_price` | Company_01 |
> | 재무 | `cha_stock` | `tbl_financial` | Company_02 |
> | 뉴스 | `cha_stock` | `NEWS_RAW_<CODE>`, `NEWS_SENTIMENT_<CODE>`, `NEWS_ESG_CLASS_<CODE>` 등 | Company_03 섹션 1~8 |


---
# ⚙️ 섹션 0 : 분석가 설정 (이 셀 하나만 수정하면 됩니다)

> **세 노트북(ESG_01 · ESG_02 · ESG_03)에서 아래 설정 셀은 글자 하나까지 똑같습니다.**
> 한 번만 값을 채운 뒤, 같은 내용을 세 노트북에 그대로 붙여넣으세요.
> 다른 셀은 건드릴 필요가 없습니다. 위에서 아래로 실행만 하면 됩니다.

| 무엇을 정해야 하나 | 변수 | 쓰는 곳 |
|---|---|---|
| 분석할 기업 | `STOCK_CODE` · `COMPANY_NAME` · `SEARCH_KEYWORD` · `TOPIC_CODE` | 01·02·03 공통 |
| DB 접속 | `DB_*` | 01·02·03 공통 |
| 주가 차트/시뮬레이션 | `STOCK_START_DATE` · `CHART_YEARS` · `BUY_DATE` · `INVEST_AMT` | 01 |
| 뉴스 기간·옵션 | `START_DATE` · `END_DATE` · `PHOTO_TYPE` · `REG_START_OFFSET` | 03 |
| 외부 키 | `OPENAI_API_KEY` · `GITHUB_TOKEN` (Colab 🔑 비밀) | 03 |


In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║                 ⚙️  분 석 가  설 정   (ANALYST CONFIG)              ║
# ║   이 셀은 ESG_01 · ESG_02 · ESG_03 에서 100% 동일합니다.            ║
# ║   여기 적힌 값만 바꾸면 됩니다. 아래 다른 셀은 수정하지 마세요.     ║
# ║   ⚠️ 비밀번호·API키는 코드에 적지 않습니다 → 아래 (5) 비밀 참고     ║
# ╚════════════════════════════════════════════════════════════════════╝
import os as _os

def _secret(name, default=''):
    """비밀값을 Colab 🔑(비밀) → 환경변수 순서로 읽음. 코드에 평문 저장 안 함."""
    try:
        from google.colab import userdata as _ud
        v = _ud.get(name)
        if v:
            return v
    except Exception:
        pass
    return _os.environ.get(name, default)

# ┌─ 1) 분석 대상 기업  (01·02·03 공통) ───────────────────────────────┐
STOCK_CODE     = '051900'         # 종목코드 6자리
COMPANY_NAME   = 'LG생활건강'     # 회사명 (그래프 제목 등에 표시)
SEARCH_KEYWORD = 'LG생활건강'     # 뉴스 검색어            (03 사용)
TOPIC_CODE     = 'LGHNH'          # 뉴스 테이블 접미사(영문) (03 사용)

# ┌─ 2) DB 접속  (01·02·03 공통) ──────────────────────────────────────┐
#    호스트/아이디/비밀번호는 코드에 적지 않습니다.
#    Colab 🔑(비밀)에 DB_HOST · DB_USER · DB_PASSWORD 를 등록하세요.
DB_HOST     = _secret('DB_HOST')
DB_USER     = _secret('DB_USER')
DB_PASSWORD = _secret('DB_PASSWORD')
DB_NAME     = 'cha_stock'         # DB 이름(비밀 아님)
TABLE_NAME  = 'tbl_stock_price'   # 01 주가 테이블
FIN_TABLE   = 'tbl_financial'     # 02 재무 테이블

# ┌─ 3) [ESG_01 주가] 전용 ────────────────────────────────────────────┐
STOCK_START_DATE = '19970101'     # 주가 수집 시작일(YYYYMMDD) · 이어받기 자동
CHART_YEARS = 3                   # 차트 기본 표시 기간(년)
MA_PERIODS  = [5, 20, 60, 120, 240]   # 이동평균선
BUY_DATE    = '2022-01-03'        # 투자 시뮬레이션 매수일
INVEST_AMT  = 10_000_000          # 투자 시뮬레이션 금액(원)

# ┌─ 4) [ESG_03 뉴스·ESG] 전용 ────────────────────────────────────────┐
START_DATE = '2019-01-01'         # 뉴스 수집 시작일(포함)
END_DATE   = '2026-06-12'         # 뉴스 수집 종료일(포함)
PHOTO_TYPE = 3                    # 0=전체 1=포토 2=동영상 3=지면 4=보도자료
REG_START_OFFSET = 0              # 회귀 시작연도: 0=전체 / +N=앞N년 제외 / -N=자동보정

MAX_WORKERS          = 5          # 동시 처리 스레드 수
SIMILARITY_THRESHOLD = 0.4        # ESG 이슈 매칭 코사인 임계값
ESG_TABLE            = 'ESG_ISSUES_EMBED'
TOP_N_ISSUES         = 5          # 상위 ESG 이슈 개수
MOVING_AVG_WINDOW    = 12         # 이동평균 개월
MAX_LAG_MONTHS       = 12         # 시차 회귀 최대 개월
N_CLUSTERS           = 4          # 주제 군집 수
MIN_ARTICLES         = 10
TARGET_ESG_ISSUES    = []

# ┌─ 5) 외부 API 키 / 비밀  (03에서만 필요 · 01/02는 무시) ────────────┐
#    Colab: 왼쪽 🔑(비밀) 메뉴에 아래 이름으로 등록하세요.
#      OPENAI_API_KEY · GITHUB_TOKEN · DB_HOST · DB_USER · DB_PASSWORD
OPENAI_KEY   = _secret('OPENAI_API_KEY')
GITHUB_TOKEN = _secret('GITHUB_TOKEN')

GITHUB_USERNAME = 'sdkparkforbi'  # ← 본인 GitHub 계정으로 변경 (03 HTML 업로드용)
GITHUB_REPO     = 'STOCK_ANALYZE' # 바꾸지 말 것
GITHUB_BRANCH   = 'main'          # 바꾸지 말 것
GITHUB_FOLDER   = 'storybooks'    # 바꾸지 말 것

print('⚙️ 분석가 설정 로드 완료:', COMPANY_NAME, '('+STOCK_CODE+')')
if not (DB_HOST and DB_USER and DB_PASSWORD):
    print('   ⚠️ DB 비밀 미설정 — Colab 🔑 비밀에 DB_HOST/DB_USER/DB_PASSWORD 등록 필요')


---
# 섹션 2 : 설치 및 초기화

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀  4]  섹션 2-1 : 라이브러리 설치')
print('=' * 70)

!pip install -q pymysql sqlalchemy koreanize-matplotlib lxml python-dateutil openai scikit-learn networkx PyGithub
print('설치 완료!')


In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀  5]  섹션 2-2 : matplotlib 설정')
print('=' * 70)

%matplotlib inline

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀  6]  섹션 2-3 : 라이브러리 임포트 + DB 테이블 생성')
print('=' * 70)

import ast, base64, json, math, os, random, re, time, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from io import BytesIO
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import pymysql, requests
from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta
from openai import OpenAI
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text
from tqdm import tqdm
import networkx as nx
import koreanize_matplotlib
warnings.filterwarnings('ignore')

client = OpenAI(api_key=OPENAI_KEY)

START_DATE = pd.Timestamp(START_DATE).strftime('%Y-%m-%d')
END_DATE   = pd.Timestamp(END_DATE).strftime('%Y-%m-%d')

T_RAW  = 'NEWS_RAW_'          + TOPIC_CODE
T_ALL  = 'NEWS_RAW_ALL_'      + TOPIC_CODE
T_EMBD = 'REVIEW_NEWS_EMBED_' + TOPIC_CODE
T_SENT = 'NEWS_SENTIMENT_'    + TOPIC_CODE
T_ESG  = 'NEWS_ESG_CLASS_'    + TOPIC_CODE

PHOTO_LABELS = {0:'전체',1:'포토',2:'동영상',3:'지면기사',4:'보도자료'}

PALETTE = {
    'blue':'#3B82F6','green':'#10B981','red':'#EF4444',
    'amber':'#F59E0B','purple':'#8B5CF6','teal':'#14B8A6',
    'pink':'#EC4899','gray':'#6B7280','pos':'#10B981','neg':'#EF4444','neu':'#9CA3AF',
}

# DB 자동 생성
_tmp = pymysql.connect(host=DB_HOST,user=DB_USER,password=DB_PASSWORD,
                        charset='utf8mb4',autocommit=True)
_tmp.cursor().execute('CREATE DATABASE IF NOT EXISTS `'+DB_NAME+'`'
                       ' DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci')
_tmp.close()

def make_engine(db=DB_NAME):
    return create_engine('mysql+pymysql://'+DB_USER+':'+DB_PASSWORD
                         +'@'+DB_HOST+'/'+db+'?charset=utf8mb4')
engine = make_engine()

# 모든 테이블 미리 생성
with engine.begin() as conn:
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+T_RAW+' ('
        '  topic_code VARCHAR(100) NOT NULL, search_keyword VARCHAR(100) NOT NULL,'
        '  publish_date VARCHAR(10) NOT NULL, press_name VARCHAR(100) NOT NULL,'
        '  press_url VARCHAR(500), naver_url VARCHAR(500) NOT NULL,'
        '  title VARCHAR(500), content_preview MEDIUMTEXT,'
        '  is_related TINYINT(1) NOT NULL, filter_reason VARCHAR(500),'
        '  UNIQUE KEY uq (topic_code(30),search_keyword(30),publish_date,press_name(30),naver_url(150))'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'))
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+T_ALL+' ('
        '  topic_code VARCHAR(100) NOT NULL, search_keyword VARCHAR(100) NOT NULL,'
        '  publish_date VARCHAR(10) NOT NULL, press_name VARCHAR(100) NOT NULL,'
        '  press_url VARCHAR(500), naver_url VARCHAR(500) NOT NULL,'
        '  title VARCHAR(500), contents MEDIUMTEXT,'
        '  PRIMARY KEY (topic_code(30),search_keyword(30),publish_date,press_name(30),naver_url(150))'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'))
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+T_EMBD+' ('
        '  code VARCHAR(100) NOT NULL, query VARCHAR(100) NOT NULL,'
        '  date VARCHAR(10) NOT NULL, press VARCHAR(100) NOT NULL,'
        '  url VARCHAR(500) NOT NULL, title TEXT, contents MEDIUMTEXT,'
        '  Keyword_JSON JSON, Embedding_Input TEXT, Embedding_Vector JSON,'
        '  UNIQUE KEY uq (code(30),query(30),date,press(30),url(150))'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'))
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+T_SENT+' ('
        '  code VARCHAR(100) NOT NULL, query VARCHAR(255) NOT NULL,'
        '  date VARCHAR(10) NOT NULL, press VARCHAR(100) NOT NULL,'
        '  url VARCHAR(500) NOT NULL, title TEXT, contents MEDIUMTEXT,'
        '  Keyword_JSON TEXT, Embedding_Input TEXT, Embedding_Vector LONGTEXT,'
        '  sentiment VARCHAR(10),'
        '  PRIMARY KEY (code(30),query(30),date,press(30),url(150))'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'))
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+T_ESG+' ('
        '  code VARCHAR(50) NOT NULL, query VARCHAR(50) NOT NULL,'
        '  date VARCHAR(10) NOT NULL, press VARCHAR(50),'
        '  url TEXT, esg_category VARCHAR(50), esg_issue VARCHAR(150), similarity FLOAT,'
        '  UNIQUE KEY uq_esg (code(20),query(20),date,press(20),url(100),esg_category(20),esg_issue(50))'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'))

print('초기화 완료!')
print('  기업    :', COMPANY_NAME, '('+SEARCH_KEYWORD+')')
print('  수집기간:', START_DATE, '~', END_DATE)
print('  기사유형:', PHOTO_LABELS.get(PHOTO_TYPE))
print('  테이블  :', T_RAW)
print('  API 키  :', '설정 완료' if OPENAI_KEY else '미설정!')


---
# 섹션 3 : DB 초기화

> 처음 실행하거나 처음부터 다시 수집하고 싶을 때만 실행하세요.
> 이미 수집된 데이터가 있고 이어받기를 원하면 **건너뛰세요**.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 🔍 5개 테이블 간단 확인
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
from sqlalchemy import text

TABLES = [T_RAW, T_ALL, T_EMBD, T_SENT, T_ESG]

print(f'{"테이블":<30} {"행수":>10} {"컬럼수":>6}')
print('-' * 50)

for tbl in TABLES:
    try:
        with engine.connect() as conn:
            rows = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).scalar()
            cols = conn.execute(text(
                f"SELECT COUNT(*) FROM information_schema.columns "
                f"WHERE table_schema=DATABASE() AND table_name='{tbl}'"
            )).scalar()
        print(f'{tbl:<30} {rows:>10,} {cols:>6}')
    except Exception as e:
        print(f'{tbl:<30} ❌ {e}')

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀  8]  섹션 3 : DB 초기화 (선택)')
print('=' * 70)

# ⚠️ 주의: 실행하면 이 기업의 모든 수집 데이터가 삭제됩니다!
RESET_CONFIRM = False   # False 로 바꾸면 실행 안 됨

if RESET_CONFIRM:
    with engine.begin() as conn:
        for tbl, col in [(T_RAW,'topic_code'),(T_ALL,'topic_code'),
                          (T_EMBD,'code'),(T_SENT,'code'),(T_ESG,'code')]:
            try:
                r = conn.execute(text('DELETE FROM '+tbl+' WHERE '+col+'=:v'),{'v':TOPIC_CODE})
                print('  삭제:', tbl, r.rowcount, '건')
            except: print('  없음:', tbl)
    print('\n초기화 완료! 섹션 4부터 실행하세요.')
else:
    print('초기화 건너뜀 (RESET_CONFIRM=False)')


---
# 섹션 4 : 헤드라인 수집 + GPT 필터링

> 수집 → GPT 필터링 → DB 저장 → 시각화 까지 한 번에 실행됩니다.


In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 10]  섹션 4 : 헤드라인 수집 + GPT 필터링')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 섹션 4 (개선판): 헤드라인 수집 + GPT 필터링
# ═══════════════════════════════════════════════════════════════════
# 적용된 개선 (가이드 1+2단계)
#   1) 시간 정책: 페이지 사이 3~8초, 월 사이 15~30초, 10% 확률 긴 휴식
#   2) 차단 감지: HTTP 코드, 본문 키워드, 컨테이너 부재 검사
#   3) Exponential Backoff: 60 * 2^n 초 대기 (최대 600초)
#   4) 세션 일관성: session.get() 일관 사용, UA 세션 시작 시 한 번만 정함
#   5) Sec-Ch-Ua / Sec-Fetch-* 헤더 추가
#   6) Referer 체인: 직전 URL을 자동으로 Referer로 전달
#   7) 차단으로 누락된 달은 blocked_months 리스트에 기록
# ═══════════════════════════════════════════════════════════════════

# GPT 필터링 프롬프트 (기업에 맞게 수정)
FILTERING_PROMPT = """
다음 뉴스가 {SEARCH_KEYWORD}과 실질적으로 관련이 있는지 판단해주세요.

제목: {title}
본문 미리보기: {content}

[관련 있음]
1. 실적/전략: 매출, 영업이익, 신사업, 중장기 전략
2. 제품/브랜드: 신제품, 브랜드 캠페인, 판매 실적
3. 인사/경영: 대표이사, 임원 인사, 조직 개편
4. ESG/사회공헌: 환경, 사회책임, 지배구조
5. M&A/투자: 인수합병, 지분 투자, 협력사
6. 주가/투자: 주가 분석, 목표주가, 투자의견
7. 연구개발: 신기술, 특허, 제품 혁신
8. 논란/리스크: 품질 이슈, 규제, 소송

[관련 없음]
1. 동종업계 타사 기사
2. 단순 비교 대상으로만 언급
3. 광고성 홍보 기사

JSON 형식으로만 답변하세요:
{{
    "is_related": true/false,
    "reason": "판단 이유를 한 줄로"
}}
""".replace('{SEARCH_KEYWORD}', SEARCH_KEYWORD)

# ── UA 풀 확장 (Desktop / Mobile / Linux 혼합) ───────────────────
UA_POOL = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36 Edg/121.0.0.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 14_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
]

# Sec-Ch-Ua 값 (UA의 Chrome 버전과 일치시켜야 함 - 기본 122 사용)
SEC_CH_UA = '"Chromium";v="122", "Google Chrome";v="122", "Not(A:Brand";v="24"'

# 차단 페이지 식별 키워드
BLOCK_PATTERNS = ['비정상적인 접근', '잠시 후 다시', '캡차', 'captcha']
BLOCK_PATTERNS_LOWER = ['unusual traffic', 'access denied', 'are you a robot']

# ── GPT 필터링 함수 (기존 유지) ──────────────────────────────────
def check_relevance_with_gpt(content_preview, article_title, filtering_prompt):
    system_prompt = filtering_prompt.format(title=article_title, content=content_preview)
    try:
        resp = requests.post(
            'https://api.openai.com/v1/chat/completions',
            headers={'Authorization': 'Bearer ' + OPENAI_KEY, 'Content-Type': 'application/json'},
            json={'model': 'gpt-4o-mini-2024-07-18',
                  'messages': [
                      {'role': 'system', 'content': '당신은 뉴스 분류 전문가입니다. 정확하고 일관된 판단을 제공합니다.'},
                      {'role': 'user', 'content': system_prompt}],
                  'temperature': 0.0, 'max_tokens': 150},
            timeout=30)
        content = resp.json()['choices'][0]['message']['content'].strip()
        try:    return json.loads(content)
        except: return {'is_related': False, 'reason': '파싱 실패'}
    except Exception as e:
        return {'is_related': False, 'reason': f'API 오류: {e}'}

def check_relevance_batch(articles, filtering_prompt, max_workers):
    results = {}
    def process_single(article):
        result = check_relevance_with_gpt(
            article['content'], article['title'], filtering_prompt)
        return article['idx'], result
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_article = {executor.submit(process_single, a): a for a in articles}
        for future in as_completed(future_to_article):
            try:
                idx, result = future.result()
                results[idx] = result
            except Exception as e:
                a = future_to_article[future]
                results[a['idx']] = {'is_related': False, 'reason': f'오류:{e}'}
    return results

# ═══════════════════════════════════════════════════════════════════
# 사람처럼 보이는 세션 + 차단 감지 함수
# ═══════════════════════════════════════════════════════════════════

def make_session():
    """사람처럼 보이는 세션 생성. UA는 세션 시작 시 한 번만 정함."""
    s = requests.Session()
    ua = random.choice(UA_POOL)
    s.headers.update({
        'User-Agent': ua,
        # 한국 사이트 - 한국어 우선
        'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept': ('text/html,application/xhtml+xml,application/xml;q=0.9,'
                   'image/avif,image/webp,*/*;q=0.8'),
        # Chrome이 자동으로 붙이는 보안 헤더
        'Sec-Ch-Ua': SEC_CH_UA,
        'Sec-Ch-Ua-Mobile': '?0',
        'Sec-Ch-Ua-Platform': '"Windows"',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-User': '?1',
        'Upgrade-Insecure-Requests': '1',
        'Connection': 'keep-alive',
    })
    return s

def warmup_session(s):
    """입장 동선: naver.com → search.naver.com (쿠키 발급 + 자연스러운 동선)"""
    try:
        s.get('https://www.naver.com/', timeout=10)
        time.sleep(random.uniform(2, 4))
        s.get('https://search.naver.com/', timeout=10,
              headers={'Referer': 'https://www.naver.com/'})
        time.sleep(random.uniform(2, 4))
    except Exception as e:
        print('  [경고] 워밍업 실패:', e)

def fetch_naver(session, url, referer=None, max_retries=5):
    """
    네이버 검색 페이지 안전 호출.
    반환: (soup, status)
      status: 'ok'      - 정상 + 기사 있음
              'empty'   - 정상이지만 기사 0
              'blocked' - 재시도 모두 실패 (호출자가 backoff/세션 재생성 결정)
    """
    for attempt in range(max_retries):
        try:
            extra = {'Referer': referer} if referer else {}
            resp = session.get(url, headers=extra, timeout=30)

            # (1) HTTP 상태 코드 차단 감지
            if resp.status_code in (429, 403, 503, 502):
                wait = min(60 * (2 ** attempt) + random.uniform(0, 30), 600)
                print('    [차단] HTTP', resp.status_code,
                      '(시도', attempt + 1, ') →', round(wait), '초 대기')
                time.sleep(wait)
                continue

            # (2) 차단 페이지 본문 키워드 감지
            body = resp.text[:8000]
            body_lower = body.lower()
            if (any(p in body for p in BLOCK_PATTERNS) or
                any(p in body_lower for p in BLOCK_PATTERNS_LOWER)):
                wait = min(60 * (2 ** attempt) + random.uniform(0, 30), 600)
                print('    [차단] 안내 페이지 (시도', attempt + 1, ') →',
                      round(wait), '초 대기')
                time.sleep(wait)
                continue

            soup = BeautifulSoup(resp.content, 'lxml')

            # (3) 검색 결과 컨테이너 부재 = 비정상
            if (not soup.select_one('#main_pack')
                    and not soup.select_one('.api_subject_bx')):
                wait = 30 + random.uniform(0, 15)
                print('    [경고] 결과 컨테이너 없음 (시도', attempt + 1,
                      ') →', round(wait), '초 대기')
                time.sleep(wait)
                continue

            # (4) 정상
            title_links = soup.select('a[nocr="1"][data-heatmap-target=".tit"]')
            return soup, ('ok' if title_links else 'empty')

        except requests.exceptions.RequestException as e:
            wait = 10 + random.uniform(0, 10)
            print('    [경고] 네트워크 오류:', e, '→', round(wait), '초 대기')
            time.sleep(wait)

    return None, 'blocked'

# ═══════════════════════════════════════════════════════════════════
# 한 달치 수집 (메인 루프와 회수 셀이 공유하는 함수)
# ═══════════════════════════════════════════════════════════════════
def collect_one_month(session, ds, de, end_ts, prev_url):
    """
    한 달치 헤드라인 수집 + 필터링 + DB 저장.
    반환: (month_cnt, month_rel, blocked, last_url, session)
      blocked: True면 그 달에 2회 이상 차단 발생 → 호출자가 blocked_months 기록
      session: 차단 시 새로 발급되어 반환될 수 있음
    """
    page_start = 1
    empty_pages = 0
    blocked_pages = 0
    month_cnt = 0
    month_rel = 0
    blocked = False

    while True:
        kw_enc = requests.utils.quote(SEARCH_KEYWORD)
        url = ('https://search.naver.com/search.naver?where=news'
               '&query=%22' + kw_enc + '%22&sm=tab_opt'
               '&sort=2&photo=' + str(PHOTO_TYPE) + '&field=0&pd=3'
               '&ds=' + ds + '&de=' + de + '&start=' + str(page_start)
               + '&refresh_start=0&docid=&related=0'
               '&mynews=1&office_type=1&office_section_code=1'
               '&nso=so%3Add%2Cp%3Aall&is_sug_officeid=0')

        soup, status = fetch_naver(session, url, referer=prev_url)

        # ── 차단 처리 ────────────────────────────────────────────
        if status == 'blocked':
            blocked_pages += 1
            if blocked_pages >= 2:
                blocked = True
                print('    [차단확정] 5분 휴식 + 새 세션')
                time.sleep(300)
                session = make_session()
                warmup_session(session)
                break
            print('    [차단1차] 2분 휴식 후 재시도')
            time.sleep(120 + random.uniform(0, 60))
            continue

        # ── 빈 결과 처리 ────────────────────────────────────────
        if status == 'empty':
            empty_pages += 1
            if empty_pages >= 2:
                break
            time.sleep(random.uniform(5, 10))
            page_start += 10
            continue

        # ── 정상 응답 - 파싱 ────────────────────────────────────
        empty_pages = 0
        prev_url = url

        title_links   = soup.select('a[nocr="1"][data-heatmap-target=".tit"]')
        content_links = soup.select('a[nocr="1"][data-heatmap-target=".body"]')
        nav_links     = soup.select('a[data-heatmap-target=".nav"]')
        press_tags    = soup.select('.sds-comps-profile-info-title-text')
        all_subtexts  = soup.select('[class*="subtext"]')
        page_articles = []

        for idx in range(len(title_links)):
            if idx >= len(content_links): continue
            naver_url = nav_links[idx]['href'] if idx < len(nav_links) else None
            if not naver_url: continue
            press_name = press_tags[idx].get_text(strip=True) if idx < len(press_tags) else '언론사없음'
            press_url  = title_links[idx].get('href', '')
            date_idx   = idx * 4 + 2
            raw_date   = ''
            if date_idx < len(all_subtexts):
                txt = all_subtexts[date_idx].get_text(strip=True)
                if re.search(r'\d{4}\.\d{2}\.\d{2}', txt) or re.search(r'\d+(분|시간|일|초|주) 전', txt):
                    raw_date = txt

            rel = re.search(r'(\d+)(초|분|시간|일|주) 전', raw_date)
            if rel:
                nv, u = int(rel.group(1)), rel.group(2)
                if u in ('초', '분', '시간'):
                    pub = end_ts.strftime('%Y%m%d')
                elif u == '일':
                    pub = (end_ts - pd.DateOffset(days=nv)).strftime('%Y%m%d')
                else:
                    pub = (end_ts - pd.DateOffset(weeks=nv)).strftime('%Y%m%d')
            else:
                m = re.search(r'(20\d{2})\.(\d{1,2})\.(\d{1,2})', raw_date)
                if m:
                    yyyy, mm, dd = m.group(1), m.group(2).zfill(2), m.group(3).zfill(2)
                    pub = yyyy + mm + dd
                else:
                    pub = end_ts.strftime('%Y%m%d')

            try:
                pub_dt = pd.to_datetime(pub, format='%Y%m%d')
                if pub_dt < pd.Timestamp(START_DATE) or pub_dt > pd.Timestamp(END_DATE):
                    pub = end_ts.strftime('%Y%m%d')
            except:
                pub = end_ts.strftime('%Y%m%d')

            t_span = title_links[idx].select_one('.sds-comps-text-type-headline1')
            c_span = content_links[idx].select_one('.sds-comps-text-type-body1')
            page_articles.append({
                'idx': idx,
                'title':   t_span.get_text(strip=True) if t_span else '제목없음',
                'content': c_span.get_text(strip=True) if c_span else '내용없음',
                'press_name': press_name, 'press_url': press_url,
                'naver_url': naver_url,   'publish_date': pub
            })

        # ── GPT 필터링 + DB 저장 (기존 로직) ─────────────────────
        if page_articles:
            page_no = (page_start - 1) // 10 + 1
            print('    [페이지', page_no, ']', len(page_articles), '건 받음 → GPT 필터링 중...')
            filter_results = check_relevance_batch(page_articles, FILTERING_PROMPT, MAX_WORKERS)
            with engine.begin() as conn:
                for art in page_articles:
                    month_cnt += 1
                    fr = filter_results.get(art['idx'], {'is_related': False, 'reason': ''})
                    ir = 1 if fr.get('is_related') else 0
                    month_rel += ir
                    conn.execute(text(
                        'INSERT IGNORE INTO ' + T_RAW
                        + ' (topic_code,search_keyword,publish_date,press_name,press_url,'
                        '  naver_url,title,content_preview,is_related,filter_reason)'
                        ' VALUES (:tc,:kw,:pd,:pn,:pu,:nu,:ti,:cp,:ir,:fr)'),
                        dict(tc=TOPIC_CODE, kw=SEARCH_KEYWORD, pd=art['publish_date'],
                             pn=art['press_name'], pu=art['press_url'], nu=art['naver_url'],
                             ti=art['title'], cp=art['content'], ir=ir, fr=fr.get('reason', '')))
            print('             누적: 수집', month_cnt, '건 / 관련', month_rel, '건')

        page_start += 10
        if page_start > 2000:
            break

        # ── 사람처럼: 페이지 사이 3~8초 ──────────────────────────
        time.sleep(random.uniform(3, 8))

        # ── 10% 확률로 긴 휴식 (자리 비움) ───────────────────────
        if random.random() < 0.1:
            long_pause = random.uniform(20, 60)
            print('    [휴식]', round(long_pause), '초 자리 비움')
            time.sleep(long_pause)

    return month_cnt, month_rel, blocked, prev_url, session

# ═══════════════════════════════════════════════════════════════════
# 체크포인트 확인 (기존 유지)
# ═══════════════════════════════════════════════════════════════════
with engine.connect() as conn:
    row = conn.execute(text(
        'SELECT MAX(publish_date), COUNT(*), SUM(is_related=1) FROM ' + T_RAW
        + ' WHERE topic_code=:tc AND search_keyword=:kw'
        + ' AND publish_date >= :sd AND publish_date <= :ed'),
        {'tc': TOPIC_CODE, 'kw': SEARCH_KEYWORD,
         'sd': START_DATE.replace('-', ''),
         'ed': END_DATE.replace('-', '')}).fetchone()

last_date    = row[0]
total_saved  = row[1] or 0
actual_start = START_DATE

if total_saved > 0 and last_date:
    next_d = (datetime.strptime(last_date, '%Y%m%d') + timedelta(days=1)).strftime('%Y-%m-%d')
    if next_d <= END_DATE:
        actual_start = next_d
        print('체크포인트:', actual_start, '부터 이어서 진행')
    else:
        print('이미 수집 완료 (마지막:', last_date + ')')
        actual_start = None
else:
    print('처음부터 수집합니다:', actual_start, '~', END_DATE)

# ═══════════════════════════════════════════════════════════════════
# 메인 수집 루프 (개선판)
# ═══════════════════════════════════════════════════════════════════
blocked_months = []   # 차단으로 누락된 달 (회수 셀에서 사용)

if actual_start and actual_start <= END_DATE:
    period_start  = datetime.strptime(actual_start, '%Y-%m-%d')
    period_end_dt = datetime.strptime(END_DATE,    '%Y-%m-%d')

    # ── 사람처럼 보이는 세션 시작 + 입장 동선 ──────────────────
    session  = make_session()
    warmup_session(session)
    prev_url = 'https://search.naver.com/'

    total_cnt   = 0
    related_cnt = 0
    current_month = period_start

    while current_month <= period_end_dt:
        end_of_month = min(current_month + relativedelta(months=1) - timedelta(days=1),
                           period_end_dt)
        ds = current_month.strftime('%Y.%m.%d')
        de = end_of_month.strftime('%Y.%m.%d')
        end_ts = pd.Timestamp(end_of_month)

        month_cnt, month_rel, blocked, prev_url, session = collect_one_month(
            session, ds, de, end_ts, prev_url)

        total_cnt   += month_cnt
        related_cnt += month_rel

        if blocked:
            blocked_months.append((current_month.strftime('%Y-%m-%d'),
                                   end_of_month.strftime('%Y-%m-%d')))
            print('  ', ds, '~', de, ': [차단으로 중단 - 회수 대상]')
        else:
            print('  ', ds, '~', de, ': 수집', month_cnt, '건 / 관련', month_rel, '건')

        current_month += relativedelta(months=1)

        # ── 사람처럼: 월 사이 15~30초 휴식 ─────────────────────
        time.sleep(random.uniform(15, 30))

    print('\n수집 완료! 총:', total_cnt, '건 / 관련:', related_cnt, '건')

    if blocked_months:
        print('\n[!] 차단으로 누락된 달:', len(blocked_months), '개')
        for ds_b, de_b in blocked_months:
            print('  -', ds_b, '~', de_b)
        print('→ 다음 셀(섹션 4-회수)을 실행하여 누락된 달을 재수집하세요.')

In [ ]:
# ── 시각화 ──────────────────────────────────────────────────────
df_raw = pd.read_sql('SELECT * FROM '+T_RAW,engine)
if len(df_raw)==0:
    print('수집된 데이터가 없습니다. 위 수집 코드를 확인하세요.')
else:
    df_raw['date']=pd.to_datetime(df_raw['publish_date'],format='%Y%m%d',errors='coerce')
    df_raw['ym']=df_raw['date'].dt.to_period('M').astype(str)
    total=len(df_raw); related=int(df_raw['is_related'].sum())
    unrel=total-related; pct=related/total*100 if total>0 else 0
    press_n=df_raw['press_name'].nunique(); day_n=df_raw['date'].dt.date.nunique()
    fig=plt.figure(figsize=(18,10)); gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.35)
    ax0=fig.add_subplot(gs[0,:]); ax0.axis('off')
    cards=[('총 수집',f'{total:,}건',PALETTE['blue']),
           ('관련 기사',f'{related:,}건',PALETTE['green']),
           ('무관 기사',f'{unrel:,}건',PALETTE['gray']),
           ('관련 비율',f'{pct:.1f}%',PALETTE['amber']),
           ('언론사 수',f'{press_n:,}개',PALETTE['purple']),
           ('수집 기간',f'{day_n:,}일',PALETTE['teal'])]
    for i,(label,val,color) in enumerate(cards):
        x=i/6+0.02
        ax0.add_patch(mpatches.FancyBboxPatch((x,0.1),0.14,0.8,boxstyle='round,pad=0.02',
            facecolor=color,alpha=0.15,edgecolor=color,linewidth=2,transform=ax0.transAxes))
        ax0.text(x+0.07,0.68,label,ha='center',fontsize=11,color=color,transform=ax0.transAxes,fontweight='bold')
        ax0.text(x+0.07,0.32,val,ha='center',fontsize=17,color=color,transform=ax0.transAxes,fontweight='bold')
    ax0.set_title(COMPANY_NAME+' 뉴스 수집 현황  ('+START_DATE+' ~ '+END_DATE+')',
                 fontsize=14,fontweight='bold',pad=8)
    ax1=fig.add_subplot(gs[1,0])
    monthly=df_raw.groupby('ym')['is_related'].agg(['count','sum']).reset_index()
    monthly.columns=['month','total','related']
    x=range(len(monthly))
    ax1.bar(x,monthly['total'],color=PALETTE['blue'],alpha=0.4,label='전체')
    ax1.bar(x,monthly['related'],color=PALETTE['green'],alpha=0.85,label='관련')
    ax1.set_xticks(x); ax1.set_xticklabels(monthly['month'],rotation=45,ha='right',fontsize=8)
    ax1.set_title('월별 수집 기사 수',fontsize=12,fontweight='bold'); ax1.legend(fontsize=9); ax1.grid(True,alpha=0.25,axis='y')
    ax2=fig.add_subplot(gs[1,1])
    press_top=df_raw[df_raw['is_related']==1]['press_name'].value_counts().head(10)
    ax2.barh(press_top.index[::-1],press_top.values[::-1],color=PALETTE['teal'])
    ax2.set_title('관련 기사 언론사 Top 10',fontsize=12,fontweight='bold'); ax2.grid(True,alpha=0.25,axis='x')
    ax3=fig.add_subplot(gs[1,2])
    ax3.pie([related,unrel],labels=['관련','무관'],autopct='%1.1f%%',
           colors=[PALETTE['green'],PALETTE['gray']],startangle=90)
    ax3.set_title('관련/무관 비율',fontsize=12,fontweight='bold')
    plt.suptitle(COMPANY_NAME+' 헤드라인 수집 현황',fontsize=15,fontweight='bold',y=1.01)
    plt.tight_layout(); plt.show()
    print('섹션 4 완료!')

In [ ]:
# ── 시각화 ──────────────────────────────────────────────────────
df_raw = pd.read_sql('SELECT * FROM ' + T_RAW, engine)

if len(df_raw) == 0:
    print('수집된 데이터가 없습니다. 위 수집 코드를 확인하세요.')
else:
    df_raw['date'] = pd.to_datetime(df_raw['publish_date'],
                                    format='%Y%m%d', errors='coerce')
    df_raw['ym'] = df_raw['date'].dt.to_period('M').astype(str)

    total   = len(df_raw)
    related = int(df_raw['is_related'].sum())
    unrel   = total - related
    pct     = related / total * 100 if total > 0 else 0
    press_n = df_raw['press_name'].nunique()
    day_n   = df_raw['date'].dt.date.nunique()

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── ax0: KPI 카드 ────────────────────────────────────────
    ax0 = fig.add_subplot(gs[0, :])
    ax0.axis('off')
    cards = [('총 수집',   f'{total:,}건',   PALETTE['blue']),
             ('관련 기사', f'{related:,}건', PALETTE['green']),
             ('무관 기사', f'{unrel:,}건',   PALETTE['gray']),
             ('관련 비율', f'{pct:.1f}%',    PALETTE['amber']),
             ('언론사 수', f'{press_n:,}개', PALETTE['purple']),
             ('수집 기간', f'{day_n:,}일',   PALETTE['teal'])]
    for i, (label, val, color) in enumerate(cards):
        x = i / 6 + 0.02
        ax0.add_patch(mpatches.FancyBboxPatch(
            (x, 0.1), 0.14, 0.8, boxstyle='round,pad=0.02',
            facecolor=color, alpha=0.15, edgecolor=color, linewidth=2,
            transform=ax0.transAxes))
        ax0.text(x + 0.07, 0.68, label, ha='center', fontsize=11,
                 color=color, transform=ax0.transAxes, fontweight='bold')
        ax0.text(x + 0.07, 0.32, val, ha='center', fontsize=17,
                 color=color, transform=ax0.transAxes, fontweight='bold')
    ax0.set_title(COMPANY_NAME + ' 뉴스 수집 현황  (' +
                  START_DATE + ' ~ ' + END_DATE + ')',
                  fontsize=14, fontweight='bold', pad=8)

    # ── ax1: 월별 수집 기사 수 (Stacked Bar) ─────────────────
    ax1 = fig.add_subplot(gs[1, 0])
    monthly = (df_raw.groupby('ym')
                     .agg(total=('is_related', 'size'),
                          related=('is_related', 'sum'))
                     .reset_index()
                     .rename(columns={'ym': 'month'})
                     .sort_values('month'))
    monthly['unrelated'] = monthly['total'] - monthly['related']

    x_idx = range(len(monthly))
    ax1.bar(x_idx, monthly['related'],
            label='관련 (Related)', color=PALETTE['green'])
    ax1.bar(x_idx, monthly['unrelated'],
            bottom=monthly['related'],
            label='무관 (Unrelated)', color=PALETTE['gray'])

    # 연도 단위 x축 레이블
    year_positions = []
    year_labels    = []
    seen_years     = set()
    for i, ym in enumerate(monthly['month']):
        year = ym[:4]
        if year not in seen_years:
            year_positions.append(i)
            year_labels.append(year)
            seen_years.add(year)

    ax1.set_xticks(year_positions)
    ax1.set_xticklabels(year_labels, rotation=0, ha='center',
                        fontsize=12, fontweight='bold')
    for pos in year_positions[1:]:
        ax1.axvline(pos - 0.5, color='gray',
                    linewidth=0.5, linestyle='--', alpha=0.4)

    ax1.tick_params(axis='y', labelsize=10)
    ax1.set_title('월별 수집 기사 수 (Monthly Article Count)',
                  fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.25, axis='y')

    # ── ax2: 관련 기사 언론사 Top 10 ─────────────────────────
    ax2 = fig.add_subplot(gs[1, 1])
    press_top = (df_raw[df_raw['is_related'] == 1]['press_name']
                 .value_counts().head(10))
    ax2.barh(press_top.index[::-1], press_top.values[::-1],
             color=PALETTE['teal'])
    ax2.set_title('관련 기사 언론사 Top 10',
                  fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.25, axis='x')

    # ── ax3: 관련/무관 비율 Pie Chart ────────────────────────
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.pie([related, unrel],
            labels=['관련', '무관'],
            autopct='%1.1f%%',
            colors=[PALETTE['green'], PALETTE['gray']],
            startangle=90)
    ax3.set_title('관련/무관 비율', fontsize=12, fontweight='bold')

    # ── 최종 레이아웃 ────────────────────────────────────────
    plt.suptitle(COMPANY_NAME + ' 헤드라인 수집 현황',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
    print('섹션 3 완료!')

---
# 섹션 4-회수 : 누락된 달 재수집

> 메인 수집에서 차단으로 누락되었거나 너무 적게 수집된 달을 자동 식별해 다시 도는 셀입니다.
> 메인 수집(섹션 4)이 한 번 끝난 뒤 실행하세요. `RECOVER = False`로 두면 건너뜁니다.
>
> | 동작 | 설명 |
> |---|---|
> | 식별 | DB에서 월별 기사 수가 `MIN_EXPECTED`(기본 3건) 미만인 달을 추출 |
> | 휴식 | 시작 전 3분 대기 + 새 세션 발급 |
> | 수집 | 섹션 4의 `collect_one_month()` 함수를 그대로 재사용 |
> | 종료 | 또 차단되면 그 달은 건너뛰고 다음 달로 진행


In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 12]  섹션 4-회수 : 누락된 달 재수집')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 섹션 4-회수: 차단으로 누락된 달 재수집
# ═══════════════════════════════════════════════════════════════════
# DB 기준으로 0건 또는 극히 저조한 달을 자동 식별해 그 달들만 다시 수집합니다.
# MAX 기반 체크포인트는 0건으로 지나간 달을 재시도하지 않으므로, 이 셀이 빈틈을 보완합니다.

MIN_EXPECTED = 3       # 이보다 적은 달은 차단 의심
RECOVER      = True    # False로 바꾸면 실행 안 됨

if RECOVER:
    df_chk = pd.read_sql(
        text('SELECT publish_date FROM ' + T_RAW + ' WHERE topic_code = :tc'),
        engine, params={'tc': TOPIC_CODE})

    if len(df_chk) == 0:
        print('수집된 데이터가 없습니다. 섹션 4를 먼저 실행하세요.')
    else:
        df_chk['ym'] = pd.to_datetime(df_chk['publish_date'],
                                      format='%Y%m%d', errors='coerce').dt.to_period('M')
        monthly = df_chk.groupby('ym').size()

        all_months = pd.period_range(START_DATE, END_DATE, freq='M')
        to_recover = []
        for m in all_months:
            cnt = int(monthly.get(m, 0))
            if cnt < MIN_EXPECTED:
                ds_str = m.start_time.strftime('%Y.%m.%d')
                de_str = m.end_time.strftime('%Y.%m.%d')
                end_ts = pd.Timestamp(m.end_time.date())
                to_recover.append((m, ds_str, de_str, end_ts, cnt))

        print('재수집 대상:', len(to_recover), '개월 (각', MIN_EXPECTED, '건 미만)')
        for m, _, _, _, cnt in to_recover[:15]:
            print('  -', m, '(현재', cnt, '건)')
        if len(to_recover) > 15:
            print('  ... 외', len(to_recover) - 15, '개')

        if to_recover:
            print('\n새 세션으로 재수집 시작 (3분 대기 - 누적 점수 감쇠)')
            time.sleep(180)

            session  = make_session()
            warmup_session(session)
            prev_url = 'https://search.naver.com/'

            rec_total       = 0
            rec_related     = 0
            still_blocked   = []

            for i, (m, ds, de, end_ts, _) in enumerate(to_recover, 1):
                print('[' + str(i) + '/' + str(len(to_recover)) + ']', m, '재수집 시작')
                month_cnt, month_rel, blocked, prev_url, session = collect_one_month(
                    session, ds, de, end_ts, prev_url)
                rec_total   += month_cnt
                rec_related += month_rel

                if blocked:
                    still_blocked.append(str(m))
                    print('   ', m, ': [재차단] 다음 달로 진행')
                else:
                    print('   ', m, ': +', month_cnt, '건 / 관련 +', month_rel, '건')

                # 월 사이 휴식 (메인 루프보다 길게)
                time.sleep(random.uniform(20, 40))

            print('\n재수집 완료!')
            print('  추가 수집: +', rec_total, '건 / 관련 +', rec_related, '건')
            if still_blocked:
                print('  여전히 차단된 달:', len(still_blocked), '개')
                for s in still_blocked:
                    print('    -', s)
                print('\n→ 잠시 후(또는 내일) RECOVER 셀을 다시 실행하세요.')
        else:
            print('\n재수집 대상 없음 - 모든 달이', MIN_EXPECTED, '건 이상 수집됨')
else:
    print('재수집 비활성화 (RECOVER = False)')


---
# 섹션 5 : 전체 본문 수집

> 관련 기사(is_related=1)의 전체 본문을 수집합니다.

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 14]  섹션 5 : 전체 본문 수집')
print('=' * 70)

def remove_emoji(text):
    return re.compile('[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]+',
                      flags=re.UNICODE).sub('',text)

def clean_body(text):
    text=remove_emoji(text)
    text=re.sub(r'\s{2,}',' ',text).strip()
    enc=text.encode('utf-8')
    if len(enc)>65535:
        trunc=enc[:65535]
        while len(trunc)>0 and trunc[-1]&0b11000000==0b10000000: trunc=trunc[:-1]
        text=trunc.decode('utf-8',errors='ignore')
    return text

def crawl_body(url):
    try:
        resp=requests.get(url,headers={'User-Agent':random.choice(UA_POOL),'Referer':'https://www.google.com/'},timeout=10)
        resp.raise_for_status()
        soup=BeautifulSoup(resp.text,'lxml')
        area=soup.select_one('#dic_area')
        if not area: return ''
        for tag in area.find_all(['span','div','img','br','a','b','li','ul','iframe']): tag.decompose()
        return clean_body(area.get_text(strip=True))
    except: return ''

# 체크포인트
src_df=pd.read_sql('SELECT * FROM '+T_RAW+' WHERE is_related=1',engine)
try:    done_urls=set(pd.read_sql('SELECT naver_url FROM '+T_ALL,engine)['naver_url'])
except: done_urls=set()
to_crawl=src_df[~src_df['naver_url'].isin(done_urls)].reset_index(drop=True)
print('관련 기사:',len(src_df),'건 / 완료:',len(done_urls),'건 / 수집 대상:',len(to_crawl),'건')

if len(to_crawl)>0:
    BATCH=MAX_WORKERS*2; success=0; fail=0
    pbar=tqdm(total=len(to_crawl),desc='본문 수집')
    for start in range(0,len(to_crawl),BATCH):
        batch=to_crawl.iloc[start:start+BATCH]
        arts=[{'idx':i,'naver_url':r['naver_url'],'topic_code':r['topic_code'],
               'search_keyword':r['search_keyword'],'publish_date':r['publish_date'],
               'press_name':r['press_name'],'press_url':r.get('press_url',''),'title':r['title']}
              for i,(_,r) in enumerate(batch.iterrows())]
        results={}
        def _p(a): return a['idx'],crawl_body(a['naver_url'])
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for f in as_completed({ex.submit(_p,a):a for a in arts}):
                idx,body=f.result(); results[idx]=body
        with engine.begin() as conn:
            for art in arts:
                body=results.get(art['idx'],'')
                if body:
                    conn.execute(text(
                        'INSERT INTO '+T_ALL
                        +' (topic_code,search_keyword,publish_date,press_name,press_url,naver_url,title,contents)'
                        ' VALUES (:tc,:kw,:pd,:pn,:pu,:nu,:ti,:co)'
                        ' ON DUPLICATE KEY UPDATE title=VALUES(title),contents=VALUES(contents)'),
                        dict(tc=art['topic_code'],kw=art['search_keyword'],pd=art['publish_date'],
                             pn=art['press_name'],pu=art.get('press_url',''),nu=art['naver_url'],
                             ti=art['title'],co=body))
                    success+=1
                else: fail+=1
        pbar.update(len(arts)); pbar.set_postfix(성공=success,실패=fail)
    pbar.close()
    print('본문 수집 완료! 성공:',success,'건 / 실패:',fail,'건')
else:
    print('이미 모든 본문이 수집되어 있습니다.')

In [ ]:
# 시각화
df_all=pd.read_sql('SELECT * FROM '+T_ALL,engine)
if len(df_all)>0:
    df_all['body_len']=df_all['contents'].str.len().fillna(0).astype(int)
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    axes[0].hist(df_all['body_len'],bins=30,color=PALETTE['blue'],alpha=0.8,edgecolor='white')
    axes[0].axvline(df_all['body_len'].median(),color=PALETTE['red'],linestyle='--',
                   label=f'중앙값: {int(df_all["body_len"].median()):,}자')
    axes[0].set_title('본문 길이 분포',fontsize=12,fontweight='bold'); axes[0].legend(); axes[0].grid(True,alpha=0.25)
    has=( df_all['body_len']>100).sum(); no=len(df_all)-has
    axes[1].pie([has,no],labels=['본문 있음','본문 없음'],colors=[PALETTE['green'],PALETTE['gray']],
               autopct='%1.1f%%',startangle=90)
    axes[1].set_title('본문 수집 성공률',fontsize=12,fontweight='bold')
    df_all['date'] = pd.to_datetime(df_all['publish_date'], format='%Y%m%d', errors='coerce')
df_all['ym']   = df_all['date'].dt.to_period('M').astype(str)
ym_cnt = df_all.groupby('ym').size()

axes[2].bar(range(len(ym_cnt)), ym_cnt.values,
            color=PALETTE['teal'], alpha=0.85)

# ── X축 레이블: 각 연도의 첫 월 위치만 표시 ──────────────────
year_positions = []
year_labels = []
seen_years = set()
for i, ym in enumerate(ym_cnt.index):
    year = ym[:4]
    if year not in seen_years:
        year_positions.append(i)
        year_labels.append(year)
        seen_years.add(year)

axes[2].set_xticks(year_positions)
axes[2].set_xticklabels(year_labels,
                        rotation=0,
                        ha='center',
                        fontsize=12,
                        fontweight='bold')

# 연도 경계선
for pos in year_positions[1:]:
    axes[2].axvline(pos - 0.5, color='gray',
                    linewidth=0.5, linestyle='--', alpha=0.4)

axes[2].tick_params(axis='y', labelsize=10)
axes[2].set_title('월별 본문 수집 기사 수', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.25, axis='y')
plt.suptitle(COMPANY_NAME+' 본문 수집 현황',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()
print('섹션 5 완료! 총',len(df_all),'건 / 평균 본문 길이:',int(df_all['body_len'].mean()),'자')


---
# 섹션 6 : 키워드 추출 + 임베딩

> GPT로 키워드를 추출하고 text-embedding-3-large로 3072차원 벡터를 생성합니다.

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 16]  섹션 6-1 : 임베딩 테이블 초기화 (선택)')
print('=' * 70)

# 키워드 추출이 잘못된 경우 T_EMBD만 초기화 (수집한 본문은 유지)
with engine.begin() as conn:
    r = conn.execute(text('DELETE FROM '+T_EMBD+' WHERE code=:c'), {'c': TOPIC_CODE})
    print('삭제 완료:', r.rowcount, '건')

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 17]  섹션 6-2 : 키워드 추출 + 임베딩')
print('=' * 70)

def extract_keywords(query, body):
    prompt = (
    f'[{query}] 관련 뉴스에서 키워드를 추출합니다.\n'
    f'[{query}]이 주요 주체로 다뤄지지 않으면: {{"keywords": "NaN"}}\n\n'

    '## 규칙\n'
    '1. 품사: 명사·고유명사만 허용. 동사·형용사·부사·어미 포함 시 탈락\n'
    '2. 원형: 조사·어미를 모두 제거한 사전 표제어 형태\n'
    '3. 길이: 2글자 이상, 합성명사는 붙여쓰기\n'
    '4. 최소 10개\n'
    f'5. 제외: 숫자/날짜, 불용어(기자,뉴스,보도,기사,관련), [{query}] 자체\n\n'

    '## 나쁜 예 → 올바른 처리\n'
    '쿠팡은 → 쿠팡 (조사 제거)\n'
    '실적이 → 실적 (조사 제거)\n'
    '최동형은 → 최동형 (조사 제거)\n'
    '밀쳤다 → 탈락 (동사이므로 제외)\n'
    '성장하는 → 탈락 (동사 관형형이므로 제외)\n'
    '빠르게 → 탈락 (부사이므로 제외)\n\n'

    'JSON으로만 응답: {"keywords": {"탄소중립목표": 3, "온실가스배출": 2}}'
)
    try:
        r=client.chat.completions.create(
            model='gpt-4o-mini-2024-07-18',
            messages=[{'role':'system','content':prompt},{'role':'user','content':body}],
            temperature=0.0,max_tokens=1000)
        raw=r.choices[0].message.content.strip()
        raw=re.sub(r'^```json|```$','',raw,flags=re.MULTILINE).strip()
        parsed=ast.literal_eval(raw)
        kw=parsed.get('keywords','NaN')
        if isinstance(kw,dict):
            import re as _re
            _P = _re.compile(
                r'(으로부터|에 대한|로부터|에서의|이라는|이라고|으로서|에서는|에게는'
                r'|에서|에게|부터|까지|에는|으로|이고|이며|이나|라고|하고|이라|라는'
                r'|은|는|이|가|을|를|의|에|로|와|과|도|만|들|적|화|형|용)$')
            def _strip(w):
                c = _P.sub('', w)
                return c if len(c) >= 2 else w
            kw = {_strip(k):v for k,v in kw.items() if len(_strip(k))>=2}
            if not kw: return {'keywords':'NaN'}
            parsed['keywords']=kw
        return parsed
    except: return {'keywords':'NaN'}

def build_embed_text(kj):
    try:
        kw=kj.get('keywords',{})
        if kw in ['NaN',None] or not isinstance(kw,dict): return ''
        return ' '.join([w+' '*freq for w,freq in kw.items()]).strip()
    except: return ''

def embed_text(txt):
    try:
        if not txt or not txt.strip(): return [0.0]*3072
        r=client.embeddings.create(input=txt,model='text-embedding-3-large')
        return r.data[0].embedding
    except: return [0.0]*3072

df_a=pd.read_sql('SELECT * FROM '+T_ALL,engine)
df_a=df_a.rename(columns={'topic_code':'code','search_keyword':'query',
                            'publish_date':'date','press_name':'press','naver_url':'url'})
try:    done=set(pd.read_sql('SELECT url FROM '+T_EMBD,engine)['url'])
except: done=set()
to_proc=df_a[~df_a['url'].isin(done)].reset_index(drop=True)
print('전체:',len(df_a),'건 / 완료:',len(done),'건 / 처리 대상:',len(to_proc),'건')

if len(to_proc)>0:
    BATCH=10; success=0; fail=0
    pbar=tqdm(total=len(to_proc),desc='키워드+임베딩')
    for start in range(0,len(to_proc),BATCH):
        batch=to_proc.iloc[start:start+BATCH]
        arts=[{'idx':i,'code':r['code'],'query':r['query'],'date':r['date'],
               'press':r['press'],'url':r['url'],'press_url':r.get('press_url',''),
               'title':r['title'],'contents':r['contents']}
              for i,(_,r) in enumerate(batch.iterrows())]
        results={}
        def _proc(art):
            idx=art['idx']
            try:
                if not isinstance(art['contents'],str) or not art['contents'].strip():
                    return {'idx':idx,'kj':{},'ei':'','ev':[0.0]*3072,'ok':False}
                kj=extract_keywords(art['query'],art['contents'])
                ei=build_embed_text(kj)
                ev=embed_text(ei)
                return {'idx':idx,'kj':kj,'ei':ei,'ev':ev,'ok':True}
            except: return {'idx':idx,'kj':{},'ei':'','ev':[0.0]*3072,'ok':False}
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for f in as_completed({ex.submit(_proc,a):a for a in arts}):
                r=f.result(); results[r['idx']]=r
        with engine.begin() as conn:
            for art in arts:
                res=results.get(art['idx'])
                if res and res['ok']:
                    conn.execute(text(
                        'REPLACE INTO '+T_EMBD
                        +' (code,query,date,press,url,title,contents,Keyword_JSON,Embedding_Input,Embedding_Vector)'
                        ' VALUES (:co,:qu,:da,:pr,:ur,:ti,:ct,:kj,:ei,:ev)'),
                        dict(co=art['code'],qu=art['query'],da=art['date'],pr=art['press'],
                             ur=art['url'],ti=art['title'],ct=art['contents'],
                             kj=json.dumps(res['kj'],ensure_ascii=False),
                             ei=res['ei'],ev=json.dumps(res['ev'])))
                    success+=1
                else: fail+=1
        pbar.update(len(arts)); pbar.set_postfix(성공=success,실패=fail)
    pbar.close()
    print('키워드+임베딩 완료! 성공:',success,'건')
else:
    print('이미 처리 완료!')

In [ ]:
# 시각화
df_emb=pd.read_sql('SELECT * FROM '+T_EMBD,engine)
if len(df_emb)>0:
    from collections import Counter
    all_kws=Counter()
    for _,row in df_emb.iterrows():
        try:
            kj=json.loads(row['Keyword_JSON']) if isinstance(row['Keyword_JSON'],str) else row['Keyword_JSON']
            kw=kj.get('keywords',{})
            if isinstance(kw,dict):
                for k,v in kw.items(): all_kws[k]+=v
        except: pass
    top20=dict(all_kws.most_common(20))
    if top20:
        fig,axes=plt.subplots(1,2,figsize=(16,6))
        kn=list(top20.keys())[::-1]; kv=list(top20.values())[::-1]
        axes[0].barh(kn,kv,color=plt.cm.Blues(np.linspace(0.4,0.9,len(kn))))
        axes[0].set_title('상위 20 키워드',fontsize=13,fontweight='bold'); axes[0].grid(True,alpha=0.2,axis='x')
        for i,v in enumerate(kv): axes[0].text(v+0.3,i,str(v),va='center',fontsize=9)
        axes[1].axis('off'); axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)
        axes[1].set_title('키워드 워드맵',fontsize=13,fontweight='bold')
        items=list(dict(all_kws.most_common(40)).items()); max_v=max(v for _,v in items) if items else 1
        import random as _rnd; _rnd.seed(42); placed=[]
        clrs=[PALETTE['blue'],PALETTE['teal'],PALETTE['purple'],PALETTE['amber'],PALETTE['green']]
        for word,freq in sorted(items,key=lambda x:-x[1]):
            fsize=8+28*(freq/max_v)
            for _ in range(60):
                x=_rnd.uniform(0.05,0.95); y=_rnd.uniform(0.05,0.95)
                if not any(abs(x-px)<0.13 and abs(y-py)<0.07 for px,py in placed):
                    axes[1].text(x,y,word,fontsize=fsize,ha='center',va='center',
                                color=_rnd.choice(clrs),fontweight='bold' if freq>max_v*0.5 else 'normal')
                    placed.append((x,y)); break
        plt.suptitle(COMPANY_NAME+' 키워드 분석',fontsize=14,fontweight='bold')
        plt.tight_layout(); plt.show()
    print('섹션 6 완료! 고유 키워드:',len(all_kws),'개')

---
# 섹션 7 : 감성 분석

> GPT가 기사 본문을 읽고 긍정/부정/중립으로 분류합니다.

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 19]  섹션 7 : 감성 분석')
print('=' * 70)

def classify_sent(body,query):
    prompt=('['+query+'] 관련 뉴스 감성을 판단합니다.\n'
            '긍정: 호재/성장/혁신/수상  부정: 악재/논란/하락/사고  중립: 단순 사실\n\n'
            'JSON으로만: {"sentiment": "긍정" 또는 "부정" 또는 "중립"}\n\n'
            '[기사]:\n'+body.strip())
    try:
        r=client.chat.completions.create(
            model='gpt-4o-mini-2024-07-18',
            messages=[{'role':'user','content':prompt}],temperature=0)
        return json.loads(r.choices[0].message.content.strip()).get('sentiment','중립')
    except: return '중립'

df_e=pd.read_sql('SELECT * FROM '+T_EMBD,engine)
df_e=df_e[df_e['Embedding_Input'].notna()&(df_e['Embedding_Input']!='')]
df_e.drop_duplicates(subset=['code','query','date','press','url'],inplace=True)
try:    done=set(pd.read_sql('SELECT url FROM '+T_SENT,engine)['url'])
except: done=set()
to_proc=df_e[~df_e['url'].isin(done)].reset_index(drop=True)
print('유효 기사:',len(df_e),'건 / 완료:',len(done),'건 / 처리 대상:',len(to_proc),'건')

if len(to_proc)>0:
    BATCH=20; sc={'긍정':0,'부정':0,'중립':0}
    pbar=tqdm(total=len(to_proc),desc='감성 분석')
    for start in range(0,len(to_proc),BATCH):
        batch=to_proc.iloc[start:start+BATCH]
        arts=[{'idx':i,'code':r['code'],'query':r['query'],'date':r['date'],
               'press':r['press'],'url':r['url'],'title':r['title'],'contents':r['contents'],
               'Keyword_JSON':r['Keyword_JSON'],'Embedding_Input':r['Embedding_Input'],
               'Embedding_Vector':r['Embedding_Vector']}
              for i,(_,r) in enumerate(batch.iterrows())]
        results={}
        def _p(a): return a['idx'],classify_sent(a['contents'],SEARCH_KEYWORD)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for f in as_completed({ex.submit(_p,a):a for a in arts}):
                idx,s=f.result(); results[idx]=s
        with engine.begin() as conn:
            for art in arts:
                s=results.get(art['idx'],'중립')
                if s in sc: sc[s]+=1
                conn.execute(text(
                    'REPLACE INTO '+T_SENT
                    +' (code,query,date,press,url,title,contents,'
                    '  Keyword_JSON,Embedding_Input,Embedding_Vector,sentiment)'
                    ' VALUES (:co,:qu,:da,:pr,:ur,:ti,:ct,:kj,:ei,:ev,:se)'),
                    dict(co=art['code'],qu=art['query'],da=art['date'],pr=art['press'],
                         ur=art['url'],ti=art['title'],ct=art['contents'],
                         kj=art['Keyword_JSON'],ei=art['Embedding_Input'],
                         ev=art['Embedding_Vector'],se=s))
        pbar.update(len(arts)); pbar.set_postfix(**sc)
    pbar.close()
    total=sum(sc.values())
    print('감성 분석 완료!')
    for s,c in sc.items():
        print('  ',s,':',c,'건 ('+str(round(c/total*100,1))+'%)' if total>0 else '')
else:
    print('이미 처리 완료!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 섹션 7 시각화 (3-panel: 파이 + 막대 + 월별 추이)
# ═══════════════════════════════════════════════════════════════════
df_sent = pd.read_sql('SELECT * FROM ' + T_SENT, engine)

if len(df_sent) > 0:
    # ── 데이터 정제 ───────────────────────────────────────────────
    def _cs(x):
        if pd.isna(x): return '중립'
        return re.sub(r'[\*\[\]\(\)_`<>]', '', str(x)).strip()

    df_sent['sc'] = df_sent['sentiment'].apply(_cs)
    df_sent['sv'] = df_sent['sc'].map({'긍정': 1, '중립': 0, '부정': -1})
    df_sent['date'] = pd.to_datetime(
        df_sent['date'].astype(str).str.replace('-', ''),
        format='%Y%m%d', errors='coerce'
    )
    df_sent['ym'] = df_sent['date'].dt.to_period('M').astype(str)

    SC = {'긍정': PALETTE['pos'], '부정': PALETTE['neg'], '중립': PALETTE['neu']}

    # ── Figure 생성 ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── [1] 파이 차트: 감성 분포 ──────────────────────────────────
    sc2 = df_sent['sc'].value_counts()
    labels = [s for s in ['긍정', '부정', '중립'] if s in sc2.index]
    vals = [sc2[s] for s in labels]
    clrs = [SC[s] for s in labels]

    axes[0].pie(
        vals, labels=labels, autopct='%1.1f%%',
        colors=clrs, startangle=90,
        textprops={'fontsize': 12, 'fontweight': 'bold'}
    )
    axes[0].set_title('감성 분포', fontsize=14, fontweight='bold')

    # ── [2] 막대 차트: 감성별 기사 수 ─────────────────────────────
    axes[1].bar(labels, vals, color=clrs, width=0.5)
    axes[1].set_title('감성별 기사 수', fontsize=14, fontweight='bold')
    axes[1].tick_params(axis='both', labelsize=12)
    axes[1].grid(True, alpha=0.25, axis='y')
    for i, v in enumerate(vals):
        axes[1].text(i, v + 0.3, str(v), ha='center',
                     fontsize=13, fontweight='bold')

    # ── [3] 월별 평균 감성 지수 (X축: 연도 단위) ──────────────────
    monthly_idx = df_sent.groupby('ym')['sv'].mean().reset_index()
    clrs_idx = [PALETTE['pos'] if v >= 0 else PALETTE['neg']
                for v in monthly_idx['sv']]
    axes[2].bar(range(len(monthly_idx)), monthly_idx['sv'],
                color=clrs_idx, alpha=0.85)
    axes[2].axhline(0, color='black', linewidth=0.8)

    # X축 레이블: 각 연도의 첫 월 위치만 표시
    year_positions = []
    year_labels = []
    seen_years = set()
    for i, ym in enumerate(monthly_idx['ym']):
        year = ym[:4]                          # 'YYYY-MM' → 'YYYY'
        if year not in seen_years:
            year_positions.append(i)
            year_labels.append(year)
            seen_years.add(year)

    axes[2].set_xticks(year_positions)
    axes[2].set_xticklabels(year_labels,
                            rotation=0,        # 회전 제거 (가로 표시)
                            ha='center',
                            fontsize=13,       # 폰트 확대
                            fontweight='bold')

    # 연도 경계선 (시각적 구분)
    for pos in year_positions[1:]:             # 첫 연도 제외
        axes[2].axvline(pos - 0.5, color='gray',
                        linewidth=0.5, linestyle='--', alpha=0.4)

    axes[2].tick_params(axis='y', labelsize=11)
    axes[2].set_title('월별 평균 감성 지수', fontsize=14, fontweight='bold')
    axes[2].grid(True, alpha=0.25, axis='y')

    # ── 전체 제목 + 출력 ──────────────────────────────────────────
    plt.suptitle(COMPANY_NAME + ' 감성 분석',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('섹션 7 완료!')

---
# 섹션 8 : ESG 이슈 분류

> 코사인 유사도로 각 기사를 ESG 이슈에 매핑합니다.

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 21]  섹션 8 : ESG 이슈 분류')
print('=' * 70)

def spv(x):
    if isinstance(x,list): return x
    if isinstance(x,str):
        try: return json.loads(x)
        except: return [0.0]*3072
    return [0.0]*3072

esg_desc=pd.read_sql('SELECT * FROM '+ESG_TABLE,engine)
esg_desc['Embedding_Vector']=esg_desc['Embedding_Vector'].apply(spv)
print('ESG 이슈:',len(esg_desc),'개')

df_emb2=pd.read_sql('SELECT * FROM '+T_EMBD,engine)
if 'press' in df_emb2.columns:
    df_emb2['press']=df_emb2['press'].str.replace('언론사','',regex=False).str.strip()
df_emb2.drop_duplicates(subset=['code','query','date','press','url'],inplace=True)
df_valid=df_emb2[df_emb2['Embedding_Input'].notna()&(df_emb2['Embedding_Input']!='')].copy()
try:    done=set(pd.read_sql('SELECT DISTINCT url FROM '+T_ESG,engine)['url'])
except: done=set()
to_proc=df_valid[~df_valid['url'].isin(done)].reset_index(drop=True)
print('유효:',len(df_valid),'건 / 완료:',len(done),'건 / 처리 대상:',len(to_proc),'건')

if len(to_proc)>0:
    news_vecs=np.array(to_proc['Embedding_Vector'].apply(spv).tolist())
    esg_vecs=np.array(esg_desc['Embedding_Vector'].tolist())
    sim_mat=cosine_similarity(news_vecs,esg_vecs)
    print('유사도 계산 완료:', sim_mat.shape, '임계값:', SIMILARITY_THRESHOLD)
    match_rows=[]
    for i,(_,nr) in enumerate(to_proc.iterrows()):
        for j,(_,er) in enumerate(esg_desc.iterrows()):
            if sim_mat[i,j]>=SIMILARITY_THRESHOLD:
                match_rows.append({'code':nr['code'],'query':nr['query'],
                                   'date':nr['date'],'press':nr['press'],'url':nr['url'],
                                   'esg_category':er['Category'],'esg_issue':er['Issue'],
                                   'similarity':round(float(sim_mat[i,j]),4)})
    match_df=pd.DataFrame(match_rows)
    if len(match_df)>0:
        BATCH=1000
        with engine.begin() as conn:
            for start in tqdm(range(0,len(match_df),BATCH),desc='ESG 저장'):
                batch=match_df.iloc[start:start+BATCH]
                conn.execute(text(
                    'REPLACE INTO '+T_ESG
                    +' (code,query,date,press,url,esg_category,esg_issue,similarity)'
                    ' VALUES (:co,:qu,:da,:pr,:ur,:ec,:ei,:si)'),
                    [{'co':r['code'],'qu':r['query'],'da':r['date'],'pr':r['press'],
                      'ur':r['url'],'ec':r['esg_category'],'ei':r['esg_issue'],
                      'si':float(r['similarity'])} for _,r in batch.iterrows()])
        print('ESG 분류 완료! 매칭:',len(match_df),'건')
    else:
        print('임계값 이상 매칭 없음 — SIMILARITY_THRESHOLD를 낮춰보세요 (현재:',SIMILARITY_THRESHOLD,')')
else:
    print('이미 처리 완료!')


In [ ]:
# 시각화
df_esg=pd.read_sql('SELECT * FROM '+T_ESG,engine)
if len(df_esg)>0:
    CAT_COLORS={'Environment':PALETTE['green'],'Social Capital':PALETTE['blue'],
                'Human Capital':PALETTE['amber'],'Business Model & Innovation':PALETTE['purple'],
                'Leadership & Governance':PALETTE['red']}
    cat_cnt=df_esg['esg_category'].value_counts()
    fig,axes=plt.subplots(1,3,figsize=(18,5))
    c_list=[CAT_COLORS.get(c,PALETTE['gray']) for c in cat_cnt.index]
    axes[0].pie(cat_cnt.values,labels=None,autopct='%1.1f%%',colors=c_list,startangle=90)
    axes[0].legend(cat_cnt.index,loc='lower center',bbox_to_anchor=(0.5,-0.3),fontsize=8,ncol=1)
    axes[0].set_title('ESG 카테고리 분포',fontsize=12,fontweight='bold')
    axes[1].barh(cat_cnt.index[::-1],cat_cnt.values[::-1],color=c_list[::-1])
    axes[1].set_title('카테고리별 매칭 수',fontsize=12,fontweight='bold'); axes[1].grid(True,alpha=0.25,axis='x')
    issue_cnt=df_esg.groupby(['esg_category','esg_issue']).size().reset_index(name='cnt').nlargest(10,'cnt')
    clrs_i=[CAT_COLORS.get(c,PALETTE['gray']) for c in issue_cnt['esg_category']]
    axes[2].barh(range(len(issue_cnt)),issue_cnt['cnt'],color=clrs_i)
    axes[2].set_yticks(range(len(issue_cnt))); axes[2].set_yticklabels(issue_cnt['esg_issue'],fontsize=9)
    axes[2].set_title('ESG 이슈 Top 10',fontsize=12,fontweight='bold'); axes[2].grid(True,alpha=0.2,axis='x')
    plt.suptitle(COMPANY_NAME+' ESG 분류',fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()
    print('섹션 8 완료!')
else:
    print('ESG 매칭 데이터 없음')


## 테이블 로컬 다운로드(선택)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 📘 Excel 통합 리포트 생성 (경로 자동 생성 버전)
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
import os
from datetime import datetime
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ── 🔧 출력 경로 설정 (환경에 맞게 선택) ───────────────
# 옵션 A: 현재 작업 디렉토리 (가장 안전, 권장)
output_dir  = '.'

# 옵션 B: 특정 경로 지정 시 — 자동으로 디렉토리 생성
# output_dir = '/mnt/user-data/outputs'
# output_dir = '/content/drive/MyDrive/ESG_Reports'   # Colab + Drive

os.makedirs(output_dir, exist_ok=True)   # ⭐ 디렉토리 자동 생성

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = os.path.join(output_dir,
                           f'{COMPANY_NAME}_통합리포트_{timestamp}.xlsx')

print('='*70)
print(f'📘 Excel 통합 리포트 생성 시작')
print('='*70)
print(f'대상 기업 : {COMPANY_NAME} ({TOPIC_CODE})')
print(f'분석 기간 : {START_DATE} ~ {END_DATE}')
print(f'출력 경로 : {os.path.abspath(output_path)}\n')

# ──────────────────────────────────────────────────────
# [Step 1] 테이블 로드
# ──────────────────────────────────────────────────────
print('[Step 1] DB에서 테이블 로드')
print('-'*70)

df_all  = pd.read_sql(
    f'SELECT * FROM {T_ALL} WHERE topic_code=%(tc)s',
    engine, params={'tc': TOPIC_CODE})

df_sent = pd.read_sql(
    f'SELECT code, query, date, press, url, title, sentiment '
    f'FROM {T_SENT} WHERE code=%(tc)s',
    engine, params={'tc': TOPIC_CODE})

df_esg = pd.read_sql(
    f'SELECT * FROM {T_ESG} WHERE code=%(tc)s',
    engine, params={'tc': TOPIC_CODE})

print(f'  T_ALL  (본문)          : {len(df_all):,}건')
print(f'  T_SENT (감성)          : {len(df_sent):,}건')
print(f'  T_ESG  (M:N ESG 매칭)  : {len(df_esg):,}건')

# ──────────────────────────────────────────────────────
# [Step 2] 05_ESGMatch: 기사당 1행으로 집계 + 본문 컬럼 추가
# ──────────────────────────────────────────────────────
print('\n[Step 2] ESG 매칭을 기사당 1행으로 집계 + 본문 추가')
print('-'*70)

df_esg_sorted = df_esg.sort_values('similarity', ascending=False)

def _join_unique(series):
    vals = [str(v) for v in series if pd.notna(v) and str(v).strip()]
    seen, out = set(), []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return ' | '.join(out)

esg_agg = (df_esg_sorted.groupby('url', as_index=False)
           .agg(
               code =('code',  'first'),
               query=('query', 'first'),
               date =('date',  'first'),
               press=('press', 'first'),
               esg_match_count=('esg_issue',    'count'),
               top_issue      =('esg_issue',    'first'),
               top_category   =('esg_category', 'first'),
               max_similarity =('similarity',   'max'),
               avg_similarity =('similarity',   'mean'),
               esg_issues     =('esg_issue',    _join_unique),
               esg_categories =('esg_category', _join_unique),
           ))

esg_agg['max_similarity'] = esg_agg['max_similarity'].round(4)
esg_agg['avg_similarity'] = esg_agg['avg_similarity'].round(4)

df_05 = esg_agg.merge(
    df_sent[['url', 'title', 'sentiment']],
    on='url', how='left'
)

df_05 = df_05.merge(
    df_all[['naver_url', 'contents']].rename(columns={'naver_url': 'url'}),
    on='url', how='left'
)

# 컬럼 순서: title 바로 다음에 contents
df_05 = df_05[[
    'code', 'query', 'date', 'press', 'url',
    'title', 'contents',
    'sentiment',
    'esg_match_count', 'top_issue', 'top_category',
    'max_similarity', 'avg_similarity',
    'esg_issues', 'esg_categories',
]].sort_values('date', ascending=False)

has_body = df_05['contents'].notna().sum()
print(f'  원본 M:N 매칭      : {len(df_esg):,}건')
print(f'  집계 후 기사 수    : {len(df_05):,}건 (기사당 1행)')
print(f'  본문 확보된 기사   : {has_body:,}건 / {len(df_05):,}건')
print(f'  기사당 평균 이슈   : {df_05["esg_match_count"].mean():.2f}개')

# ──────────────────────────────────────────────────────
# [Step 3] 06_MonthlyStats
# ──────────────────────────────────────────────────────
print('\n[Step 3] 월별 집계')
print('-'*70)

df_05_tmp = df_05.copy()
df_05_tmp['date_dt']    = pd.to_datetime(df_05_tmp['date'].astype(str),
                                          format='%Y%m%d', errors='coerce')
df_05_tmp['year_month'] = df_05_tmp['date_dt'].dt.to_period('M').astype(str)

monthly = (df_05_tmp.groupby('year_month')
           .agg(total_articles =('url',       'count'),
                positive       =('sentiment', lambda x: (x == '긍정').sum()),
                negative       =('sentiment', lambda x: (x == '부정').sum()),
                neutral        =('sentiment', lambda x: (x == '중립').sum()),
                press_count    =('press',     'nunique'),
                avg_similarity =('avg_similarity',  'mean'),
                avg_issue_count=('esg_match_count', 'mean'))
           .reset_index())

monthly['positive_ratio'] = (monthly['positive'] / monthly['total_articles'] * 100).round(1)
monthly['negative_ratio'] = (monthly['negative'] / monthly['total_articles'] * 100).round(1)
monthly['avg_similarity']   = monthly['avg_similarity'].round(4)
monthly['avg_issue_count']  = monthly['avg_issue_count'].round(2)

monthly = monthly[[
    'year_month', 'total_articles',
    'positive', 'negative', 'neutral',
    'positive_ratio', 'negative_ratio',
    'press_count', 'avg_issue_count', 'avg_similarity',
]]

print(f'  월별 집계 결과: {len(monthly):,}개월')

# ──────────────────────────────────────────────────────
# [Step 4] Excel 파일 작성
# ──────────────────────────────────────────────────────
print('\n[Step 4] Excel 파일 작성')
print('-'*70)

def _truncate_cell(series, max_len=32000):
    return series.astype(str).str[:max_len]

df_all_excel = df_all.copy()
if 'contents' in df_all_excel.columns:
    df_all_excel['contents'] = _truncate_cell(df_all_excel['contents'])

df_05_excel = df_05.copy()
if 'contents' in df_05_excel.columns:
    df_05_excel['contents'] = _truncate_cell(df_05_excel['contents'])

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_all_excel.to_excel(writer, sheet_name='03_NewsBody', index=False)
    print(f'  ✅ 03_NewsBody     ({len(df_all_excel):,}행)')

    df_sent.to_excel(writer, sheet_name='04_Sentiment', index=False)
    print(f'  ✅ 04_Sentiment    ({len(df_sent):,}행)')

    df_05_excel.to_excel(writer, sheet_name='05_ESGMatch', index=False)
    print(f'  ✅ 05_ESGMatch     ({len(df_05_excel):,}행, title 다음 contents)')

    monthly.to_excel(writer, sheet_name='06_MonthlyStats', index=False)
    print(f'  ✅ 06_MonthlyStats ({len(monthly):,}행)')

    # 스타일링
    wb = writer.book
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF', size=11)

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        ws.row_dimensions[1].height = 25
        ws.freeze_panes = 'A2'

        for col_idx, col in enumerate(ws.columns, 1):
            col_letter = get_column_letter(col_idx)
            header_val = ws.cell(row=1, column=col_idx).value
            if header_val == 'contents':
                ws.column_dimensions[col_letter].width = 50
            else:
                max_len = max((len(str(cell.value)) for cell in col
                              if cell.value is not None), default=10)
                ws.column_dimensions[col_letter].width = min(max_len + 2, 40)
# ──────────────────────────────────────────────────────
# [Step 5] 완료 + 로컬 자동 다운로드
# ──────────────────────────────────────────────────────
file_size_mb = os.path.getsize(output_path) / 1024 / 1024

print('\n' + '='*70)
print('🎉 Excel 통합 리포트 생성 완료!')
print('='*70)
print(f'  파일 경로 : {os.path.abspath(output_path)}')
print(f'  파일 크기 : {file_size_mb:.2f} MB')
print(f'  총 시트   : 4개')
print(f'  ⭐ 05_ESGMatch: title 바로 다음에 contents 배치')

# ── 🔽 로컬 자동 다운로드 ──────────────────────────────
print('\n' + '-'*70)
try:
    from google.colab import files
    print(f'📥 로컬로 다운로드 시작...')
    files.download(output_path)
    print(f'   브라우저 기본 다운로드 폴더를 확인하세요.')
except ImportError:
    print(f'💡 로컬 Jupyter 환경 감지 — 수동으로 파일 경로에서 다운로드하세요.')
    print(f'   경로: {os.path.abspath(output_path)}')
except Exception as e:
    print(f'⚠️ 자동 다운로드 실패: {e}')
    print(f'   수동 다운로드 경로: {os.path.abspath(output_path)}')

---
# 섹션 9 : ESG 감성지수 + 주가 회귀분석 (v3 방법론)

> 재무DB(KRFSDATA)와 주가DB(investar)가 필요합니다.

| 단계 | 내용 |
|------|------|
| 9-1 | 라이브러리 & 데이터 로드 |
| 9-2 | ESG 감성지수 계산 (유사도 가중) |
| 9-3 | 이동평균 & 시각화 |
| 9-4 | 재무정보 & 주가 로드 |
| 9-5 | 데이터 병합 |
| 9-6 | PCA 재무요인 추출 |
| 9-7 | Model 1 vs Model 2 비교 |
| 9-8 | 회귀 진단 |
| 9-9 | 시차 회귀분석 |
| 9-10 | 개별 ESG 이슈 회귀분석 |

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 23]  섹션 9-1 : 라이브러리 + DB 연결')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-1. 라이브러리 & DB 연결
# ═══════════════════════════════════════════════════════════════════
import statsmodels.api as sm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.tsa.stattools import adfuller, acf
from scipy import stats
from sqlalchemy import create_engine

# ── 분석 설정 (섹션1 변수 연결) ─────────────────────────────────
company_name      = COMPANY_NAME
stock_code        = STOCK_CODE
top_n_issues      = TOP_N_ISSUES
moving_avg_window = MOVING_AVG_WINDOW
max_lag_months    = MAX_LAG_MONTHS

_u, _p, _h = DB_USER, DB_PASSWORD, DB_HOST

# ── DB 연결 (Company_01/02/03이 cha_stock에 저장한 데이터 사용) ──
# 주가 : Company_01 → tbl_stock_price
# 재무 : Company_02 → tbl_financial
# 뉴스 : Company_03 섹션 1~8 → T_SENT, T_ESG
engine_news    = engine   # 섹션 2에서 이미 만든 cha_stock 엔진
engine_finance = engine
engine_stock   = engine

sentiment_table = T_SENT        # NEWS_SENTIMENT_LGHNH
esg_class_table = T_ESG         # NEWS_ESG_CLASS_LGHNH
STOCK_TABLE     = 'tbl_stock_price'
FIN_TABLE       = 'tbl_financial'


# ── 감성 라벨 정리 함수 (v3 동일) ───────────────────────────────
def clean_sentiment_label(label):
    if pd.isna(label): return None
    cleaned = re.sub(r'[\*\[\]\(\)_`<>]', '', str(label)).strip()
    if '긍정' in cleaned: return '긍정'
    if '부정' in cleaned: return '부정'
    if '중립' in cleaned: return '중립'
    return cleaned

# ── 뉴스 데이터 로드 (IADB) ──────────────────────────────────────
print("=" * 50)
print("1단계: 뉴스 데이터 로드 (IADB)")
print("=" * 50)
print(f"  뉴스 DB:        IADB")
print(f"  감성 테이블:    {sentiment_table}")
print(f"  ESG 분류 테이블: {esg_class_table}")

sentiment_df = pd.read_sql(f"SELECT * FROM {sentiment_table}", engine_news)
esg_class_df = pd.read_sql(f"SELECT * FROM {esg_class_table}", engine_news)

print(f"\n감성분석 데이터: {len(sentiment_df):,}건")
print(f"ESG 분류 데이터: {len(esg_class_df):,}건")
print("\n💡 데이터가 적다면 IADB_TOPIC_CODE를 확인하세요.")
print("   사용 가능한 테이블 목록:")
try:
    tbls = pd.read_sql("SHOW TABLES LIKE 'NEWS_SENTIMENT_%'", engine_news)
    print(tbls.to_string(index=False))
except:
    print("   (테이블 목록 조회 실패 — DB 연결 확인)")


In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 24]  섹션 9-1-1 : 데이터 로드 + 날짜 검증')
print('=' * 70)

sentiment_df = pd.read_sql(f"SELECT * FROM {sentiment_table}", engine_news)
esg_class_df = pd.read_sql(f"SELECT * FROM {esg_class_table}", engine_news)

print(f"\n감성분석 데이터: {len(sentiment_df):,}건")
print(f"ESG 분류 데이터: {len(esg_class_df):,}건")

# ✅ 추가: 날짜 유효성 필터
print("\n" + "=" * 50)
print("1-1단계: 날짜 유효성 검증")
print("=" * 50)

def filter_valid_dates(df, col='date',
                       min_date=START_DATE, max_date=END_DATE):
    before = len(df)
    df = df.copy()
    df[col] = df[col].astype(str).str.replace('-', '', regex=False).str.strip()
    df = df[df[col].str.len() == 8]
    df = df[df[col].str.isdigit()]
    parsed = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')
    mask = (parsed >= pd.Timestamp(min_date)) & (parsed <= pd.Timestamp(max_date))
    df = df[mask.fillna(False)]
    removed = before - len(df)
    print(f"  {before:,}건 → {len(df):,}건 (제거 {removed:,}건)")
    return df

print("감성 데이터:")
sentiment_df = filter_valid_dates(sentiment_df)
print("ESG 분류 데이터:")
esg_class_df = filter_valid_dates(esg_class_df)

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 25]  섹션 9-2 : ESG 감성지수 계산')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-2. ESG 감성지수 계산 (v3 유사도 가중 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("2단계: 상위 ESG 이슈 추출")
print("=" * 50)

top_esg_issues = (
    esg_class_df['esg_issue']
    .value_counts()
    .head(top_n_issues)
    .index
    .tolist()
)

print(f"상위 {top_n_issues}개 ESG 이슈:")
for i, issue in enumerate(top_esg_issues, 1):
    count = len(esg_class_df[esg_class_df['esg_issue'] == issue])
    print(f"  {i}. {issue}: {count:,}건")

avg_matches = esg_class_df.groupby('url').size().mean()
print(f"\n⚠️ 뉴스당 평균 ESG 이슈 매칭 수: {avg_matches:.1f}개")

# ── 4단계: 감성 점수 변환 ───────────────────────────────────────
print("\n" + "=" * 50)
print("3단계: 감성 점수 변환 및 병합")
print("=" * 50)

sentiment_df['sentiment_clean'] = sentiment_df['sentiment'].apply(clean_sentiment_label)
sentiment_map = {'긍정': 1, '중립': 0, '부정': -1}
sentiment_df['response_eval'] = sentiment_df['sentiment_clean'].map(sentiment_map)

print("감성 분포:")
print(sentiment_df['sentiment_clean'].value_counts())

sentiment_df['date'] = sentiment_df['date'].astype(str).str.replace('-', '')

esg_filtered = esg_class_df[esg_class_df['esg_issue'].isin(top_esg_issues)].copy()
esg_with_sentiment = esg_filtered.merge(
    sentiment_df[['url', 'response_eval']],
    on='url', how='left'
)
print(f"\nESG-감성 병합 데이터: {len(esg_with_sentiment):,}건")

# ── 5단계: 유사도 가중 월별 감성지수 ────────────────────────────
print("\n" + "=" * 50)
print("4단계: 유사도 가중 월별 ESG 감성 지수 계산")
print("=" * 50)

esg_with_sentiment['weighted_sentiment'] = (
    esg_with_sentiment['similarity'] * esg_with_sentiment['response_eval']
)

esg_with_sentiment['date'] = pd.to_datetime(
    esg_with_sentiment['date'].astype(str), format="%Y%m%d", errors='coerce')
esg_with_sentiment['year_month'] = esg_with_sentiment['date'].dt.to_period('M').astype(str)

monthly_by_issue = (
    esg_with_sentiment
    .groupby(['year_month', 'esg_issue'])['weighted_sentiment']
    .mean()
    .reset_index()
)

monthly_eval = monthly_by_issue.pivot(
    index='year_month', columns='esg_issue', values='weighted_sentiment'
).reset_index()
monthly_eval.columns.name = None
eval_cols = [col for col in monthly_eval.columns if col != 'year_month']

print(f"월별 ESG 감성 지수: {len(monthly_eval)}개월")
print(f"ESG 이슈 수: {len(eval_cols)}개")
print("\n월별 ESG 이슈별 감성 지수 샘플:")
print(monthly_eval.tail())

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 26]  섹션 9-3 : 이동평균 + 시각화')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-3. 이동평균 & ESG 감성지수 시각화
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("5단계: 이동평균 계산")
print("=" * 50)

monthly_eval_ma = monthly_eval.copy()
monthly_eval_ma['year_month'] = pd.to_datetime(monthly_eval_ma['year_month'])
monthly_eval_ma.set_index('year_month', inplace=True)

ma_df = monthly_eval_ma.rolling(window=moving_avg_window, min_periods=1).mean()
ma_df['Overall_Sentiment_Avg'] = ma_df[eval_cols].mean(axis=1)

print(f"{moving_avg_window}개월 이동평균 계산 완료")
print(f"전체 평균 감성 지수 범위: {ma_df['Overall_Sentiment_Avg'].min():.4f} ~ {ma_df['Overall_Sentiment_Avg'].max():.4f}")

# 📊 ESG 감성 지수 추이 시각화
plt.figure(figsize=(12, 6))
colors  = plt.cm.tab10.colors
markers = ['o', 's', '^', 'D', 'v']

for i, col in enumerate(eval_cols):
    plt.plot(ma_df.index, ma_df[col],
             label=col, color=colors[i % len(colors)],
             marker=markers[i % len(markers)],
             markersize=4, linewidth=1.5, alpha=0.8)

plt.plot(ma_df.index, ma_df['Overall_Sentiment_Avg'],
         label='Overall Average', color='black',
         linewidth=3, linestyle='--', marker='X', markersize=6)

plt.title(f'{company_name} ESG 감성 지수 추이 ({moving_avg_window}개월 이동평균, 유사도 가중)')
plt.xlabel('연월')
plt.ylabel('가중 감성 지수 (유사도 × 감성)')
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 셀 식별자 ────────────────────────────────────
print()
print('=' * 70)
print('[셀 27]  섹션 9-4 : 재무정보(통합) + 주가 로드')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-4. 재무정보 & 주가 로드 (tbl + leg 통합 버전)
#      tbl_financial을 우선 사용, 부족한 연도는 leg_financial로 보충
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("6단계: 재무정보 통합 로드 (tbl + leg)")
print("=" * 50)

# ── (1) tbl_financial: 우선 출처 (네이버 wisereport 기반) ──────────
tbl_query = f"""
SELECT * FROM tbl_financial
WHERE code = '{stock_code}'
ORDER BY year
"""
fs_tbl = pd.read_sql(tbl_query, engine_finance)
fs_tbl['_source'] = 'tbl'
print(f"  tbl_financial: {len(fs_tbl)}개 연도 "
      f"({fs_tbl['year'].min() if len(fs_tbl) else '-'} ~ "
      f"{fs_tbl['year'].max() if len(fs_tbl) else '-'})")

# ── (2) leg_financial: 보충 출처 (VALUESearch 기반) ────────────────
leg_query = f"""
SELECT * FROM leg_financial
WHERE code = '{stock_code}'
ORDER BY year
"""
fs_leg = pd.read_sql(leg_query, engine_finance)
fs_leg['_source'] = 'leg'
print(f"  leg_financial: {len(fs_leg)}개 연도 "
      f"({fs_leg['year'].min() if len(fs_leg) else '-'} ~ "
      f"{fs_leg['year'].max() if len(fs_leg) else '-'})")

# ── (3) 통합: tbl 우선, tbl에 없는 연도만 leg에서 보충 ─────────────
if len(fs_tbl) == 0 and len(fs_leg) == 0:
    raise RuntimeError(
        f"tbl_financial과 leg_financial 모두 {stock_code} 데이터 없음")

tbl_years = set(fs_tbl['year'].astype(str)) if len(fs_tbl) else set()
fs_leg_supplement = fs_leg[~fs_leg['year'].astype(str).isin(tbl_years)].copy()

fs_df = pd.concat([fs_tbl, fs_leg_supplement], ignore_index=True)
fs_df = fs_df.sort_values('year').reset_index(drop=True)

print(f"\n  통합 결과: {len(fs_df)}개 연도")
print(f"    - tbl에서 가져옴: {len(fs_tbl)}개 연도")
print(f"    - leg에서 보충함: {len(fs_leg_supplement)}개 연도 "
      f"({sorted(fs_leg_supplement['year'].tolist()) if len(fs_leg_supplement) else '없음'})")

# ── (4) 영문 → 한글 컬럼 rename (기존 셀 27과 동일) ──────────────
fs_df = fs_df.rename(columns={
    'total_assets':   '자산총계',
    'current_assets': '유동자산',
    'current_liab':   '유동부채',
    'equity':         '자기자본',
    'revenue':        '매출액',
    'net_income':     '순이익',
    'adj_price':      '수정주가',
    'adj_shares':     '수정기말발행주식수',
})
for col in ['자산총계','유동자산','유동부채','자기자본','매출액','순이익',
            '수정주가','수정기말발행주식수']:
    fs_df[col] = pd.to_numeric(fs_df[col], errors='coerce')

# ── (5) year를 정수로 변환 + 정렬 (pct_change 정확성을 위해) ─────
fs_df['year'] = fs_df['year'].astype(int)
fs_df = fs_df.sort_values('year').reset_index(drop=True)

# ── (6) 재무비율 6개 계산 (기존 셀 27과 동일) ─────────────────────
fs_df['매출액순이익률'] = (fs_df['순이익']   / fs_df['매출액'].replace(0, np.nan)) * 100
fs_df['매출액증가율']   =  fs_df['매출액'].pct_change() * 100
fs_df['유동비율']       = (fs_df['유동자산'] / fs_df['유동부채'].replace(0, np.nan)) * 100
fs_df['자기자본비율']   = (fs_df['자기자본'] / fs_df['자산총계'].replace(0, np.nan)) * 100
fs_df['총자산회전율']   =  fs_df['매출액']   /  fs_df['자산총계'].replace(0, np.nan)
시가총액 = fs_df['수정기말발행주식수'] * fs_df['수정주가']
fs_df['PBR'] = 시가총액 / (fs_df['자기자본'].replace(0, np.nan) * 1e8)
print("재무비율 계산 완료")

# ── (7) 통합 결과 검증 표시 ───────────────────────────────────────
print("\n  [통합 재무 데이터]")
display_cols = ['year', '_source', '자산총계', '매출액', '순이익',
                '자기자본', '매출액증가율', 'PBR']
display_cols = [c for c in display_cols if c in fs_df.columns]
display(fs_df[display_cols].round(2))

# ── (8) 인접 연도 단층 점검 (자기자본 기준) ──────────────────────
print("\n  [출처 경계 단층 점검]")
boundary_idx = fs_df.index[fs_df['_source'] != fs_df['_source'].shift(1)].tolist()
boundary_idx = [i for i in boundary_idx if i > 0]
if not boundary_idx:
    print("    경계 없음 (단일 출처)")
else:
    for i in boundary_idx:
        prev = fs_df.iloc[i-1]
        curr = fs_df.iloc[i]
        for col in ['자산총계', '자기자본', '매출액']:
            pv, cv = prev[col], curr[col]
            if pd.notna(pv) and pd.notna(cv) and pv != 0:
                diff = (cv - pv) / pv * 100
                mark = '✓' if abs(diff) < 30 else '⚠ 큰 변동'
                print(f"    {prev['year']}({prev['_source']}) → "
                      f"{curr['year']}({curr['_source']}) | "
                      f"{col}: {int(pv):>10,} → {int(cv):>10,} "
                      f"({diff:+6.1f}%) {mark}")

# ── (9) 📊 재무비율 추이 시각화 (기존과 동일) ────────────────────
metrics = ['매출액순이익률', '매출액증가율', '유동비율', '자기자본비율', '총자산회전율', 'PBR']
colors_fs = ['blue', 'green', 'red', 'purple', 'orange', 'brown']
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()
for i, metric in enumerate(metrics):
    axes[i].plot(fs_df['year'], fs_df[metric],
                 marker='o', color=colors_fs[i], linewidth=2)
    # 출처별 색상 점으로 구분 (옵션)
    for src, marker_color in [('tbl', '#1E3A8A'), ('leg', '#DC2626')]:
        mask = fs_df['_source'] == src
        if mask.any():
            axes[i].scatter(fs_df.loc[mask, 'year'],
                          fs_df.loc[mask, metric],
                          color=marker_color, s=50, zorder=5,
                          label=f'출처: {src}' if i == 0 else None)
    axes[i].set_title(metric)
    axes[i].set_xlabel('연도')
    axes[i].grid(True, alpha=0.3)
axes[0].legend(loc='best', fontsize=8)
plt.suptitle(f'{company_name} 재무비율 추이 (tbl=파랑, leg=빨강)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── (10) 주가 데이터 로드 (기존 셀 27 후반부와 동일) ────────────
print("\n" + "=" * 50)
print("7단계: 주가 데이터 로드")
print("=" * 50)

stock_query = f"""
SELECT date, close
FROM {STOCK_TABLE}
WHERE code = '{stock_code}'
ORDER BY date
"""
stock_df = pd.read_sql(stock_query, engine_stock)
print(f"주가 데이터: {len(stock_df):,}일")
if len(stock_df) == 0:
    raise RuntimeError(f"{STOCK_TABLE}에 {stock_code} 데이터 없음 — Company_01을 먼저 실행하세요.")

stock_df['date'] = pd.to_datetime(stock_df['date'])
stock_df['year_month'] = stock_df['date'].dt.to_period('M')
month_end_df = stock_df.sort_values('date').groupby('year_month').tail(1).copy()
print(f"월말 데이터: {len(month_end_df)}개월")

# 📊 주가 추이 시각화
plt.figure(figsize=(12, 5))
plt.plot(month_end_df['date'], month_end_df['close'],
         linestyle='--', marker='o', color='black', markersize=3)
plt.title(f'{company_name} ({stock_code}) 월말 수정주가 추이')
plt.xlabel('날짜')
plt.ylabel('수정주가')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 28]  섹션 9-5 : 데이터 병합')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-5. 데이터 병합 (v3 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("8단계: 데이터 병합")
print("=" * 50)

sentiment_ma = ma_df.reset_index()
sentiment_ma['year']     = sentiment_ma['year_month'].dt.year
sentiment_ma['year_lag'] = sentiment_ma['year'] -1  # 전년도 재무 사용

stock_merge = month_end_df.copy()
if pd.api.types.is_period_dtype(stock_merge['year_month']):
    stock_merge['year_month'] = stock_merge['year_month'].dt.to_timestamp()

merged = sentiment_ma.merge(fs_df, left_on='year_lag', right_on='year', how='inner')
merged = merged.merge(stock_merge[['year_month', 'close']], on='year_month', how='inner')

control_vars = ['자산총계', '매출액증가율', '매출액순이익률', '유동비율', '자기자본비율', 'PBR']
sentiment_vars = eval_cols + ['Overall_Sentiment_Avg']
final_vars = ['year_month', 'close'] + sentiment_vars + control_vars
final_vars = [v for v in final_vars if v in merged.columns]
final_df = merged[final_vars].dropna()

print(f"최종 분석 데이터: {len(final_df)}개월")
print(f"변수: {len(final_vars)}개")
display(final_df.head())

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 29]  섹션 9-6 : PCA 재무요인 추출')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-6. PCA 재무요인 추출 (v3 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("9단계: PCA 재무요인 추출")
print("=" * 50)

pca_vars = ['유동비율', '매출액순이익률', '총자산회전율', '자산총계', '자기자본비율', '매출액증가율']
pca_vars = [v for v in pca_vars if v in final_df.columns]

scaler_pca = StandardScaler()
X_financial_scaled = scaler_pca.fit_transform(final_df[pca_vars].values)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_financial_scaled)

final_df = final_df.copy()
final_df['PC1'] = pca_result[:, 0]
final_df['PC2'] = pca_result[:, 1]

print(f"\n[PCA 분석 결과]")
print(f"사용된 재무변수: {pca_vars}")
print(f"\n설명된 분산 비율:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.4f} ({pca.explained_variance_ratio_[0]*100:.2f}%)")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.4f} ({pca.explained_variance_ratio_[1]*100:.2f}%)")
print(f"  누적: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_)*100:.2f}%)")

loadings = pd.DataFrame(
    pca.components_.T, columns=['PC1', 'PC2'], index=pca_vars)
print(f"\n[요인적재량 (Factor Loadings)]")
display(loadings.round(4))

pc1_high = loadings['PC1'].abs().nlargest(3).index.tolist()
pc2_high = loadings['PC2'].abs().nlargest(3).index.tolist()
print(f"\n[주성분 해석]")
print(f"  PC1: {pc1_high}")
print(f"  PC2: {pc2_high}")

# 📊 Scree Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax1 = axes[0]
components = range(1, len(pca.explained_variance_ratio_) + 1)
ax1.bar(components, pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
ax1.plot(components, np.cumsum(pca.explained_variance_ratio_), 'ro-', linewidth=2)
ax1.set_xlabel('주성분'); ax1.set_ylabel('설명 분산 비율'); ax1.set_title('Scree Plot')
ax1.set_xticks(components)

ax2 = axes[1]
im = ax2.imshow(loadings.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax2.set_xticks([0, 1]); ax2.set_xticklabels(['PC1', 'PC2'])
ax2.set_yticks(range(len(pca_vars))); ax2.set_yticklabels(pca_vars)
ax2.set_title('요인적재량 히트맵')
plt.colorbar(im, ax=ax2)
for i in range(len(pca_vars)):
    for j in range(2):
        ax2.text(j, i, f'{loadings.values[i,j]:.2f}',
                 ha='center', va='center', color='black', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 30]  섹션 9-7 : Model 1 vs Model 2 비교')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-7. Model 1 vs Model 2 비교 분석 (v3 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("10단계: Model 1 vs Model 2 비교 분석")
print("=" * 50)

analysis_df = final_df.dropna(subset=['PC1', 'PC2', 'Overall_Sentiment_Avg', 'close'])

y = analysis_df['close'].values
y_scaled = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()

# Model 1: 재무요인만 (PC1, PC2)
X1 = analysis_df[['PC1', 'PC2']].values
X1_scaled = StandardScaler().fit_transform(X1)
X1_const = sm.add_constant(X1_scaled)
model1 = sm.OLS(y_scaled, X1_const).fit()

# Model 2: 재무요인 + ESG
X2 = analysis_df[['PC1', 'PC2', 'Overall_Sentiment_Avg']].values
X2_scaled = StandardScaler().fit_transform(X2)
X2_const = sm.add_constant(X2_scaled)
model2 = sm.OLS(y_scaled, X2_const).fit()

print("\n[Model 1: 재무요인만 (PC1, PC2)]")
print(f"  Adj-R²: {model1.rsquared_adj:.4f}")
print(f"  MSE:    {model1.mse_resid:.4f}")
print(f"  AIC:    {model1.aic:.4f}")
print(f"  BIC:    {model1.bic:.4f}")

print("\n[Model 2: 재무요인 + ESG (PC1, PC2, Overall_Sentiment)]")
print(f"  Adj-R²: {model2.rsquared_adj:.4f}")
print(f"  MSE:    {model2.mse_resid:.4f}")
print(f"  AIC:    {model2.aic:.4f}")
print(f"  BIC:    {model2.bic:.4f}")

# F-검정 (Nested Model Comparison)
n = len(y_scaled)
rss1, rss2 = model1.ssr, model2.ssr
df1, df2 = model1.df_resid, model2.df_resid
p_added = df1 - df2
f_stat   = ((rss1 - rss2) / p_added) / (rss2 / df2)
f_pvalue = 1 - stats.f.cdf(f_stat, p_added, df2)

print("\n[모형 비교 F-검정]")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value:     {f_pvalue:.6f}")
print(f"  MSE Ratio (Model1/Model2): {model1.mse_resid/model2.mse_resid:.4f}")

if   f_pvalue < 0.001: print("\n  ✓ Model 2 우수 (p < 0.001)")
elif f_pvalue < 0.05:  print("\n  ✓ Model 2 우수 (p < 0.05)")
elif f_pvalue < 0.1:   print("\n  △ Model 2 약한 우위 (p < 0.10)")
else:                  print("\n  △ 두 모형 간 유의미한 차이 없음")

# 📊 모형 비교 테이블
comparison_table = pd.DataFrame({
    'Metrics': ['Adjusted R²', 'MSE', 'MSE Ratio (M1/M2)', 'F-statistic', 'p-value'],
    'Model 1': [f'{model1.rsquared_adj:.4f}', f'{model1.mse_resid:.4f}', '-', '-', '-'],
    'Model 2': [f'{model2.rsquared_adj:.4f}', f'{model2.mse_resid:.4f}',
                f'{model1.mse_resid/model2.mse_resid:.4f}',
                f'{f_stat:.4f}', f'{f_pvalue:.6f}']
}).set_index('Metrics')
print("\n[표 2] 회귀모형 성능 비교 결과 (Model 1 vs Model 2)")
display(comparison_table)

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 31]  섹션 9-8 : 회귀 진단')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-8. 회귀 진단 (v3 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("11단계: Model 2 회귀 진단 검정")
print("=" * 50)

# [1] VIF
print("\n[1] VIF (분산팽창계수) - 다중공선성 검정")
vif_data = pd.DataFrame({
    'Variable': ['PC1', 'PC2', 'Overall_Sentiment_Avg'],
    'VIF': [variance_inflation_factor(X2_scaled, i) for i in range(X2_scaled.shape[1])]
})
display(vif_data)
print("  ✓ 모든 VIF < 10: 다중공선성 문제 없음" if vif_data['VIF'].max() < 10
      else "  ✗ VIF > 10인 변수 있음: 다중공선성 우려")

# [2] Breusch-Pagan
print("\n[2] Breusch-Pagan 검정 - 등분산성")
bp = het_breuschpagan(model2.resid, model2.model.exog)
print(f"  LM Statistic: {bp[0]:.4f}  p-value: {bp[1]:.4f}")
print("  ✓ p > 0.1: 등분산성 충족" if bp[1] > 0.1 else "  ✗ p ≤ 0.1: 이분산성 우려")

# [3] Jarque-Bera
print("\n[3] Jarque-Bera 검정 - 잔차 정규성")
jb = jarque_bera(model2.resid)
print(f"  JB Statistic: {jb[0]:.4f}  p-value: {jb[1]:.4f}")
print("  ✓ p > 0.1: 정규성 충족" if jb[1] > 0.1 else "  △ p ≤ 0.1: 정규성 위반 가능성")

# [4] Durbin-Watson
print("\n[4] Durbin-Watson 검정 - 잔차 독립성")
dw = durbin_watson(model2.resid)
print(f"  DW Statistic: {dw:.4f}")
if 1.5 < dw < 2.5:   print("  ✓ DW ≈ 2: 자기상관 없음")
elif dw < 1.5:        print("  △ DW < 1.5: 양의 자기상관 가능성 → ACF 확인 필요")
else:                 print("  △ DW > 2.5: 음의 자기상관 가능성")

# [5] ADF
print("\n[5] ADF 검정 - 시계열 정상성")
adf = adfuller(model2.resid)
print(f"  ADF Statistic: {adf[0]:.4f}  p-value: {adf[1]:.4f}")
print("  ✓ p < 0.05: 정상성 충족" if adf[1] < 0.05 else "  ✗ p ≥ 0.05: 비정상 시계열 우려")

# 📊 진단 플롯 (2×2)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax1 = axes[0,0]
ax1.scatter(model2.fittedvalues, model2.resid, alpha=0.6)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_xlabel('Fitted Values'); ax1.set_ylabel('Residuals')
ax1.set_title('잔차 vs 적합값 (등분산성 확인)')

ax2 = axes[0,1]
stats.probplot(model2.resid, dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot (정규성 확인)')

ax3 = axes[1,0]
ax3.hist(model2.resid, bins=15, edgecolor='black', alpha=0.7)
ax3.set_xlabel('Residuals'); ax3.set_ylabel('Frequency')
ax3.set_title('잔차 분포 (정규성 확인)')

ax4 = axes[1,1]
acf_values = acf(model2.resid, nlags=12)
ax4.bar(range(len(acf_values)), acf_values, color='steelblue')
ci = 1.96/np.sqrt(len(model2.resid))
ax4.axhline( ci, color='red', linestyle='--', label='95% CI')
ax4.axhline(-ci, color='red', linestyle='--')
ax4.set_xlabel('Lag'); ax4.set_ylabel('ACF'); ax4.set_title('자기상관함수 (ACF)')
ax4.legend()

plt.tight_layout()
plt.show()

# VIF 상세 + Model 2 summary
print("\n[표 3] 독립변수별 VIF")
display(vif_data.set_index('Variable'))
print("\n[Model 2 회귀분석 상세 결과]")
print(model2.summary())

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 32]  섹션 9-9 : 시차 회귀분석')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# ★ 회귀 분석 기간 조정 (REG_START_OFFSET 적용)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("회귀 분석 기간 설정")
print("=" * 50)

# 안전장치: REG_START_OFFSET이 정의되지 않으면 0으로
try:
    _offset = int(REG_START_OFFSET)
except NameError:
    _offset = 0
    print("  ⚠ REG_START_OFFSET 미정의 → 0으로 자동 설정")

# final_df 기준 분석 가능 범위
data_start = pd.to_datetime(final_df['year_month']).min()
data_end   = pd.to_datetime(final_df['year_month']).max()
data_first_year = data_start.year

# 요청된 시작 연도
requested_start_year = data_first_year + _offset

# 데이터 범위 밖이면 자동 보정
if _offset < 0:
    actual_start_year = data_first_year
    print(f"  ⚠ 음수 오프셋({_offset:+d}) → 데이터 시작 연도({data_first_year})로 자동 보정")
elif requested_start_year > data_end.year:
    actual_start_year = data_first_year
    print(f"  ⚠ 오프셋({_offset:+d})이 너무 커서 데이터 끝({data_end.year}) 초과 → 전체 사용으로 보정")
else:
    actual_start_year = requested_start_year

# 분석 대상 final_df 슬라이싱
analysis_start = pd.Timestamp(year=actual_start_year, month=1, day=1)
mask = pd.to_datetime(final_df['year_month']) >= analysis_start
final_df_reg = final_df[mask].copy().reset_index(drop=True)

print(f"\n  설정 오프셋:        REG_START_OFFSET = {_offset:+d}")
print(f"  데이터 전체 범위:   {data_start:%Y-%m} ~ {data_end:%Y-%m} ({len(final_df)}개월)")
print(f"  회귀 분석 범위:     {analysis_start:%Y-%m} ~ {data_end:%Y-%m} ({len(final_df_reg)}개월)")
print(f"  제외된 앞부분:      {len(final_df) - len(final_df_reg)}개월")

if len(final_df_reg) < 24:
    print(f"\n  ⚠ 경고: 회귀 분석 표본이 {len(final_df_reg)}개월밖에 없음")
    print(f"     시차 분석은 시차마다 표본이 더 줄어드므로 결과 신뢰성 낮을 수 있음")

# ═══════════════════════════════════════════════════════════════════
# 9-9. 시차 회귀분석 (v3 방식, 기존 로직 그대로)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print(f"12단계: 시차 회귀분석 (PC1, PC2 + ESG 감성지수)")
print(f"  분석 기간: {analysis_start:%Y-%m} ~ {data_end:%Y-%m}")
print("=" * 50)

control_vars_pca  = ['PC1', 'PC2']
explanatory_vars  = ['Overall_Sentiment_Avg']

results_model1, results_model2 = [], []

for lag in range(0, max_lag_months + 1):
    df_shifted = final_df_reg.copy()    # ← 기존 final_df → final_df_reg 로 변경
    df_shifted['close_lag'] = df_shifted['close'].shift(-lag)
    df_lagged = df_shifted.dropna(
        subset=control_vars_pca + explanatory_vars + ['close_lag'])

    if len(df_lagged) < 10:
        continue

    y_l = df_lagged['close_lag'].values
    y_l_sc = StandardScaler().fit_transform(y_l.reshape(-1,1)).ravel()

    X1_l = StandardScaler().fit_transform(df_lagged[control_vars_pca].values)
    m1 = sm.OLS(y_l_sc, sm.add_constant(X1_l)).fit()

    X2_l = StandardScaler().fit_transform(
        df_lagged[control_vars_pca + explanatory_vars].values)
    m2 = sm.OLS(y_l_sc, sm.add_constant(X2_l)).fit()

    results_model1.append({'lag': f'M+{lag}', 'adj_r2': m1.rsquared_adj, 'mse': m1.mse_resid})
    results_model2.append({'lag': f'M+{lag}', 'adj_r2': m2.rsquared_adj, 'mse': m2.mse_resid,
                           'esg_coef': m2.params[-1], 'esg_pval': m2.pvalues[-1]})

result_m1 = pd.DataFrame(results_model1).set_index('lag')
result_m2 = pd.DataFrame(results_model2).set_index('lag')

print("\n[시차별 ESG 감성지수 회귀계수 및 유의확률]")
display(result_m2[['esg_coef', 'esg_pval', 'adj_r2', 'mse']].round(4))

# 📊 시차별 비교 테이블
comparison_df = pd.DataFrame({
    'M1_Adj_R²': result_m1['adj_r2'],
    'M2_Adj_R²': result_m2['adj_r2'],
    'R²_향상':   result_m2['adj_r2'] - result_m1['adj_r2'],
    'ESG_Coef':  result_m2['esg_coef'],
    'ESG_pval':  result_m2['esg_pval']
})
print("\n[시차별 Model 1 vs Model 2 성능 비교]")
display(comparison_df.round(4))

# 📊 시각화
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

ax1 = axes[0]
x = range(len(result_m2))
bar_colors = ['blue' if c >= 0 else 'red' for c in result_m2['esg_coef']]
ax1.bar(x, result_m2['esg_coef'], color=bar_colors, alpha=0.7)
ax1.set_ylabel('ESG 표준화 회귀계수', fontsize=12)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_xticks(x); ax1.set_xticklabels(result_m2.index, rotation=45)
ax1_twin = ax1.twinx()
ax1_twin.plot(x, result_m2['esg_pval'], color='gray', linestyle='--', marker='o', linewidth=2)
ax1_twin.axhline(0.05, color='red',    linestyle=':', alpha=0.5, label='p=0.05')
ax1_twin.axhline(0.10, color='orange', linestyle=':', alpha=0.5, label='p=0.10')
ax1_twin.set_ylabel('p-value', fontsize=12)
ax1_twin.legend(loc='upper right')
# ★ 제목에 분석 기간 명시
ax1.set_title(
    f'{company_name} ESG 감성지수 → 주가 영향 (시차별 회귀계수)\n'
    f'분석 기간: {analysis_start:%Y-%m} ~ {data_end:%Y-%m} ({len(final_df_reg)}개월, offset={_offset:+d})',
    fontsize=13)

ax2 = axes[1]
width = 0.35
x2 = np.arange(len(result_m1))
ax2.bar(x2 - width/2, result_m1['adj_r2'], width, label='Model 1 (재무만)', color='lightgray')
ax2.bar(x2 + width/2, result_m2['adj_r2'], width, label='Model 2 (재무+ESG)', color='steelblue')
ax2.set_ylabel('Adjusted R²', fontsize=12)
ax2.set_xlabel('시차 (개월)', fontsize=12)
ax2.set_xticks(x2); ax2.set_xticklabels(result_m1.index, rotation=45)
ax2.legend()
ax2.set_title('Model 1 vs Model 2 설명력 비교', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 33]  섹션 9-10 : 개별 ESG 이슈 회귀분석')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════
# 9-10. 개별 ESG 이슈 회귀분석 (v3 방식)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 50)
print("13단계: 개별 ESG 이슈 회귀분석")
print("=" * 50)

individual_vars = [col for col in eval_cols if col in final_df.columns]
control_vars_full = ['자산총계', '매출액증가율', '매출액순이익률', '유동비율', '자기자본비율', 'PBR']
control_vars_full = [c for c in control_vars_full if c in final_df.columns]

if len(individual_vars) > 0:
    reg_df = final_df.copy()
    X = reg_df[control_vars_full + individual_vars]
    y = reg_df['close']
    scaler_ind = StandardScaler()
    X_scaled_ind = scaler_ind.fit_transform(X)
    y_scaled_ind  = StandardScaler().fit_transform(y.values.reshape(-1,1)).ravel()
    X_const_ind   = sm.add_constant(X_scaled_ind)
    model_ind = sm.OLS(y_scaled_ind, X_const_ind).fit()
    print(model_ind.summary())
else:
    print("개별 ESG 이슈 변수가 없습니다.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 9-11. ESG와 주가의 상관 분석
# ═══════════════════════════════════════════════════════════════════

# ─── 셀 식별자 ────────────────────────────────────
print()
print('=' * 70)
print('[시각화] 시차별 ESG ↔ 주가 산점도 격자 (M+0 ~ M+12)')
print('=' * 70)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# ═══════════════════════════════════════════════════════════════════
# 시차별 산점도 13개 (M+0 ~ M+12)
# - x축: 현재 시점 t의 ESG 감성지수
# - y축: t+lag 시점의 주가 (lag만큼 미래 주가)
# - 즉 lag=0은 동시, lag=12는 12개월 후 주가
# ═══════════════════════════════════════════════════════════════════

df_v = final_df.copy().sort_values('year_month').reset_index(drop=True)
df_v['ts'] = pd.to_datetime(df_v['year_month'])

esg = df_v['Overall_Sentiment_Avg'].values
prc = df_v['close'].values
ts  = df_v['ts'].values

# ── 13개 산점도 격자 (4 × 4, 마지막 3칸은 비움) ─────────────────
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes_flat = axes.flatten()

# 시차별 상관계수 미리 계산 (제목에 표시)
lag_results = []
for lag in range(13):
    if lag == 0:
        e_l, p_l, ts_l = esg, prc, ts
    else:
        e_l = esg[:len(esg) - lag]
        p_l = prc[lag:]
        ts_l = ts[lag:]
    valid = ~(np.isnan(e_l) | np.isnan(p_l))
    if valid.sum() >= 3:
        r, pv = stats.pearsonr(e_l[valid], p_l[valid])
        lag_results.append({
            'lag': lag, 'r': r, 'p': pv, 'n': int(valid.sum()),
            'e': e_l, 'p_arr': p_l, 'ts': ts_l, 'valid': valid
        })

# x축, y축 범위 통일 (모든 그래프를 같은 스케일로 비교)
all_e = np.concatenate([r['e'][r['valid']] for r in lag_results])
all_p = np.concatenate([r['p_arr'][r['valid']] for r in lag_results])
e_min, e_max = all_e.min() * 0.98, all_e.max() * 1.02
p_min, p_max = all_p.min() * 0.95, all_p.max() * 1.05

# r값에 따른 색상 (음수=빨강, 양수=파랑)
def get_title_color(r, p):
    if p < 0.05:
        return '#DC2626' if r < 0 else '#1E3A8A'   # 유의: 진한 색
    elif p < 0.10:
        return '#F87171' if r < 0 else '#60A5FA'   # 약유의: 중간 색
    else:
        return '#9CA3AF'                            # 무의미: 회색

# 13개 시차 그리기
for i, res in enumerate(lag_results):
    ax = axes_flat[i]
    lag = res['lag']
    e_arr = res['e']
    p_arr = res['p_arr']
    ts_arr = res['ts']
    valid  = res['valid']

    # 시간 흐름 색상 (오래됨=옅음, 최근=진함)
    n_pts = valid.sum()
    scatter = ax.scatter(e_arr[valid], p_arr[valid],
                         c=range(n_pts), cmap='viridis',
                         s=35, alpha=0.75,
                         edgecolors='white', linewidth=0.4)

    # OLS 회귀선
    if valid.sum() >= 2:
        z = np.polyfit(e_arr[valid], p_arr[valid], 1)
        xr = np.linspace(e_arr[valid].min(), e_arr[valid].max(), 50)
        ax.plot(xr, z[0]*xr + z[1],
                color='red', linestyle='--', linewidth=1.8,
                alpha=0.85)

    # 통일된 축
    ax.set_xlim(e_min, e_max)
    ax.set_ylim(p_min, p_max)
    ax.grid(True, alpha=0.3)

    # 제목 (시차, r, p, n)
    sig_mark = '✓' if res['p'] < 0.05 else ('△' if res['p'] < 0.10 else '')
    title_color = get_title_color(res['r'], res['p'])
    title = (f"M+{lag}  |  r={res['r']:+.3f}  p={res['p']:.3f}  {sig_mark}\n"
             f"n={res['n']}")
    ax.set_title(title, fontsize=11, fontweight='bold',
                 color=title_color, pad=6)

    # 행/열에 따른 레이블 (외곽만)
    if i % 4 == 0:
        ax.set_ylabel(f'주가 (t+{lag})', fontsize=9)
    if i >= 9:    # 하단 행
        ax.set_xlabel('ESG (t)', fontsize=9)

    ax.tick_params(axis='both', labelsize=8)

# 사용 안 하는 칸 숨기기
for j in range(len(lag_results), len(axes_flat)):
    axes_flat[j].set_visible(False)

# 마지막 칸에 시차별 r 변화 그래프 (요약)
ax_summary = fig.add_subplot(4, 4, 16)
lags = [r['lag'] for r in lag_results]
rs   = [r['r']   for r in lag_results]
ps   = [r['p']   for r in lag_results]
colors = [get_title_color(r, p) for r, p in zip(rs, ps)]

bars = ax_summary.bar(lags, rs, color=colors, alpha=0.85,
                      edgecolor='black', linewidth=0.5)
ax_summary.axhline(0, color='black', linewidth=0.8)
ax_summary.set_xlabel('시차 (M+k)', fontsize=10)
ax_summary.set_ylabel('Pearson r', fontsize=10)
ax_summary.set_title('시차별 상관계수 변화', fontsize=11, fontweight='bold')
ax_summary.set_xticks(lags)
ax_summary.set_xticklabels([f'M+{l}' for l in lags], rotation=45, fontsize=8)
ax_summary.grid(True, alpha=0.3, axis='y')

# 유의수준 표시
ax_summary.axhline(0, color='black', linewidth=0.5)
for bar, r, p in zip(bars, rs, ps):
    if p < 0.05:
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

# 전체 제목 + 컬러바 설명
plt.suptitle(
    f'{company_name} ({stock_code}) — 시차별 ESG ↔ 주가 산점도 (M+0 ~ M+12)\n'
    f'(점 색: 시간 흐름 / 빨강 제목: 음의 상관·유의 / 파랑 제목: 양의 상관·유의 / 회색: 유의 X)',
    fontsize=14, fontweight='bold', y=1.00
)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════
# 텍스트 요약 — 부호 전환 시점 자동 탐지
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"시차별 상관계수 추이 — 부호 전환 점검")
print(f"{'='*60}\n")

print(f"  {'시차':<6} {'r':>10} {'p':>10} {'n':>5}  {'부호':<8}  {'유의성'}")
print(f"  {'─'*6} {'─'*10} {'─'*10} {'─'*5}  {'─'*8}  {'─'*8}")

prev_sign = None
sign_change_lags = []
for res in lag_results:
    sign = 'POS (+)' if res['r'] > 0 else 'NEG (-)'
    sig  = '✓ 유의'   if res['p'] < 0.05 else ('△ 약유의' if res['p'] < 0.10 else '— 무의미')

    # 부호 전환 표시
    transition = ''
    if prev_sign is not None and prev_sign != ('POS' if res['r']>0 else 'NEG'):
        transition = '  ◀ 부호 전환!'
        sign_change_lags.append(res['lag'])
    prev_sign = 'POS' if res['r']>0 else 'NEG'

    print(f"  M+{res['lag']:<3d} {res['r']:>+10.4f} {res['p']:>10.4f} "
          f"{res['n']:>5d}  {sign:<8}  {sig}{transition}")

print()
if sign_change_lags:
    print(f"  ⚡ 부호 전환 시점: M+{', M+'.join(map(str, sign_change_lags))}")
    print(f"     → ESG와 주가의 관계가 이 시차에서 방향이 바뀜")
else:
    print(f"  → 모든 시차에서 부호가 일관됨 (전환 없음)")

# ESG가 양/음의 영향을 주는 구간 정리
pos_lags = [r['lag'] for r in lag_results if r['r'] > 0 and r['p'] < 0.10]
neg_lags = [r['lag'] for r in lag_results if r['r'] < 0 and r['p'] < 0.10]
print(f"\n  유의(p<0.10)한 양의 상관 시차: {pos_lags if pos_lags else '없음'}")
print(f"  유의(p<0.10)한 음의 상관 시차: {neg_lags if neg_lags else '없음'}")

# ═══════════════════════════════════════════════════════════════════
# ★ HTML 보고서용 base64 변환 + 요약 데이터 저장 (추가)
# ═══════════════════════════════════════════════════════════════════
import io as _io_fig3
import base64 as _b64_fig3

# 위에서 만든 fig 객체를 base64로 변환
buf_fig3 = _io_fig3.BytesIO()
fig.savefig(buf_fig3, format='png', dpi=110, bbox_inches='tight', facecolor='white')
buf_fig3.seek(0)
fig3_b64 = _b64_fig3.b64encode(buf_fig3.read()).decode('utf-8')

# HTML 표·텍스트용 요약 데이터
_best = max(lag_results, key=lambda x: abs(x['r']))
fig3_summary = {
    'lag_results': [
        {
            'lag':  int(r['lag']),
            'r':    float(r['r']),
            'p':    float(r['p']),
            'n':    int(r['n']),
            'sign': '+' if r['r'] > 0 else '-',
            'sig':  ('high' if r['p'] < 0.05
                     else ('weak' if r['p'] < 0.10 else 'none')),
        }
        for r in lag_results
    ],
    'sign_change_lags': [int(l) for l in sign_change_lags],
    'pos_lags':         [int(l) for l in pos_lags],
    'neg_lags':         [int(l) for l in neg_lags],
    'best_lag':         int(_best['lag']),
    'best_r':           float(_best['r']),
    'best_p':           float(_best['p']),
}

print(f"\n  ✓ HTML 보고서용 fig3_b64 변환 완료: {len(fig3_b64):,}자")
print(f"  ✓ HTML 보고서용 fig3_summary 저장 완료 ({len(fig3_summary['lag_results'])}개 시차)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 9-12. 논문 형식 HTML을 위한 paper_data 패키징
# ═══════════════════════════════════════════════════════════════════
import io as _io
import base64 as _b64
import matplotlib.pyplot as plt

# ── base64 변환 헬퍼 ─────────────────────────────────────────────
def _fig_to_b64(fig):
    buf = _io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    return _b64.b64encode(buf.read()).decode('utf-8')

# ── 그림 1: 12개월 이동평균 PNG ─────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(10, 4.5), facecolor='white')
for col in eval_cols:
    if col in ma_df.columns:
        ax1.plot(ma_df.index, ma_df[col], label=col[:30], linewidth=1.6, alpha=0.85)
ax1.plot(ma_df.index, ma_df['Overall_Sentiment_Avg'],
         label='전체 평균', color='black', linewidth=2.2, linestyle='--')
ax1.set_xlabel('연월', fontsize=10)
ax1.set_ylabel('감성지수 (12M MA)', fontsize=10)
ax1.legend(loc='best', fontsize=8, ncol=2)
ax1.grid(alpha=0.3)
plt.tight_layout()
fig1_b64 = _fig_to_b64(fig1)
plt.close(fig1)

# ── 그림 2: 시차별 회귀계수 PNG (분석기간 명시 + 이중축) ─────
fig2, ax2 = plt.subplots(figsize=(11, 4.5), facecolor='white')

x = list(range(len(results_model2)))
coefs  = [r['esg_coef'] for r in results_model2]
pvals  = [r['esg_pval'] for r in results_model2]
labels = [f"M+{i}" for i in range(len(results_model2))]

# 회귀계수 막대 (양수 파랑 / 음수 빨강)
bar_colors = ['#3b82f6' if c >= 0 else '#ef4444' for c in coefs]
ax2.bar(x, coefs, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('ESG 표준화 회귀계수', fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=45, fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# p-value 점선 (이중축)
ax2_twin = ax2.twinx()
ax2_twin.plot(x, pvals, color='#6b7280', linestyle='--',
              marker='o', linewidth=2, markersize=5)
ax2_twin.axhline(0.05, color='#dc2626',    linestyle=':', alpha=0.6, label='p=0.05')
ax2_twin.axhline(0.10, color='#f59e0b',    linestyle=':', alpha=0.6, label='p=0.10')
ax2_twin.set_ylabel('p-value', fontsize=11, fontweight='bold')
ax2_twin.legend(loc='upper right', fontsize=9)

# 제목 — 분석 기간 명시 (셀 32에서 만든 변수 사용; 없으면 안전하게 fallback)
try:
    title_period = (f'분석 기간: {analysis_start:%Y-%m} ~ {data_end:%Y-%m} '
                    f'({len(final_df_reg)}개월, offset={_offset:+d})')
except NameError:
    # 셀 32 변수가 없으면 final_df 기준으로 대체
    _ts = pd.to_datetime(final_df['year_month'])
    title_period = f'분석 기간: {_ts.min():%Y-%m} ~ {_ts.max():%Y-%m} ({len(final_df)}개월)'

ax2.set_title(
    f'{company_name} ESG 감성지수 → 주가 영향 (시차별 회귀계수)\n{title_period}',
    fontsize=12, fontweight='bold', pad=12
)

plt.tight_layout()
fig2_b64 = _fig_to_b64(fig2)
plt.close(fig2)


# ── 표 1: 데이터 수집 및 감성 분류 결과 ───────────────────────────
# ✅ s10_sent → sentiment_df, s10_esg → esg_class_df
sent_dist = sentiment_df['sentiment_clean'].value_counts()

table1 = {
    'period':       f"{START_DATE} ~ {END_DATE}",
    'total_news':   len(sentiment_df),
    'esg_related':  sentiment_df['url'].nunique(),  # 고유 기사 수
    'sent_valid':   len(sentiment_df),
    'pos_count':    int(sent_dist.get('긍정', 0)),
    'neg_count':    int(sent_dist.get('부정', 0)),
    'neu_count':    int(sent_dist.get('중립', 0)),
    'esg_match':    len(esg_class_df),               # 매칭 총 건수 (1기사가 여러 이슈 매핑 가능)
    'avg_per_news': round(len(esg_class_df) / max(sentiment_df['url'].nunique(), 1), 1),
}
table1['esg_related_pct'] = round(table1['esg_related'] / max(table1['total_news'], 1) * 100, 1)

# ── 표 2: 상위 ESG 이슈 ───────────────────────────────────────────
# ✅ top_esg_issues는 이미 9-2에서 정의됨
issue_counts = esg_class_df['esg_issue'].value_counts().head(min(5, len(top_esg_issues)))
table2 = [{'rank': i+1, 'issue': iss, 'count': int(cnt)}
          for i, (iss, cnt) in enumerate(issue_counts.items())]

# ── 표 3: Model 1 vs Model 2 ─────────────────────────────────────
# ✅ model1, model2는 statsmodels OLS 결과 객체
# ✅ f_stat, f_pvalue는 개별 변수
table3 = {
    'adj_r2_m1':  round(model1.rsquared_adj, 4),
    'adj_r2_m2':  round(model2.rsquared_adj, 4),
    'mse_m1':     round(model1.mse_resid, 4),
    'mse_m2':     round(model2.mse_resid, 4),
    'aic_m1':     round(model1.aic, 2),
    'aic_m2':     round(model2.aic, 2),
    'bic_m1':     round(model1.bic, 2),
    'bic_m2':     round(model2.bic, 2),
    'mse_ratio':  round(model1.mse_resid / model2.mse_resid, 4),
    'f_stat':     round(f_stat, 4),
    'f_pval':     round(f_pvalue, 6),
}

# ── 표 4: 시차별 회귀계수 ─────────────────────────────────────────
# ✅ results_model2는 list of dict [{'lag', 'adj_r2', 'mse', 'esg_coef', 'esg_pval'}, ...]
table4 = []
for r in results_model2:
    p = r['esg_pval']
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else 'n.s.'
    table4.append({
        'lag':    r['lag'],                # 이미 'M+0', 'M+1' 형식
        'coef':   round(r['esg_coef'], 4),
        'pval':   round(p, 4),
        'adj_r2': round(r['adj_r2'], 4),
        'sig':    sig,
    })

# ── 표 5: 회귀 진단 ───────────────────────────────────────────────
# ✅ bp, jb는 튜플 (LM/JB stat, p-value, ...)
# ✅ dw는 단일 float
# ✅ adf는 튜플 (ADF stat, p-value, ...)
# ✅ vif_data는 DataFrame {'Variable', 'VIF'}
table5 = [
    {'name': 'VIF (다중공선성)',
     'stat': f"PC1={vif_data.iloc[0]['VIF']:.2f}, PC2={vif_data.iloc[1]['VIF']:.2f}, Sent={vif_data.iloc[2]['VIF']:.2f}",
     'result': '문제 없음' if vif_data['VIF'].max() < 10 else '다중공선성 의심',
     'pass':   bool(vif_data['VIF'].max() < 10)},
    {'name': 'Breusch-Pagan (등분산성)',
     'stat': f"LM={bp[0]:.2f}, p={bp[1]:.4f}",
     'result': '등분산성 충족' if bp[1] >= 0.05 else '이분산성 존재',
     'pass':   bool(bp[1] >= 0.05)},
    {'name': 'Jarque-Bera (정규성)',
     'stat': f"JB={jb[0]:.2f}, p={jb[1]:.4f}",
     'result': '정규성 충족' if jb[1] >= 0.05 else '정규성 위반',
     'pass':   bool(jb[1] >= 0.05)},
    {'name': 'Durbin-Watson (독립성)',
     'stat': f"DW={dw:.4f}",
     'result': '독립성 충족' if 1.5 <= dw <= 2.5 else '자기상관 가능성',
     'pass':   bool(1.5 <= dw <= 2.5)},
    {'name': 'ADF (정상성)',
     'stat': f"ADF={adf[0]:.2f}, p={adf[1]:.4f}",
     'result': '정상성 충족' if adf[1] < 0.05 else '단위근 존재',
     'pass':   bool(adf[1] < 0.05)},
]

# ── 메타 정보 ────────────────────────────────────────────────────
# ✅ company_name은 소문자 (셀 29에서 사용 확인)
# ✅ stock_code는 9-4에서 정의됨
meta = {
    'company':         company_name if 'company_name' in globals() else COMPANY_NAME,
    'stock_code':      stock_code if 'stock_code' in globals() else TOPIC_CODE,
    'period':          f"{START_DATE} ~ {END_DATE}",
    'months':          len(monthly_eval),
    'embedding_model': 'text-embedding-3-large (3,072D)',
    'sentiment_model': 'GPT-4o-mini',
    'analysis_date':   pd.Timestamp.now().strftime('%Y년 %m월 %d일'),
}

# ── 핵심 발견 ─────────────────────────────────────────────────────
sig_lags = [r for r in results_model2 if r['esg_pval'] < 0.10]
key_findings = {
    'r2_improvement':  round((table3['adj_r2_m2'] - table3['adj_r2_m1']) * 100, 2),
    'sig_lag_count':   len(sig_lags),
    'pos_ratio':       round(table1['pos_count'] / max(table1['sent_valid'], 1) * 100, 1),
    'top_issue':       table2[0]['issue'] if table2 else '',
    'top_issue_count': table2[0]['count']  if table2 else 0,
}

# ── 통합 dict ─────────────────────────────────────────────────────
paper_data = {
    'meta':            meta,
    'table1':          table1,
    'table2':          table2,
    'table3':          table3,
    'table4':          table4,
    'table5':          table5,
    'figure1_b64':     fig1_b64,
    'figure2_b64':     fig2_b64,
    'figure3_b64':     fig3_b64,
    'figure3_summary': fig3_summary,
    'key_findings':    key_findings,
}

# ── 결과 검증 ─────────────────────────────────────────────────────
print('=' * 60)
print('✅ paper_data 패키징 완료')
print('=' * 60)
print(f'  표 1 (수집 결과)    : 총 {table1["total_news"]:,}건, 긍/부/중={table1["pos_count"]}/{table1["neg_count"]}/{table1["neu_count"]}')
print(f'  표 2 (TOP ESG 이슈) : {len(table2)}개 (1위: {table2[0]["issue"][:30] if table2 else "N/A"})')
print(f'  표 3 (Model 비교)   : Adj-R² {table3["adj_r2_m1"]} vs {table3["adj_r2_m2"]} (F={table3["f_stat"]}, p={table3["f_pval"]:.4f})')
print(f'  표 4 (시차별 계수)  : {len(table4)}개 시차')
print(f'  표 5 (회귀 진단)    : {len(table5)}개 검정')
print(f'  그림 1 (이동평균)   : {len(fig1_b64):,}자 base64')
print(f'  그림 2 (회귀계수)   : {len(fig2_b64):,}자 base64')
print(f'  그림 3 (시차 산점도): {len(fig3_b64):,}자 base64')
print(f'  요약 (시차별 r)     : {len(fig3_summary["lag_results"])}개 시차')
print(f'  핵심 발견           : R²개선 +{key_findings["r2_improvement"]}%p, 유의 시차 {key_findings["sig_lag_count"]}개')

---
# 섹션 10 : 주제분석 + 네트워크 + HTML 스토리북

In [ ]:
COLORS=['#FDE047','#86EFAC','#7DD3FC','#FF6B9D','#A78BFA','#C084FC']
EDGE_COLORS=['#CA8A04','#16A34A','#0369A1','#BE185D','#7C3AED','#9333EA']
SHAPES=['o','s','^','D','v','p']

viz_params = {
    # ── 해상도 (300 DPI 이상) ─────────────────────────────────
    'figure_dpi':       300,        # 100 → 300 (인쇄/논문 품질)
    'figure_width':     16,         # 22 → 16 (DPI 상향에 맞춰 적정화)
    'figure_height':    12,         # 18 → 12

    # ── 노드 크기 (DPI 비례 확대) ─────────────────────────────
    'node_size_min':    1500,       # 800 → 1500
    'node_size_max':    6500,       # 4000 → 6500

    # ── 엣지 (선) 설정 ───────────────────────────────────────
    'edge_threshold':   0.05,
    'edge_width_min':   3.0,        # 2.0 → 3.0 (시인성 강화)
    'edge_width_max':   12.0,       # 8.0 → 12.0
    'edge_alpha_min':   0.4,
    'edge_alpha_max':   0.9,

    # ── 레이아웃 ─────────────────────────────────────────────
    'cluster_radius':   4.5,
    'min_node_distance': 1.8,

    # ── 폰트 크기 (대폭 확대) ────────────────────────────────
    'label_fontsize':   18,         # 13 → 18 (노드 라벨)
    'label_fontweight': 'bold',     # 신규 (가독성 강화)
    'title_fontsize':   22,         # 16 → 22 (그래프 제목)
    'legend_fontsize':  14,         # 8 → 14 (범례)
    'edge_linewidth':   3.5,        # 신규 (노드 테두리)

    # ── 출력 ─────────────────────────────────────────────────
    'savefig_dpi':      300,        # base64 저장 시 DPI
}

# ── 헬퍼: Keyword_JSON 파싱 ──────────────────────────────────────
def _parse_kj(raw):
    """Keyword_JSON 컬럼 → dict 반환. 실패 시 빈 dict."""
    if isinstance(raw, dict):
        kw = raw.get('keywords', {})
        return kw if isinstance(kw, dict) else {}
    if isinstance(raw, str):
        try:
            parsed = json.loads(raw)
            kw = parsed.get('keywords', {})
            return kw if isinstance(kw, dict) else {}
        except:
            try:
                parsed = ast.literal_eval(raw)
                kw = parsed.get('keywords', {})
                return kw if isinstance(kw, dict) else {}
            except:
                return {}
    return {}

# ── 변경 1: do_clustering — Keyword_JSON 기반 키워드 집계 ─────────
def do_clustering(df_data, n=4):
    embs = np.array(df_data['emb'].tolist())
    n_adj = max(min(n, len(df_data)//5, 6), 2)
    km = KMeans(n_clusters=n_adj, random_state=42, n_init=10)
    df_data = df_data.copy()
    df_data['cluster'] = km.fit_predict(embs)
    clusters = {}
    for cid in range(n_adj):
        cdf = df_data[df_data['cluster'] == cid]

        # Keyword_JSON에서 키워드 빈도 집계
        kw_counter = defaultdict(float)
        for _, r in cdf.iterrows():
            kw = _parse_kj(r.get('Keyword_JSON', '{}'))
            for word, freq in kw.items():
                kw_counter[word] += float(freq)

        sorted_kw = sorted(kw_counter.items(), key=lambda x: -x[1])
        top_kw = [w for w, _ in sorted_kw[:15]]
        kw_sc  = {w: s for w, s in sorted_kw[:15]}

        clusters[cid] = {
            'df': cdf, 'size': len(cdf),
            'keywords': top_kw, 'kw_scores': kw_sc,
            'titles': cdf['title'].head(5).tolist()
        }
    return {'clusters': clusters, 'n': n_adj}

def summarize(ci, issue, sn):
    sums = []
    for cid, cd in ci['clusters'].items():
        titles = [str(t)[:50] for t in cd['titles'][:3] if t]
        prompt = ("'" + issue + "' " + sn + " 뉴스.\n"
                  "키워드: " + ', '.join(cd['keywords'][:10]) + "\n"
                  "제목: " + '; '.join(titles) + "\n2-3문장 요약 (마크업 없이).")
        try:
            r = client.chat.completions.create(model='gpt-4o-mini',
                messages=[{'role':'user','content':prompt}], temperature=0.3, max_tokens=300)
            content = r.choices[0].message.content.strip()
        except Exception as e:
            content = '요약 실패: ' + str(e)
        sums.append({'cid':cid, 'size':cd['size'],
                     'theme':', '.join(cd['keywords'][:5]), 'content':content})
        time.sleep(0.5)
    return sums

def narrate(sums, issue, sn, total):
    prompt = ("'" + issue + "' " + sn + " 뉴스 " + str(total) + "개:\n\n")
    for i, c in enumerate(sums, 1):
        prompt += str(i) + '. ' + c['theme'] + ' (' + str(c['size']) + '개): ' + c['content'] + '\n'
    prompt += "\n학술 논문 스타일 500-700자 서술 (마크업 없이)."
    try:
        r = client.chat.completions.create(model='gpt-4o',
            messages=[{'role':'user','content':prompt}], temperature=0.4, max_tokens=1000)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return '서술 실패: ' + str(e)

# ── 변경 2: build_net — Keyword_JSON 기반 동시출현 ────────────────
def build_net(ci):
    G = nx.Graph()
    all_kws = {}
    imp = {}
    cooc = defaultdict(lambda: defaultdict(int))

    for cid, cd in ci['clusters'].items():
        kws = cd['keywords'][:10]
        all_kws[cid] = set(kws)
        for kw in kws:
            imp[kw] = cd['kw_scores'].get(kw, 0.01)
            G.add_node(kw)

        # 각 기사의 Keyword_JSON에서 동시출현 계산
        for _, row in cd['df'].iterrows():
            article_kws = set(_parse_kj(row.get('Keyword_JSON', '{}')).keys())
            present = [kw for kw in kws if kw in article_kws]
            for k1 in present:
                for k2 in present:
                    if k1 < k2:
                        cooc[k1][k2] += 1

    all_c = [c for k1 in cooc for c in cooc[k1].values()]
    if all_c:
        mx = max(all_c)
        for k1 in cooc:
            for k2, cnt in cooc[k1].items():
                if cnt / mx >= viz_params['edge_threshold']:
                    G.add_edge(k1, k2, weight=cnt / mx)

    bridges = set()
    assign = {}
    for kw in list(G.nodes()):
        ap = [cid for cid, kws in all_kws.items() if kw in kws]
        if len(ap) >= 2:
            bridges.add(kw)
        assign[kw] = ap[0] if ap else 0

    for node in [nd for nd in G.nodes() if G.degree(nd) < 2]:
        G.remove_node(node)
        assign.pop(node, None)
        bridges.discard(node)

    return G, all_kws, imp, assign, bridges

def layout_net(G, all_kws, imp, assign, bridges, n):
    r = viz_params['cluster_radius']
    centers = {i: (r*math.cos(2*math.pi*i/n - math.pi/2),
                   r*math.sin(2*math.pi*i/n - math.pi/2)) for i in range(n)}
    pos = {}
    for cid in range(n):
        nodes = [nd for nd in G.nodes() if assign.get(nd) == cid and nd not in bridges]
        cx, cy = centers[cid]
        nl = sorted([(nd, imp.get(nd, 1)) for nd in nodes], key=lambda x: -x[1])
        for idx, (node, _) in enumerate(nl):
            if idx == 0:
                pos[node] = (cx, cy)
            else:
                layer = (idx-1)//6 + 1
                angle = 2*math.pi*((idx-1)%6)/6
                pos[node] = (cx + (1.2+layer*0.9)*math.cos(angle),
                             cy + (1.2+layer*0.9)*math.sin(angle))
    for bk in bridges:
        con = [cid for cid, kws in all_kws.items() if bk in kws]
        if len(con) >= 2:
            pos[bk] = (sum(centers[c][0] for c in con)/len(con),
                       sum(centers[c][1] for c in con)/len(con))
    nd_ = viz_params['min_node_distance']
    nl_ = list(pos.keys())
    for _ in range(150):
        moved = False
        for i in range(len(nl_)):
            for j in range(i+1, len(nl_)):
                x1, y1 = pos[nl_[i]]
                x2, y2 = pos[nl_[j]]
                d = math.sqrt((x1-x2)**2 + (y1-y2)**2)
                if d < nd_ and d > 0:
                    ov = nd_ - d + 0.4
                    dx, dy = (x1-x2)/d, (y1-y2)/d
                    pos[nl_[i]] = (x1+dx*ov/2, y1+dy*ov/2)
                    pos[nl_[j]] = (x2-dx*ov/2, y2-dy*ov/2)
                    moved = True
        if not moved:
            break
    return pos

def draw_net(G, pos, assign, bridges, imp, sums, issue, sn):
    """네트워크 그래프 출력 — 300 DPI + 큰 글씨"""
    fig = plt.figure(
        figsize=(viz_params['figure_width'], viz_params['figure_height']),
        facecolor='white',
        dpi=viz_params['figure_dpi']
    )

    # ── 엣지 그리기 ───────────────────────────────────────────
    if G.number_of_edges() > 0:
        rw = [G[u][v]['weight'] for u, v in G.edges()]
        mn, mx = min(rw), max(rw)
        rng = mx - mn if mx > mn else 1
        for (u, v), w in zip(G.edges(), rw):
            nw = (w - mn) / rng
            width = viz_params['edge_width_min'] + nw * (
                viz_params['edge_width_max'] - viz_params['edge_width_min'])
            alpha = viz_params['edge_alpha_min'] + nw * (
                viz_params['edge_alpha_max'] - viz_params['edge_alpha_min'])
            nx.draw_networkx_edges(
                G, pos, [(u, v)],
                alpha=alpha, width=width, edge_color='#444444'
            )

    # ── 노드 그리기 (클러스터별) ──────────────────────────────
    ai = list(imp.values()) or [1]
    mni, mxi = min(ai), max(ai)
    ri = mxi - mni if mxi > mni else 1

    for cid in range(len(sums)):
        nodes = [nd for nd in G.nodes()
                 if assign.get(nd) == cid and nd not in bridges]
        if not nodes:
            continue
        imps = [imp.get(nd, 1) for nd in nodes]
        mnc, mxc = min(imps), max(imps)
        rc = mxc - mnc if mxc > mnc else 1
        sizes = [
            viz_params['node_size_min'] + (i - mnc) / rc * (
                viz_params['node_size_max'] - viz_params['node_size_min'])
            for i in imps
        ]
        nx.draw_networkx_nodes(
            G.subgraph(nodes),
            {nd: pos[nd] for nd in nodes if nd in pos},
            node_color=COLORS[cid % len(COLORS)],
            node_shape=SHAPES[cid % len(SHAPES)],
            node_size=sizes, alpha=0.9,
            edgecolors=EDGE_COLORS[cid % len(EDGE_COLORS)],
            linewidths=viz_params['edge_linewidth']
        )

    # ── 브릿지 노드 (여러 클러스터에 걸친 키워드) ──────────────
    for bk in bridges:
        if bk not in pos:
            continue
        cid = assign.get(bk, 0)
        i_ = imp.get(bk, 1)
        sz = viz_params['node_size_min'] + (i_ - mni) / ri * (
            viz_params['node_size_max'] - viz_params['node_size_min'])
        nx.draw_networkx_nodes(
            G.subgraph([bk]), {bk: pos[bk]},
            node_color=COLORS[cid % len(COLORS)],
            node_size=sz * 1.3, alpha=1.0,
            edgecolors='red',
            linewidths=viz_params['edge_linewidth'] + 1.5
        )

    # ── 라벨 (큰 글씨 + 진한 굵기) ────────────────────────────
    font_family = plt.rcParams['font.family'][0]
    nx.draw_networkx_labels(
        G, pos,
        {nd: nd for nd in G.nodes() if nd in pos},
        font_size=viz_params['label_fontsize'],
        font_family=font_family,
        font_weight=viz_params['label_fontweight']
    )

    # ── 제목 (큰 글씨) ────────────────────────────────────────
    plt.title(
        issue + ' - ' + sn + '\n'
        + str(G.number_of_nodes()) + '개 키워드  '
        + str(G.number_of_edges()) + '개 연결',
        fontsize=viz_params['title_fontsize'],
        fontweight='bold',
        pad=20
    )
    plt.axis('off')

    # ── 범례 (큰 글씨) ────────────────────────────────────────
    for i, s in enumerate(sums):
        plt.plot(
            [], [],
            color=COLORS[i % len(COLORS)],
            marker=SHAPES[i % len(SHAPES)],
            markersize=14,
            label='C' + str(i + 1) + ': ' + s['theme'][:25],
            linewidth=0
        )
    plt.legend(
        loc='lower right',
        fontsize=viz_params['legend_fontsize'],
        framealpha=0.95,
        edgecolor='#888888',
        fancybox=True
    )
    plt.tight_layout()

    # ── base64 저장 (300 DPI) ────────────────────────────────
    buf = BytesIO()
    plt.savefig(
        buf, format='png',
        dpi=viz_params['savefig_dpi'],     # 100 → 300
        bbox_inches='tight',
        facecolor='white'
    )
    plt.show()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

# ── 데이터 로드 ──────────────────────────────────────────────────
s10_sent = pd.read_sql('SELECT * FROM ' + T_SENT, engine)
s10_esg  = pd.read_sql('SELECT * FROM ' + T_ESG, engine)
s10_embd = pd.read_sql('SELECT url, Keyword_JSON, Embedding_Input, Embedding_Vector FROM ' + T_EMBD, engine)
#                                      ^^^^^^^^^^^^  ← 변경 3: Keyword_JSON 추가

if len(s10_sent) == 0 or len(s10_esg) == 0:
    print('섹션 7, 8을 먼저 실행하세요.')
else:
    def _cs2(x):
        if pd.isna(x): return None
        return re.sub(r'[\*\[\]\(\)_`<>]', '', str(x)).strip()
    def _pv(x):
        if isinstance(x, list): return x
        if isinstance(x, str):
            try: return json.loads(x)
            except:
                try: return ast.literal_eval(x)
                except: return None
        return None

    s10_sent['sc'] = s10_sent['sentiment'].apply(_cs2)
    s10_sent['sv'] = s10_sent['sc'].map({'긍정':1, '중립':0, '부정':-1})

    mdf = s10_esg.merge(s10_sent[['url','sv','contents','title']], on='url', how='left')
    mdf = mdf.merge(s10_embd[['url','Keyword_JSON','Embedding_Input','Embedding_Vector']],
                     on='url', how='left')
    #                      ^^^^^^^^^^^^  ← 변경 3: Keyword_JSON 포함

    df10 = mdf[(mdf['sv'].isin([1,-1])) & (mdf['Embedding_Vector'].notna())].copy()
    df10['emb'] = df10['Embedding_Vector'].apply(_pv)
    df10 = df10[df10['emb'].notna()].copy()

    esg_issues = TARGET_ESG_ISSUES if TARGET_ESG_ISSUES \
                 else df10['esg_issue'].value_counts().head(TOP_N_ISSUES).index.tolist()
    print('분석 대상 ESG 이슈 (', len(esg_issues), '개):', esg_issues)
    print('분석 데이터:', len(df10), '건 (중립 제외)')

    all_results = {}
    for iss in esg_issues:
        all_results[iss] = {}
        for sv, sn in {1:'긍정', -1:'부정'}.items():
            mask = (df10['esg_issue'] == iss) & (df10['sv'] == sv)
            df_sub = df10[mask].copy()
            print('  [', iss, sn, ']', len(df_sub), '개', end='  ')
            if len(df_sub) < MIN_ARTICLES:
                print('기사 부족 스킵')
                continue
            try:
                ci = do_clustering(df_sub, N_CLUSTERS)
                cs = summarize(ci, iss, sn)
                nv = narrate(cs, iss, sn, len(df_sub))
                G, all_kws, imp, assign, bridges = build_net(ci)
                if G.number_of_nodes() == 0:
                    print('노드 없음 스킵')
                    continue
                pos = layout_net(G, all_kws, imp, assign, bridges, ci['n'])
                img = draw_net(G, pos, assign, bridges, imp, cs, iss, sn)
                all_results[iss][sn] = {
                    'total': len(df_sub), 'n_clusters': ci['n'],
                    'clusters': cs, 'narrative': nv, 'img': img
                }
                print('완료 (' + str(ci['n']) + '개 클러스터)')
            except Exception as e:
                print('오류:', e)
    print('\n섹션 10 분석 완료!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 섹션 10-2: 네트워크 정보 → Gephi 편집용 파일 저장
# ═══════════════════════════════════════════════════════════════════
# 섹션 10에서 분석한 모든 네트워크를 GEXF / GraphML / CSV 포맷으로 저장
# Gephi, Cytoscape, yEd 등에서 직접 편집·시각화 가능
# ═══════════════════════════════════════════════════════════════════

import os
import json as _json
from datetime import datetime
import networkx as nx
import pandas as pd

# ── 저장 경로 설정 ───────────────────────────────────────────────
GEPHI_OUTPUT_DIR = 'gephi_export'
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_ROOT = f'{GEPHI_OUTPUT_DIR}/{SEARCH_KEYWORD}_{TIMESTAMP}'

os.makedirs(EXPORT_ROOT, exist_ok=True)
os.makedirs(f'{EXPORT_ROOT}/gexf', exist_ok=True)
os.makedirs(f'{EXPORT_ROOT}/graphml', exist_ok=True)
os.makedirs(f'{EXPORT_ROOT}/csv', exist_ok=True)

print('=' * 60)
print(f'📤 Gephi 내보내기 시작')
print(f'   저장 경로: {EXPORT_ROOT}/')
print('=' * 60)


# ══════════════════════════════════════════════════════════════════
# 헬퍼 함수
# ══════════════════════════════════════════════════════════════════
def _safe_filename(text):
    """파일명으로 안전한 문자열 변환"""
    import re
    text = re.sub(r'[\\/:*?"<>|]', '_', str(text))
    text = re.sub(r'\s+', '_', text)
    return text[:50]


def _build_graph_with_attrs(ci_clusters, ci_n, G_source, pos, assign, bridges, imp, sums):
    """속성이 풍부한 새 Graph 객체 생성 — Gephi에서 바로 활용 가능"""
    G_new = nx.Graph()

    # 클러스터별 색상 매핑 (COLORS 사용)
    cluster_colors = {
        i: COLORS[i % len(COLORS)]
        for i in range(ci_n)
    }
    cluster_themes = {
        i: sums[i]['theme'] if i < len(sums) else f'Cluster {i+1}'
        for i in range(ci_n)
    }

    # ── 노드 속성 추가 ─────────────────────────────────────────
    for node in G_source.nodes():
        cid = assign.get(node, 0)
        is_bridge = node in bridges
        importance_val = float(imp.get(node, 0.01))
        degree = G_source.degree(node)

        # 좌표 (layout_net 결과 사용)
        x, y = pos.get(node, (0.0, 0.0))

        # 시각적 크기 (10~40 범위로 정규화)
        all_imp = [float(imp.get(n, 0.01)) for n in G_source.nodes()]
        mn_imp = min(all_imp) if all_imp else 0.01
        mx_imp = max(all_imp) if all_imp else 1.0
        rg = mx_imp - mn_imp if mx_imp > mn_imp else 1
        node_size = 10 + (importance_val - mn_imp) / rg * 30

        # 색상 (bridge는 빨강, 아니면 클러스터 색상)
        color_hex = '#FF0000' if is_bridge else cluster_colors.get(cid, '#888888')

        G_new.add_node(
            node,
            label=str(node),
            cluster_id=int(cid),
            cluster_theme=str(cluster_themes.get(cid, '')),
            importance=round(importance_val, 4),
            is_bridge=bool(is_bridge),
            degree=int(degree),
            x=float(x),
            y=float(y),
            color=color_hex,
            size=round(float(node_size), 2),
        )

    # ── 엣지 속성 추가 ─────────────────────────────────────────
    for u, v in G_source.edges():
        weight = G_source[u][v].get('weight', 0.0)
        G_new.add_edge(
            u, v,
            weight=round(float(weight), 4),
            raw_count=int(round(weight * 100)),   # 대략적 원본 추정
        )

    return G_new


def _export_single_network(G, issue, sentiment_name, meta_info, base_dir):
    """단일 네트워크를 3가지 포맷으로 저장"""
    safe_issue = _safe_filename(issue)
    safe_sn = _safe_filename(sentiment_name)
    base_name = f'{safe_issue}__{safe_sn}'

    # 1) GEXF (Gephi 네이티브 포맷) — 추천
    gexf_path = f'{base_dir}/gexf/{base_name}.gexf'
    try:
        nx.write_gexf(G, gexf_path, version='1.2draft')
    except Exception as e:
        print(f'      ⚠️ GEXF 저장 실패: {e}')
        gexf_path = None

    # 2) GraphML (범용 포맷)
    graphml_path = f'{base_dir}/graphml/{base_name}.graphml'
    try:
        nx.write_graphml(G, graphml_path)
    except Exception as e:
        print(f'      ⚠️ GraphML 저장 실패: {e}')
        graphml_path = None

    # 3) CSV (엣지 + 노드 분리) — Excel·다른 툴 호환
    try:
        # 엣지 CSV
        edge_rows = [{
            'Source': u,
            'Target': v,
            'Type': 'Undirected',
            'Weight': G[u][v].get('weight', 0),
            'raw_count': G[u][v].get('raw_count', 0)
        } for u, v in G.edges()]
        edge_df = pd.DataFrame(edge_rows)
        edge_df.to_csv(f'{base_dir}/csv/{base_name}__edges.csv',
                       index=False, encoding='utf-8-sig')

        # 노드 CSV
        node_rows = [{
            'Id': n,
            'Label': G.nodes[n].get('label', n),
            'cluster_id': G.nodes[n].get('cluster_id', 0),
            'cluster_theme': G.nodes[n].get('cluster_theme', ''),
            'importance': G.nodes[n].get('importance', 0),
            'is_bridge': G.nodes[n].get('is_bridge', False),
            'degree': G.nodes[n].get('degree', 0),
            'x': G.nodes[n].get('x', 0),
            'y': G.nodes[n].get('y', 0),
            'color': G.nodes[n].get('color', '#888888'),
            'size': G.nodes[n].get('size', 10)
        } for n in G.nodes()]
        node_df = pd.DataFrame(node_rows)
        node_df.to_csv(f'{base_dir}/csv/{base_name}__nodes.csv',
                       index=False, encoding='utf-8-sig')
    except Exception as e:
        print(f'      ⚠️ CSV 저장 실패: {e}')

    return gexf_path, graphml_path


# ══════════════════════════════════════════════════════════════════
# 메인 내보내기 루프
# ══════════════════════════════════════════════════════════════════
export_summary = []
total_exported = 0
total_failed = 0

# 섹션 10에서 생성된 모든 이슈 × 감성 조합 순회
# all_results[issue][sentiment_name] = {'total', 'n_clusters', 'clusters', 'narrative', 'img'}
# 단, 네트워크 재구성을 위해 원본 ci, G, pos 등은 재계산 필요

print('\n[네트워크 재구성 및 저장]\n')

# 저장된 중립 제외 데이터 재로드 (섹션 10 실행 후 재사용)
for iss in esg_issues:
    if iss not in all_results:
        continue

    for sv, sn in {1:'긍정', -1:'부정'}.items():
        if sn not in all_results[iss]:
            continue

        result = all_results[iss][sn]
        print(f'  [{iss}] × [{sn}]', end=' ')

        try:
            # 원본 서브셋 추출 (df10에서 재필터링)
            mask = (df10['esg_issue'] == iss) & (df10['sv'] == sv)
            df_sub = df10[mask].copy()

            if len(df_sub) < MIN_ARTICLES:
                print('✗ 기사 부족')
                continue

            # 클러스터링·네트워크 재구성 (섹션 10과 동일 로직)
            ci = do_clustering(df_sub, N_CLUSTERS)
            G_src, all_kws, imp, assign, bridges = build_net(ci)

            if G_src.number_of_nodes() == 0:
                print('✗ 노드 없음')
                continue

            pos = layout_net(G_src, all_kws, imp, assign, bridges, ci['n'])

            # 속성 풍부한 Graph 생성
            sums_data = result['clusters']
            G_rich = _build_graph_with_attrs(
                ci['clusters'], ci['n'],
                G_src, pos, assign, bridges, imp, sums_data
            )

            # 메타 정보
            meta = {
                'issue': iss,
                'sentiment': sn,
                'total_articles': result['total'],
                'n_clusters': result['n_clusters'],
                'n_nodes': G_rich.number_of_nodes(),
                'n_edges': G_rich.number_of_edges(),
                'n_bridges': len(bridges),
                'created_at': TIMESTAMP,
                'company': SEARCH_KEYWORD,
            }

            # 3가지 포맷으로 저장
            gexf_p, graphml_p = _export_single_network(
                G_rich, iss, sn, meta, EXPORT_ROOT
            )

            export_summary.append({**meta,
                'gexf_path': gexf_p,
                'graphml_path': graphml_p})
            total_exported += 1
            print(f'✓ {G_rich.number_of_nodes()} 노드, {G_rich.number_of_edges()} 엣지')

        except Exception as e:
            print(f'✗ 오류: {e}')
            total_failed += 1


# ══════════════════════════════════════════════════════════════════
# 메타 정보 JSON + README 생성
# ══════════════════════════════════════════════════════════════════
meta_json_path = f'{EXPORT_ROOT}/_metadata.json'
with open(meta_json_path, 'w', encoding='utf-8') as f:
    _json.dump({
        'company': SEARCH_KEYWORD,
        'topic_code': TOPIC_CODE,
        'period': f'{START_DATE} ~ {END_DATE}',
        'exported_at': TIMESTAMP,
        'total_networks': total_exported,
        'failed': total_failed,
        'networks': export_summary,
    }, f, ensure_ascii=False, indent=2)

readme_path = f'{EXPORT_ROOT}/README.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(f"""# {SEARCH_KEYWORD} ESG 네트워크 분석 — Gephi 편집용

## 생성 정보
- **기업**: {SEARCH_KEYWORD} ({TOPIC_CODE})
- **분석 기간**: {START_DATE} ~ {END_DATE}
- **생성일**: {TIMESTAMP}
- **네트워크 수**: {total_exported}개

## 폴더 구조 {EXPORT_ROOT}/
├── README.md              ← 이 문서
├── _metadata.json         ← 전체 메타 정보
├── gexf/                  ← Gephi 네이티브 (권장)
│   └── {{이슈}}{{감성}}.gexf
├── graphml/               ← 범용 포맷 (Cytoscape, yEd 등)
│   └── {{이슈}}{{감성}}.graphml
└── csv/                   ← Excel·데이터 분석용
├── {{이슈}}__{{감성}}edges.csv
└── {{이슈}}{{감성}}__nodes.csv

## Gephi에서 열기

1. Gephi 다운로드: https://gephi.org (무료)
2. `File > Open` → `.gexf` 파일 선택
3. 자동으로 노드 색상·위치·크기 반영됨
4. `Layout`에서 ForceAtlas 2 등 추가 레이아웃 적용 가능

## 노드 속성 (Node Attributes)

| 속성 | 설명 |
|------|------|
| `label` | 키워드 텍스트 |
| `cluster_id` | K-means 클러스터 번호 (0~3) |
| `cluster_theme` | 클러스터 주제 (상위 키워드 5개) |
| `importance` | 키워드 중요도 (ESG 점수) |
| `is_bridge` | True면 2개 이상 클러스터 연결 (빨강 표시) |
| `degree` | 연결 차수 |
| `x`, `y` | 레이아웃 좌표 |
| `color` | HEX 색상 |
| `size` | 시각적 크기 (10~40) |

## 엣지 속성 (Edge Attributes)

| 속성 | 설명 |
|------|------|
| `weight` | 동시출현 가중치 (0.05~1.0 정규화) |
| `raw_count` | 원본 동시출현 추정치 |

## Gephi 활용 팁

- **Modularity 커뮤니티 감지**: Statistics > Modularity → Partition에서 색상 재지정
- **크기 조정**: Appearance > Nodes > Size → `importance` 기반 재조정
- **라벨 크기**: Preview > Node Labels > Font 조정
- **내보내기**: File > Export > PDF/PNG/SVG

""")


# ══════════════════════════════════════════════════════════════════
# 최종 결과 출력
# ══════════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print(f'✅ 내보내기 완료')
print('=' * 60)
print(f'  성공: {total_exported}개 네트워크')
print(f'  실패: {total_failed}개')
print(f'\n  저장 위치: {EXPORT_ROOT}/')
print(f'    ├── gexf/      (Gephi 네이티브 — 추천)')
print(f'    ├── graphml/   (범용 포맷)')
print(f'    ├── csv/       (Excel·데이터 분석)')
print(f'    ├── _metadata.json')
print(f'    └── README.md')

# ── 디렉토리 트리 출력 ────────────────────────────────────────────
print(f'\n[생성된 파일 목록]')
for root, dirs, files in os.walk(EXPORT_ROOT):
    level = root.replace(EXPORT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files)[:5]:  # 각 폴더 최대 5개만 표시
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{subindent}📄 {file} ({size_kb:.1f} KB)')
    if len(files) > 5:
        print(f'{subindent}... 외 {len(files)-5}개')


# ══════════════════════════════════════════════════════════════════
# Google Colab 사용자를 위한 압축 다운로드 (선택)
# ══════════════════════════════════════════════════════════════════
try:
    from google.colab import files as _colab_files
    import shutil

    zip_path = f'{GEPHI_OUTPUT_DIR}/{SEARCH_KEYWORD}_{TIMESTAMP}_gephi.zip'
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', EXPORT_ROOT)
    print(f'\n📦 압축 파일 생성: {zip_path}')
    print(f'   다운로드 시작...')
    _colab_files.download(zip_path)
    print(f'   ✅ 다운로드 완료')
except ImportError:
    # Colab이 아닌 환경
    print(f'\n💡 로컬 환경: {EXPORT_ROOT}/ 폴더를 직접 확인하세요.')
except Exception as e:
    print(f'\n⚠️ 압축/다운로드 오류: {e}')
    print(f'   수동으로 {EXPORT_ROOT}/ 폴더를 확인하세요.')

print('\n섹션 10-2 완료!')

# 섹션 11: HTML 스토리북 + GitHub 업로드

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-1. CSS 빌더 (논문 형식 v2)
# ═══════════════════════════════════════════════════════════════════
def build_paper_css():
    """v2 CSS 반환 — 부분별 분할 작성으로 잘림 방지"""

    # (1) 공통 + 변수
    css_base = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@400;500;600;700&family=Noto+Serif+KR:wght@500;700&display=swap');
* { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --ink:#1a1a1a; --ink-2:#4a4a4a; --ink-3:#6b7280;
  --line:#e5e7eb; --line-2:#d1d5db;
  --bg:#fff; --bg-2:#f9fafb; --bg-3:#f3f4f6;
  --accent:#1e3a5f; --accent-soft:#e0e7ef; --accent-bg:#eff4fa;
  --pos:#15803d; --pos-soft:#dcfce7; --pos-bg:#f0fdf4;
  --neg:#b91c1c; --neg-soft:#fee2e2; --neg-bg:#fef2f2;
  --warn:#a16207; --warn-soft:#fef3c7;
  --discovery:#7c3aed; --discovery-bg:#f5f3ff;
}
body { font-family:'Noto Sans KR',sans-serif; background:var(--bg); color:var(--ink); line-height:1.7; font-size:15px; }
.container { max-width:920px; margin:0 auto; padding:0 24px; }
"""

    # (2) Hero + KPI
    css_hero = """
.hero { background:var(--bg-2); border-bottom:1px solid var(--line); padding:56px 0 40px; margin-bottom:40px; }
.hero .container { display:flex; flex-direction:column; gap:16px; }
.hero-tag { font-size:12px; font-weight:600; letter-spacing:2px; color:var(--accent); text-transform:uppercase; }
.hero-title { font-family:'Noto Serif KR',serif; font-size:32px; font-weight:700; line-height:1.3; letter-spacing:-0.5px; }
.hero-sub { font-size:15px; color:var(--ink-2); }
.hero-meta { display:flex; gap:24px; font-size:13px; color:var(--ink-3); margin-top:8px; flex-wrap:wrap; }
.hero-meta span::before { content:"●"; margin-right:6px; color:var(--accent); font-size:8px; vertical-align:middle; }
.kpi-grid { display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin:32px 0 48px; }
.kpi { background:var(--bg-2); border-radius:8px; padding:18px 16px; }
.kpi-label { font-size:11px; font-weight:600; color:var(--ink-3); letter-spacing:1px; text-transform:uppercase; margin-bottom:6px; }
.kpi-value { font-size:22px; font-weight:700; line-height:1.1; }
.kpi-value .unit { font-size:13px; font-weight:400; color:var(--ink-3); margin-left:2px; }
.kpi-change { font-size:11px; color:var(--pos); margin-top:4px; font-weight:500; }
"""

    # (3) 섹션 헤더 + FAQ + 방법론
    css_section = """
section { margin:48px 0; }
h2 { font-family:'Noto Serif KR',serif; font-size:20px; font-weight:700; margin-bottom:24px; padding-bottom:10px; border-bottom:1.5px solid var(--ink); display:flex; align-items:baseline; gap:10px; }
h2 .num { font-family:'Noto Sans KR',sans-serif; font-size:12px; font-weight:600; color:var(--accent); letter-spacing:2px; }
h3 { font-size:15px; font-weight:600; margin:28px 0 10px; }
p { margin-bottom:12px; color:var(--ink-2); }
.faq-section { margin:32px 0 48px; }
.faq-row { display:grid; grid-template-columns:280px 1fr; gap:36px; padding:24px 0; border-bottom:0.5px solid var(--line); align-items:start; }
.faq-row:last-child { border-bottom:none; }
.faq-row.highlight { background:var(--accent-bg); border-radius:12px; padding:28px 28px; margin:8px 0; border-bottom:none; }
.faq-left { display:flex; gap:14px; align-items:flex-start; }
.faq-q-badge { width:32px; height:32px; min-width:32px; border-radius:50%; background:#fff; border:1.5px solid var(--accent); color:var(--accent); display:flex; align-items:center; justify-content:center; font-size:12px; font-weight:700; margin-top:2px; }
.faq-row.highlight .faq-q-badge { background:var(--accent); color:#fff; }
.faq-q-content { flex:1; }
.faq-question { font-size:14.5px; font-weight:700; color:var(--ink); line-height:1.5; margin-bottom:6px; }
.faq-conclusion { font-size:13px; font-weight:600; color:var(--pos); line-height:1.5; }
.faq-conclusion.neutral { color:var(--accent); }
.faq-conclusion::before { content:"→ "; font-weight:700; }
.faq-right { font-size:13.5px; color:var(--ink-2); line-height:1.8; }
.faq-right p { margin-bottom:8px; }
.faq-right p:last-child { margin-bottom:0; }
.faq-right strong { font-weight:600; color:var(--ink); }
.faq-right .muted { color:var(--ink-3); font-size:12.5px; }
.faq-metrics { display:grid; grid-template-columns:1fr 1fr; gap:24px; margin:14px 0 18px; padding-bottom:16px; border-bottom:0.5px solid var(--line); }
.faq-metric-block .faq-metric-label { font-size:11px; color:var(--ink-3); font-weight:500; margin-bottom:4px; }
.faq-metric-block .faq-metric-value { font-size:18px; font-weight:700; color:var(--ink); }
.faq-metric-block .faq-metric-value .highlight-num { color:var(--accent); }
.faq-metric-block .faq-metric-value .delta { color:var(--pos); font-size:14px; margin-left:4px; }
.faq-metric-block .faq-metric-sub { font-size:11px; color:var(--ink-3); margin-top:2px; }
.faq-chart { margin-top:10px; }
.faq-chart-title { font-size:11.5px; color:var(--ink-3); font-weight:500; margin-bottom:6px; }
.faq-chart-svg { width:100%; height:auto; display:block; }
.faq-chart-legend { display:flex; gap:14px; margin-top:8px; font-size:10.5px; color:var(--ink-3); align-items:center; flex-wrap:wrap; }
.faq-chart-legend .leg-item { display:inline-flex; align-items:center; gap:5px; }
.faq-chart-legend .leg-sq { width:10px; height:10px; border-radius:2px; display:inline-block; }
.faq-chart-legend .leg-note { margin-left:auto; font-style:italic; color:var(--ink-3); }
.faq-ranked-list { list-style:none; font-size:13.5px; }
.faq-ranked-list li { display:grid; grid-template-columns:24px 1fr auto; gap:10px; padding:7px 0; border-bottom:0.5px dashed var(--line); align-items:center; }
.faq-ranked-list li:last-child { border-bottom:none; }
.faq-ranked-list .rank-num { color:var(--accent); font-weight:700; font-size:14px; }
.faq-ranked-list .rank-issue { color:var(--ink); font-weight:500; }
.faq-ranked-list .rank-count { color:var(--ink-3); font-variant-numeric:tabular-nums; font-weight:600; }
.method-timeline { display:grid; grid-template-columns:repeat(5,1fr); gap:8px; margin:16px 0; }
.method-step { background:var(--bg-2); padding:14px 12px; border-radius:6px; text-align:center; }
.method-step .step-num { display:inline-block; width:22px; height:22px; border-radius:50%; background:var(--accent); color:#fff; font-size:11px; font-weight:700; line-height:22px; margin-bottom:6px; }
.method-step .step-title { font-size:12px; font-weight:600; margin-bottom:2px; }
.method-step .step-desc { font-size:10.5px; color:var(--ink-3); line-height:1.4; }
"""

    # (4) 독창성 + 핵심 발견 (v2 신규)
    css_v2 = """
.originality { background:var(--bg-2); border-radius:12px; padding:32px; margin:40px 0; }
.originality-header { display:flex; align-items:baseline; gap:12px; margin-bottom:6px; }
.originality-icon { font-size:14px; color:var(--accent); font-weight:700; letter-spacing:2px; }
.originality-title { font-family:'Noto Serif KR',serif; font-size:18px; font-weight:700; color:var(--ink); }
.originality-subtitle { font-size:12.5px; color:var(--ink-3); margin-bottom:24px; font-style:italic; }
.comparison-table { width:100%; border-collapse:collapse; font-size:13.5px; background:#fff; border-radius:8px; overflow:hidden; }
.comparison-table thead tr { background:var(--accent); }
.comparison-table thead th { padding:12px 16px; color:#fff; font-weight:600; text-align:left; font-size:12.5px; letter-spacing:0.3px; }
.comparison-table thead th:first-child { width:150px; background:var(--ink); }
.comparison-table tbody td { padding:12px 16px; border-bottom:0.5px solid var(--line); vertical-align:top; }
.comparison-table tbody tr:last-child td { border-bottom:none; }
.comparison-table tbody tr:nth-child(even) { background:var(--bg-2); }
.comp-dim { font-weight:600; color:var(--ink-2); font-size:12.5px; }
.comp-old { color:var(--ink-3); font-size:12.5px; line-height:1.5; }
.comp-new { color:var(--ink); font-weight:500; font-size:12.5px; line-height:1.5; }
.comp-new strong { color:var(--accent); font-weight:700; }
.discoveries { margin:40px 0; }
.discoveries-header { text-align:left; margin-bottom:24px; }
.discoveries-tag { display:inline-block; font-size:11px; font-weight:700; letter-spacing:2px; color:var(--discovery); background:var(--discovery-bg); padding:4px 12px; border-radius:12px; margin-bottom:10px; }
.discoveries-title { font-family:'Noto Serif KR',serif; font-size:20px; font-weight:700; color:var(--ink); margin-bottom:6px; }
.discoveries-subtitle { font-size:13px; color:var(--ink-3); font-style:italic; }
.discoveries-grid { display:grid; grid-template-columns:repeat(3,1fr); gap:16px; margin-top:20px; }
.discovery-card { background:#fff; border:1px solid var(--line); border-top:3px solid var(--discovery); border-radius:8px; padding:22px 20px; display:flex; flex-direction:column; }
.discovery-num { display:inline-block; font-size:10px; font-weight:700; color:var(--discovery); letter-spacing:2px; margin-bottom:10px; }
.discovery-title { font-size:15px; font-weight:700; color:var(--ink); line-height:1.4; margin-bottom:10px; min-height:42px; }
.discovery-core { background:var(--discovery-bg); border-radius:6px; padding:10px 12px; margin-bottom:12px; }
.discovery-core-headline { font-size:13px; font-weight:700; color:var(--discovery); line-height:1.5; margin-bottom:4px; }
.discovery-core-stat { font-size:11.5px; color:var(--ink-3); font-weight:500; }
.discovery-contrast { font-size:12px; color:var(--ink-2); line-height:1.6; padding-top:10px; border-top:0.5px solid var(--line); margin-top:auto; }
.discovery-contrast-label { display:inline-block; font-size:10px; color:var(--ink-3); font-weight:700; letter-spacing:1px; margin-bottom:4px; text-transform:uppercase; }
.discovery-contrast strong { color:var(--ink); font-weight:600; }
"""

    # (5) 표 + 그림 + 인사이트 + 한계
    css_table = """
.table-block { margin:20px 0 28px; }
.table-caption { font-size:12px; font-weight:600; color:var(--ink-2); margin-bottom:8px; }
.table-caption .tag { display:inline-block; background:var(--ink); color:#fff; font-size:10px; padding:2px 8px; border-radius:3px; margin-right:8px; letter-spacing:0.5px; font-weight:700; }
table { width:100%; border-collapse:collapse; font-size:13.5px; }
table thead tr { border-top:2px solid var(--ink); border-bottom:1px solid var(--ink); }
table th { padding:10px 12px; text-align:left; font-weight:600; background:var(--bg-2); }
table th.num-col, table td.num-col { text-align:right; font-variant-numeric:tabular-nums; }
table tbody td { padding:10px 12px; border-bottom:0.5px solid var(--line); color:var(--ink-2); }
table tbody tr:last-child td { border-bottom:2px solid var(--ink); }
.sig { font-weight:700; color:var(--accent); }
.sig-ns { color:var(--ink-3); }
.table-note { font-size:11.5px; color:var(--ink-3); margin-top:8px; font-style:italic; }
.figure-block { margin:24px 0 32px; background:var(--bg-2); border-radius:8px; padding:16px; }
.figure-caption { font-size:12px; font-weight:600; color:var(--ink-2); margin-bottom:12px; }
.figure-caption .tag { display:inline-block; background:var(--accent); color:#fff; font-size:10px; padding:2px 8px; border-radius:3px; margin-right:8px; letter-spacing:0.5px; font-weight:700; }
.figure-block img { width:100%; height:auto; display:block; background:#fff; border-radius:4px; }
.side-by-side { display:grid; grid-template-columns:3fr 2fr; gap:24px; align-items:start; }
.interpret { font-size:13.5px; color:var(--ink-2); padding:12px 0; line-height:1.8; }
.insights { display:grid; grid-template-columns:repeat(3,1fr); gap:14px; margin:20px 0; }
.insight-card { background:var(--bg); border:1px solid var(--line); border-radius:8px; padding:20px; }
.insight-num { display:inline-block; width:28px; height:28px; border-radius:50%; background:var(--accent); color:#fff; text-align:center; line-height:28px; font-size:13px; font-weight:700; margin-bottom:10px; }
.insight-title { font-size:14px; font-weight:700; margin-bottom:8px; line-height:1.4; }
.insight-body { font-size:12.5px; color:var(--ink-3); line-height:1.6; }
.limits { background:var(--bg-2); border-radius:8px; padding:18px 24px; margin:16px 0; }
.limits-title { font-size:12px; font-weight:600; color:var(--ink-3); letter-spacing:1px; margin-bottom:10px; text-transform:uppercase; }
.limits ul { list-style:none; font-size:13px; color:var(--ink-2); }
.limits li { padding:4px 0 4px 18px; position:relative; line-height:1.7; }
.limits li::before { content:"—"; position:absolute; left:0; color:var(--ink-3); }
"""

    # (6) 주제 분석 + footer + 반응형
    css_topic = """
.issue-block { margin:40px 0; border:1px solid var(--line); border-radius:12px; overflow:hidden; }
.issue-header { background:var(--bg-2); padding:20px 28px; border-bottom:1px solid var(--line); display:flex; align-items:center; gap:16px; }
.issue-rank { width:36px; height:36px; border-radius:50%; background:var(--accent); color:#fff; display:flex; align-items:center; justify-content:center; font-size:14px; font-weight:700; flex-shrink:0; }
.issue-info { flex:1; }
.issue-name { font-size:16px; font-weight:700; color:var(--ink); margin-bottom:2px; }
.issue-meta { font-size:12px; color:var(--ink-3); }
.issue-meta span { margin-right:14px; }
.issue-meta strong { color:var(--ink-2); font-weight:600; }
.issue-body { padding:24px 28px; }
.sentiment-section { margin:24px 0; padding:20px; border-radius:8px; }
.sentiment-section.pos { background:var(--pos-bg); border:1px solid var(--pos-soft); }
.sentiment-section.neg { background:var(--neg-bg); border:1px solid var(--neg-soft); }
.sentiment-header { display:flex; align-items:center; gap:10px; margin-bottom:16px; }
.sentiment-badge { display:inline-flex; align-items:center; gap:6px; padding:6px 12px; border-radius:16px; font-size:12px; font-weight:700; letter-spacing:0.5px; }
.sentiment-badge.pos { background:var(--pos); color:#fff; }
.sentiment-badge.neg { background:var(--neg); color:#fff; }
.sentiment-badge::before { content:"●"; font-size:8px; }
.sentiment-count { font-size:13px; color:var(--ink-2); font-weight:500; }
.sentiment-count strong { color:var(--ink); font-weight:700; }
.narrative-box { background:#fff; border-left:3px solid var(--accent); padding:16px 20px; border-radius:4px; margin:14px 0; font-size:13.5px; line-height:1.85; color:var(--ink-2); }
.narrative-box .narr-label { font-size:11px; font-weight:700; color:var(--accent); letter-spacing:1px; margin-bottom:8px; text-transform:uppercase; }
.network-figure { background:#fff; border-radius:6px; padding:12px; margin:16px 0; }
.network-figure img { width:100%; height:auto; border-radius:4px; display:block; }
.clusters-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-top:16px; }
.cluster-card { background:#fff; border-radius:6px; padding:14px 16px; border-top:3px solid var(--cluster-color,var(--accent)); }
.cluster-card.c1 { --cluster-color:#CA8A04; }
.cluster-card.c2 { --cluster-color:#16A34A; }
.cluster-card.c3 { --cluster-color:#0369A1; }
.cluster-card.c4 { --cluster-color:#BE185D; }
.cluster-card.c5 { --cluster-color:#7C3AED; }
.cluster-card.c6 { --cluster-color:#9333EA; }
.cluster-head { display:flex; align-items:baseline; justify-content:space-between; margin-bottom:10px; }
.cluster-id { font-size:11px; font-weight:700; color:var(--cluster-color); letter-spacing:1px; }
.cluster-size { font-size:11px; color:var(--ink-3); }
.cluster-name { font-size:13.5px; font-weight:700; color:var(--ink); margin-bottom:8px; line-height:1.4; }
.cluster-keywords { display:flex; flex-wrap:wrap; gap:4px; margin:10px 0; }
.kw-chip { background:var(--bg-3); color:var(--ink-2); font-size:11px; padding:2px 8px; border-radius:10px; font-weight:500; }
.kw-chip.top { background:var(--cluster-color); color:#fff; font-weight:600; }
.cluster-desc { font-size:12px; color:var(--ink-2); line-height:1.65; margin-top:8px; padding-top:8px; border-top:0.5px solid var(--line); }
.comparison-box { background:var(--accent-soft); border-radius:8px; padding:18px 22px; margin-top:24px; }
.comparison-label { font-size:11px; font-weight:700; color:var(--accent); letter-spacing:2px; margin-bottom:8px; }
.comparison-text { font-size:13.5px; color:var(--ink); line-height:1.8; }
footer { border-top:1px solid var(--line); margin-top:60px; padding:24px 0; text-align:center; font-size:11.5px; color:var(--ink-3); }
@media (max-width:720px) {
  .kpi-grid { grid-template-columns:repeat(2,1fr); }
  .method-timeline { grid-template-columns:1fr 1fr; }
  .insights { grid-template-columns:1fr; }
  .side-by-side { grid-template-columns:1fr; }
  .clusters-grid { grid-template-columns:1fr; }
  .hero-title { font-size:24px; }
  .issue-body { padding:16px 18px; }
  .faq-row, .faq-row.highlight { grid-template-columns:1fr; gap:16px; }
  .faq-metrics { grid-template-columns:1fr; gap:14px; }
  .discoveries-grid { grid-template-columns:1fr; }
}
"""

    return css_base + css_hero + css_section + css_v2 + css_table + css_topic


_t = build_paper_css()
print(f'✅ 11-1: build_paper_css() 정의 완료')
print(f'   CSS 길이: {len(_t):,} 문자, 클래스 정의 수 (추정): {_t.count("{")}개')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-2. HTML escape 유틸 + 비교 단락 GPT 호출
# ═══════════════════════════════════════════════════════════════════
def _esc(s):
    """HTML escape"""
    if s is None: return ''
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
                  .replace('>', '&gt;').replace('"', '&quot;'))


def gen_comparison(issue, pos_narrative, neg_narrative):
    """이슈별 긍·부정 담론 비교 단락 생성 (gpt-4o)"""
    prompt = (
        f"다음은 '{issue}' ESG 이슈에 대한 긍정·부정 감성 기사 분석 결과입니다.\n\n"
        f"[긍정 담론 종합]\n{pos_narrative}\n\n"
        f"[부정 담론 종합]\n{neg_narrative}\n\n"
        "두 담론 구조의 핵심 차이점을 200~300자로 비교 분석하세요. "
        "투자자 의사결정·평판 리스크 관점에서 시사점을 포함하고, "
        "마크다운 기호 없이 평문으로 작성하세요."
    )
    try:
        r = client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.4, max_tokens=500
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'비교 단락 생성 실패: {e}'


print('✅ 11-2: _esc(), gen_comparison() 정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 교체 11-3 : _build_hero  (타이틀만 남기고 간소화)
# ═══════════════════════════════════════════════════════════════════
def _build_hero(meta, kf, t1):
    """Hero — 제목과 부제만. 메타 정보는 초록 인포그래픽으로 이동."""
    return f"""
<div class="hero"><div class="container">
  <div class="hero-tag">ESG News Impact Analysis</div>
  <h1 class="hero-title">{_esc(meta['company'])} ESG 이슈 대응과 주가 반응 분석</h1>
  <p class="hero-sub">뉴스 기반 감성지수와 시차 회귀분석을 중심으로</p>
</div></div>
"""

print('✅ _build_hero 정의 완료')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-3b : _build_abstract (FAQ 형식 연구개요)
#         — 시차 패턴 분류 강화 + Q3 카드에 fig2 PNG 사용
# ═══════════════════════════════════════════════════════════════════

def _classify_lag_pattern(t4):
    """시차별 유의성 패턴을 자동 분류해서 적절한 설명 텍스트 생성."""
    sig_strict = sorted([int(r['lag'].replace('M+', ''))
                         for r in t4 if r['sig'] in ['**', '***']])
    sig_weak_only = sorted([int(r['lag'].replace('M+', ''))
                            for r in t4 if r['sig'] == '*'])

    if not sig_strict:
        if sig_weak_only:
            return f"약유의 효과만 산발적으로 확인 (p<0.10, {len(sig_weak_only)}개 시차)"
        return "선행 설명 효과 미확인"

    runs = []
    current = [sig_strict[0]]
    for lag_i in sig_strict[1:]:
        if lag_i == current[-1] + 1:
            current.append(lag_i)
        else:
            runs.append(current)
            current = [lag_i]
    runs.append(current)

    n_runs = len(runs)

    if n_runs == 1:
        run = runs[0]
        if run[0] == 0:
            if run[-1] == 0:
                main_text = "동시 효과만 확인 (p<0.05)"
            else:
                main_text = (f"동시~{run[-1]}개월 선행 효과 확인 "
                             f"({len(run)}개 시차 연속, p<0.05)")
            tag = "본 연구의 핵심 검증"
        else:
            if len(run) == 1:
                main_text = f"M+{run[0]}에서 단발성 선행 효과 (p<0.05)"
            else:
                main_text = (f"M+{run[0]}~M+{run[-1]} 구간 선행 효과 "
                             f"({len(run)}개월 연속, p<0.05)")
            tag = "지연 반응 패턴"
    else:
        run_strs = [(f"M+{r[0]}~M+{r[-1]}" if len(r) > 1 else f"M+{r[0]}")
                    for r in runs]
        main_text = (f"분산형 선행 효과 ({', '.join(run_strs)}, "
                     f"총 {len(sig_strict)}개 시차, p<0.05)")
        tag = "다단계 반응 패턴"

    if sig_weak_only:
        weak_str = ', '.join(f'M+{l}' for l in sig_weak_only[:3])
        if len(sig_weak_only) > 3:
            weak_str += f" 외 {len(sig_weak_only) - 3}개"
        return f"{main_text} · {tag} (약유의: {weak_str})"
    return f"{main_text} · {tag}"


def _build_abstract(paper_data, sig_count):
    """초록 — FAQ Q1~Q4 형식의 연구 개요 인포그래픽.

    Q3 카드에서 인라인 SVG 대신 fig2_b64 (셀 9-12 PNG)를 직접 임베딩하여
    Ⅲ.4 나. 시차별 주가 반응과 동일한 그림 사용.
    """
    meta = paper_data['meta']
    t1   = paper_data['table1']
    t2   = paper_data['table2']
    t3   = paper_data['table3']
    t4   = paper_data['table4']
    kf   = paper_data['key_findings']

    # ── Q2 보조: 감성 분포 (%) ─────────────────────────
    _tot  = max(t1['pos_count'] + t1['neg_count'] + t1['neu_count'], 1)
    pos_p = round(t1['pos_count'] / _tot * 100, 1)
    neg_p = round(t1['neg_count'] / _tot * 100, 1)
    neu_p = round(t1['neu_count'] / _tot * 100, 1)

    # ── Q3: ΔR² 및 선행 시차 ────────────────────────────
    delta_r2 = round((t3['adj_r2_m2'] - t3['adj_r2_m1']) * 100, 2)
    f_pval   = t3['f_pval']

    # ★ 시차 패턴 자동 분류
    lead_text = _classify_lag_pattern(t4)

    # max_sig_lag (다른 곳에서 사용 가능하므로 별도 계산 유지)
    sig_lags_strict = [int(r['lag'].replace('M+', ''))
                       for r in t4 if r['sig'] in ['**', '***']]
    max_sig_lag = max(sig_lags_strict) if sig_lags_strict else 0

    # 시차 개수 (n)
    n = len(t4)

    # ── Q3 차트: 셀 9-12에서 만든 fig2_b64를 그대로 사용 ─────
    fig2_b64 = paper_data.get('figure2_b64', '')

    # ── Q4 Top 3 이슈 ───────────────────────────────────
    top3 = t2[:3] if len(t2) >= 3 else t2
    top_items_html = ''.join([
        f'''
<li>
  <span class="rank-num">{i+1}</span>
  <span class="rank-issue">{_esc(r["issue"])}</span>
  <span class="rank-count">{int(r["count"]):,}</span>
</li>'''
        for i, r in enumerate(top3)
    ])

    # ── 총 매칭 기사당 평균 이슈 수 ─────────────────────
    avg_per_news = t1.get('avg_per_news',
                          round(t1['esg_match'] / max(t1['esg_related'], 1), 1))

    # ── 유의수준 표기 ───────────────────────────────────
    if   f_pval < 0.001: sig_note = 'p &lt; 0.001'
    elif f_pval < 0.01:  sig_note = 'p &lt; 0.01'
    elif f_pval < 0.05:  sig_note = 'p &lt; 0.05'
    elif f_pval < 0.10:  sig_note = 'p &lt; 0.10'
    else:                sig_note = 'n.s.'

    # ── HTML 조립 ───────────────────────────────────────
    html = f'''
<section>
  <h2><span class="num">FAQ</span>연구 질문 &amp; 핵심 발견</h2>
  <div class="faq-section">

    <!-- Q1. 가능성 -->
    <div class="faq-row">
      <div class="faq-left">
        <div class="faq-q-badge">Q1</div>
        <div class="faq-q-content">
          <div class="faq-question">뉴스에서 ESG 이슈를 추출할 수 있는가?</div>
          <div class="faq-conclusion">가능</div>
        </div>
      </div>
      <div class="faq-right">
        <p>GPT-4o-mini 관련성 필터 → 임베딩 (3,072D) → SASB 26 이슈 매핑</p>
        <p>
          <strong>{t1["total_news"]:,}건</strong>
          <span class="muted">→</span>
          <strong>{t1["esg_match"]:,} 매칭</strong>
          <span class="muted">(기사당 {avg_per_news}개 이슈, 다중적 연결)</span>
        </p>
      </div>
    </div>

    <!-- Q2. 타당성 -->
    <div class="faq-row">
      <div class="faq-left">
        <div class="faq-q-badge">Q2</div>
        <div class="faq-q-content">
          <div class="faq-question">지수가 제대로 구축되었는가?</div>
          <div class="faq-conclusion">타당 <span class="muted" style="font-weight:400; color:var(--ink-3);">(face validity)</span></div>
        </div>
      </div>
      <div class="faq-right">
        <p>상위 이슈 = <strong>{_esc(top3[0]['issue']) if top3 else '—'}</strong> 등 산업 중대성과 일치</p>
        <p>
          감성 분포 <strong style="color:var(--pos);">긍 {pos_p}</strong>
          / <strong style="color:var(--neg);">부 {neg_p}</strong>
          / <span class="muted">중립 {neu_p}</span>
          <span class="muted">· 12개월 MA로 노이즈 제거</span>
        </p>
      </div>
    </div>

    <!-- Q3. 선행성 (핵심 강조) -->
    <div class="faq-row highlight">
      <div class="faq-left">
        <div class="faq-q-badge">Q3</div>
        <div class="faq-q-content">
          <div class="faq-question">재무요인을 통제해도 주가를 선행 설명하는가?</div>
          <div class="faq-conclusion">{_esc(lead_text)}</div>
        </div>
      </div>
      <div class="faq-right">
        <div class="faq-metrics">
          <div class="faq-metric-block">
            <div class="faq-metric-label">재무모형 → 재무+ESG 모형</div>
            <div class="faq-metric-value">
              Adj-R² {t3['adj_r2_m1']:.4f}
              <span class="muted">→</span>
              <span class="highlight-num">{t3['adj_r2_m2']:.4f}</span>
            </div>
            <div class="faq-metric-sub"><span class="delta">+{delta_r2}%p</span></div>
          </div>
          <div class="faq-metric-block">
            <div class="faq-metric-label">증분 설명력 F-검정</div>
            <div class="faq-metric-value">
              F = {t3['f_stat']:.4f}, <span class="highlight-num">p = {f_pval:.3f}</span>
            </div>
            <div class="faq-metric-sub muted">({sig_note})</div>
          </div>
        </div>
        <div class="faq-chart">
          <img src="data:image/png;base64,{fig2_b64}" alt="시차별 회귀계수 (M+0~M+{n-1})"
               style="width:100%; max-width:680px; height:auto; display:block; margin:8px 0; border-radius:4px;"/>
        </div>
      </div>
    </div>

    <!-- Q4. Top 3 이슈 -->
    <div class="faq-row">
      <div class="faq-left">
        <div class="faq-q-badge">Q4</div>
        <div class="faq-q-content">
          <div class="faq-question">주가에 영향을 주는 이슈는 무엇인가?</div>
          <div class="faq-conclusion">상위 3개 이슈 식별</div>
        </div>
      </div>
      <div class="faq-right">
        <ul class="faq-ranked-list">
          {top_items_html}
        </ul>
      </div>
    </div>

  </div>
</section>
'''

    # ── 기존 서술형 초록 유지 ───────────────────────────
    prose = f'''
<section style="margin-top:48px;">
  <h2><span class="num">초록</span>연구 요약</h2>
  <p style="font-size:14px; color:var(--ink-2); line-height:1.9;">
    본 연구는 {_esc(meta['company'])}({_esc(meta['stock_code'])})의
    {_esc(meta['period'])} 기간 ({meta['months']}개월) 뉴스 {t1['total_news']:,}건을
    SASB 26개 머티리얼리티 이슈에 임베딩 기반으로 매핑하여
    월별 ESG 감성지수를 구축하고, 재무요인(PCA)을 통제한 시차 회귀분석을 통해
    뉴스 감성의 주가 선행 예측력을 검증한다. Model 1(재무요인) 대비 Model 2(재무+ESG)의
    조정 R²는 {t3['adj_r2_m1']}에서 {t3['adj_r2_m2']}로 개선되었으며
    (ΔR² = +{delta_r2}%p, F-검정 p = {f_pval:.3f}),
    M+0~M+{n-1} {n}개 시차 중 {sig_count}개 시점에서 ESG 감성지수의 회귀계수가 통계적으로 유의하였다.
    결과는 공시 기반 정량 ESG 점수의 후행성 한계를 뉴스 텍스트 기반 실시간 감성지수로 보완하고,
    26개 이슈 단위 시차 구조를 통해 선행 지표로서의 유용성을 실증한다는 점에서 기여한다.
  </p>
</section>
'''

    return html + prose


print('✅ 11-3b: _build_abstract (FAQ + Q3에 fig2 PNG 적용) 재정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 신규 11-3c : _build_importance  (Ⅰ. 연구문제의 중요성)
# ═══════════════════════════════════════════════════════════════════
def _build_importance(paper_data):
    """Ⅰ. 연구문제의 중요성 — 배경 · 갭 · 질문."""
    meta = paper_data['meta']
    t1   = paper_data['table1']
    return f"""
<section><h2><span class="num">Ⅰ</span>연구문제의 중요성</h2>

<h3>배경 · ESG 정보의 투자 활용성</h3>
<p>
  ESG 정보는 기업의 장기 지속가능성과 리스크 관리를 평가하는 핵심 축으로 자리잡았으나,
  현행 정량 ESG 점수는 <strong>연간 공시 기반의 후행 지표</strong>라는 근본 한계를 지닌다.
  MSCI·Sustainalytics 등 주요 평가 점수는 분기 또는 연 단위로 갱신되어
  이슈 발생 시점과 점수 반영 시점 사이에 수 개월의 지체가 발생하며,
  이는 <strong>실시간 투자 의사결정과 리스크 관리</strong>에 활용하기 어려운 구조다.
</p>

<h3>갭 · 뉴스 텍스트의 미활용</h3>
<p>
  기업을 둘러싼 ESG 담론은 공시보다 먼저 뉴스·소셜미디어에서 형성된다.
  그럼에도 기존 연구는 대체로 정량 ESG 점수 중심의 사후적 상관관계에 치중하거나,
  뉴스를 단순 키워드 빈도로만 다루어
  <em>의미 수준의 이슈 연결</em>과 <em>산업별 중대성(materiality)</em>을 반영하지 못했다.
  긍정·부정 담론을 평균하여 단일 점수로 환원함으로써 신호를 희석하는 한계도 공통적이다.
</p>

<h3>본 연구의 질문</h3>
<p>본 연구는 다음 세 가지 질문을 실증한다.</p>
<ol style="margin: 8px 0 0 22px; color:var(--ink-2); line-height:1.9;">
  <li>SASB 머티리얼리티 맵의 <strong>26개 이슈 단위</strong>로 뉴스 감성을 세분화할 때,
      재무요인을 통제한 뒤에도 ESG 감성지수는 주가에 대해 <strong>증분 설명력</strong>을 가지는가?</li>
  <li>ESG 뉴스가 주가에 반영되는 <strong>시차(lag)</strong>는 몇 개월인가?
      정보 흡수와 지속성의 구조는 어떠한가?</li>
  <li>긍정·부정 담론을 분리할 때, 두 방향의 감성은 독립적인 담론 구조를 드러내는가?</li>
</ol>

<p style="font-size:13px; color:var(--ink-3); margin-top:20px; padding: 12px 16px;
          background:var(--bg-2); border-left:3px solid var(--accent);">
  <strong>분석 대상</strong> · {_esc(meta['company'])}({_esc(meta['stock_code'])})  ·
  <strong>기간</strong> · {_esc(meta['period'])} ({meta['months']}개월) ·
  <strong>뉴스</strong> · {t1['total_news']:,}건
</p>
</section>
"""

print('✅ _build_importance 정의 완료')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 교체 11-4 : _build_originality  (Ⅱ. 방법의 독창성, 3축 + 파이프라인)
# ═══════════════════════════════════════════════════════════════════
def _build_originality():
    """Ⅱ. 방법의 독창성 — 3개 축 + 분석 파이프라인."""
    return """
<section><h2><span class="num">Ⅱ</span>방법의 독창성</h2>

<p style="margin-bottom: 20px;">
  기존 ESG–주가 실증 연구와 본 연구의 차별점은 다음 세 축에 있다.
  이슈 정의 축(SASB 26 × 임베딩), 인과 식별 축(재무요인 통제 + 시차), 담론 구조 축(긍·부 분리)이다.
</p>

<div class="originality-grid">

  <div class="originality-item">
    <div class="originality-num">1</div>
    <div class="originality-title">SASB 26이슈 × 임베딩 의미 매칭</div>
    <div class="originality-desc">
      SASB 머티리얼리티 맵의 26개 이슈를 기준 축으로 삼아,
      OpenAI <code>text-embedding-3-large</code>(3,072차원) 기반
      코사인 유사도로 뉴스를 매핑. 키워드 매칭이 놓치는
      <strong>맥락·의미 수준</strong>의 이슈 연결을 확보하고,
      산업 중대성에 부합하는 세분화된 감성지수를 구축.
    </div>
  </div>

  <div class="originality-item">
    <div class="originality-num">2</div>
    <div class="originality-title">재무요인 통제 + 시차 구조 분석</div>
    <div class="originality-desc">
      PCA로 추출한 재무요인(PC1, PC2)을 통제한 상태에서
      ESG 감성지수의 <strong>증분 설명력</strong>을 F-검정으로 검증.
      단순 상관이 아닌 <em>조건부 예측력</em>을 규명하고,
      M+0~M+12의 13개 시차에 걸쳐 회귀를 반복하여
      정보 흡수 속도와 지속성을 실증.
    </div>
  </div>

  <div class="originality-item">
    <div class="originality-num">3</div>
    <div class="originality-title">긍·부 담론 분리 + 노이즈 평활</div>
    <div class="originality-desc">
      단일 종합 점수가 긍·부정을 산술 평균해 신호를 희석하는 한계를 넘어,
      긍정·부정 담론을 <strong>K-means 클러스터링</strong>으로 분리.
      12개월 이동평균으로 단기 이벤트 소음과 구조적 ESG 추세를 분리하여
      해석 가능한 감성 시계열을 산출.
    </div>
  </div>

</div>

<h3 style="margin-top: 36px;">분석 파이프라인</h3>
<div class="method-timeline">
  <div class="method-step"><div class="step-num">1</div><div class="step-title">뉴스 수집</div><div class="step-desc">GPT-4o-mini<br>관련성 필터링</div></div>
  <div class="method-step"><div class="step-num">2</div><div class="step-title">임베딩 매핑</div><div class="step-desc">text-embedding-<br>3-large (3,072D)</div></div>
  <div class="method-step"><div class="step-num">3</div><div class="step-title">감성 분류</div><div class="step-desc">GPT 기반<br>3범주 분류</div></div>
  <div class="method-step"><div class="step-num">4</div><div class="step-title">감성지수 산출</div><div class="step-desc">유사도 가중<br>12개월 MA</div></div>
  <div class="method-step"><div class="step-num">5</div><div class="step-title">회귀분석</div><div class="step-desc">PCA + 시차<br>Model 1 vs 2</div></div>
</div>
<p style="font-size:13px; color:var(--ink-3); margin-top:10px;">
  <strong>Model 1</strong> : 재무요인만 (PC1, PC2) ·
  <strong>Model 2</strong> : Model 1 + Overall Sentiment Index ·
  시차 M+0 ~ M+12 반복 추정
</p>
</section>
"""

print('✅ _build_originality 정의 완료 · 3개 축 + 파이프라인 흡수')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-5. Ⅲ.1 (표 1) + Ⅲ.2 (표 2) + Ⅲ.3 (그림 1) 빌더
# ═══════════════════════════════════════════════════════════════════
def _build_table1(t1):
    pos_pct = round(t1['pos_count']/max(t1['sent_valid'],1)*100)
    neg_pct = round(t1['neg_count']/max(t1['sent_valid'],1)*100)
    return f"""
<section><h2><span class="num">Ⅲ.1</span>데이터 수집 및 감성 분류</h2>
<div class="side-by-side">
<div class="table-block"><div class="table-caption"><span class="tag">표 1</span>수집 및 분류 결과</div>
<table><tbody>
<tr><td>수집 기간</td><td class="num-col">{_esc(t1['period'])}</td></tr>
<tr><td>총 수집 기사</td><td class="num-col">{t1['total_news']:,}건</td></tr>
<tr><td>ESG 관련 기사</td><td class="num-col">{t1['esg_related']:,}건 ({t1['esg_related_pct']}%)</td></tr>
<tr><td>감성 유효 기사</td><td class="num-col">{t1['sent_valid']:,}건</td></tr>
<tr><td>긍 / 부 / 중립</td><td class="num-col">{t1['pos_count']:,} / {t1['neg_count']:,} / {t1['neu_count']:,}</td></tr>
<tr><td>ESG 매칭</td><td class="num-col">{t1['esg_match']:,}건 ({t1['avg_per_news']}/기사)</td></tr>
</tbody></table></div>
<div class="interpret">긍정 비율이 약 <strong>{pos_pct}%</strong>, 부정도 <strong>{neg_pct}%</strong> 수준으로 ESG 리스크 감시 기능이 작동. 1건당 평균 {t1['avg_per_news']}개의 ESG 이슈에 매핑되어 다층적으로 얽혀 있음.</div>
</div></section>
"""


def _build_table2(t2):
    rows = ''.join([
        f'<tr><td><strong>{r["rank"]}</strong></td><td>{_esc(r["issue"])}</td><td class="num-col"><strong>{r["count"]:,}</strong></td></tr>'
        if r['rank'] <= 3 else
        f'<tr><td>{r["rank"]}</td><td>{_esc(r["issue"])}</td><td class="num-col">{r["count"]:,}</td></tr>'
        for r in t2
    ])
    return f"""
<section><h2><span class="num">Ⅲ.2</span>상위 ESG 이슈 식별</h2>
<div class="table-block"><div class="table-caption"><span class="tag">표 2</span>SASB 기준 상위 ESG 이슈</div>
<table><thead><tr><th style="width:50px">순위</th><th>ESG 이슈</th><th class="num-col">매칭 건수</th></tr></thead>
<tbody>{rows}</tbody></table>
<div class="table-note">상위 3개 이슈에 대해 Ⅲ.5에서 의미 연결망 기반 주제 분석을 수행함.</div>
</div></section>
"""


def _build_figure1(fig1_b64):
    return f"""
<section><h2><span class="num">Ⅲ.3</span>ESG 감성지수의 시계열 변화</h2>
<div class="figure-block"><div class="figure-caption"><span class="tag">그림 1</span>TOP5 ESG 이슈별 12개월 이동평균 감성지수</div>
<img src="data:image/png;base64,{fig1_b64}" alt="그림 1"/></div>
<p style="font-size:13.5px;">이슈별 감성지수의 차별적 패턴이 관찰됨. 12개월 이동평균을 통해 단기 변동을 제거한 장기 추세를 확인할 수 있음.</p>
</section>
"""


print('✅ 11-5: _build_table1(), _build_table2(), _build_figure1() 정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-6. Ⅲ.4 (표 3 + 그림 2 + 표 4 + 표 5) 빌더
# ═══════════════════════════════════════════════════════════════════
def _build_table3(t3):
    sig_level = '1%' if t3['f_pval'] < 0.01 else '5%' if t3['f_pval'] < 0.05 else '10%'
    return f"""
<section><h2><span class="num">Ⅲ.4</span>ESG 감성지수의 주가 반응</h2>
<h3>가. 회귀모형 비교</h3>
<div class="side-by-side">
<div class="table-block"><div class="table-caption"><span class="tag">표 3</span>Model 1 vs Model 2</div>
<table><thead><tr><th>지표</th><th class="num-col">M1 (재무)</th><th class="num-col">M2 (재무+ESG)</th></tr></thead>
<tbody>
<tr><td>Adjusted R²</td><td class="num-col">{t3['adj_r2_m1']}</td><td class="num-col sig">{t3['adj_r2_m2']}</td></tr>
<tr><td>MSE</td><td class="num-col">{t3['mse_m1']}</td><td class="num-col sig">{t3['mse_m2']}</td></tr>
<tr><td>AIC</td><td class="num-col">{t3['aic_m1']}</td><td class="num-col sig">{t3['aic_m2']}</td></tr>
<tr><td>BIC</td><td class="num-col">{t3['bic_m1']}</td><td class="num-col sig">{t3['bic_m2']}</td></tr>
<tr><td>F-stat</td><td class="num-col">—</td><td class="num-col sig">{t3['f_stat']}</td></tr>
<tr><td>p-value</td><td class="num-col">—</td><td class="num-col sig">{t3['f_pval']:.4f}</td></tr>
</tbody></table></div>
<div class="interpret">Model 2가 <strong>유의수준 {sig_level}</strong>에서 우월. MSE Ratio {t3['mse_ratio']}배로 예측 오차가 실질적으로 감소.</div>
</div>
"""


def _build_figure2_table4(fig2_b64, t4):
    rows = ''.join([
        f'<tr><td>{r["lag"]}</td><td class="num-col">{r["coef"]}</td><td class="num-col">{r["pval"]}</td><td class="num-col">{r["adj_r2"]}</td><td>{r["sig"]}</td></tr>'
        for r in t4
    ])
    return f"""
<h3>나. 시차별 주가 반응</h3>
<div class="figure-block"><div class="figure-caption"><span class="tag">그림 2</span>시차별 표준화 회귀계수 및 유의확률</div>
<img src="data:image/png;base64,{fig2_b64}" alt="그림 2"/></div>
<div class="table-block"><div class="table-caption"><span class="tag">표 4</span>시차별 ESG 감성지수의 회귀계수</div>
<table><thead><tr><th>시차</th><th class="num-col">회귀계수</th><th class="num-col">p-value</th><th class="num-col">Adj-R²</th><th>유의성</th></tr></thead>
<tbody>{rows}</tbody></table>
<div class="table-note">*** p&lt;0.01, ** p&lt;0.05, * p&lt;0.10, n.s. = not significant</div>
</div>
"""


def _build_table5(t5):
    rows = ''.join([
        f'<tr><td>{_esc(r["name"])}</td><td>{_esc(r["stat"])}</td><td class="{"sig" if r["pass"] else "sig-ns"}">{_esc(r["result"])}</td></tr>'
        for r in t5
    ])
    return f"""
<h3>다. 회귀모형 진단</h3>
<div class="table-block"><div class="table-caption"><span class="tag">표 5</span>Model 2 진단 검정</div>
<table><thead><tr><th>검정</th><th>통계량 / p-value</th><th>해석</th></tr></thead>
<tbody>{rows}</tbody></table></div>
</section>
"""


print('✅ 11-6: _build_table3(), _build_figure2_table4(), _build_table5() 정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-6.5. 그림 3 (시차별 산점도 격자) HTML 빌더
# ═══════════════════════════════════════════════════════════════════

def _build_figure3(fig3_b64, summary):
    """그림 3 (시차별 ESG ↔ 주가 산점도 격자) + 시차별 상관 요약 표 HTML 블록"""

    # 시차별 행 생성 (표 6용)
    rows = ''
    for r in summary['lag_results']:
        sig_label_map = {
            'high': '** p&lt;0.05',
            'weak': '* p&lt;0.10',
            'none': 'n.s.',
        }
        sig_class_map = {
            'high': 'sig',
            'weak': 'sig',
            'none': 'sig-ns',
        }
        sign_color = '#DC2626' if r['sign'] == '-' else '#1E3A8A'
        rows += (
            f'<tr><td>M+{r["lag"]}</td>'
            f'<td class="num-col" style="color:{sign_color};font-weight:600">{r["r"]:+.4f}</td>'
            f'<td class="num-col">{r["p"]:.4f}</td>'
            f'<td class="num-col">{r["n"]}</td>'
            f'<td class="{sig_class_map[r["sig"]]}">{sig_label_map[r["sig"]]}</td></tr>'
        )

    # 부호 전환 텍스트
    if summary['sign_change_lags']:
        change_lags = ', '.join(f'M+{l}' for l in summary['sign_change_lags'])
        change_msg = (
            f'<strong>부호 전환 시점:</strong> {change_lags}에서 '
            f'ESG와 주가의 관계 방향이 바뀜.'
        )
    else:
        change_msg = '<strong>부호 일관성:</strong> 모든 시차에서 상관관계의 부호가 일관됨.'

    # 양/음 시차 요약
    pos_text = (', '.join(f'M+{l}' for l in summary['pos_lags'])
                if summary['pos_lags'] else '없음')
    neg_text = (', '.join(f'M+{l}' for l in summary['neg_lags'])
                if summary['neg_lags'] else '없음')

    return f"""
<h3>다. 시차별 단순 상관관계 (산점도 격자)</h3>
<p style="font-size:13.5px;margin-bottom:16px;">
  재무요인을 통제하지 않은 ESG 감성지수(t)와 주가(t+k)의 단순 상관관계를 시차별로 시각화함.
  Pearson 상관계수의 부호 변화를 통해 ESG의 즉시 효과와 누적 효과의 차이를 확인할 수 있음.
</p>
<div class="figure-block">
  <div class="figure-caption"><span class="tag">그림 3</span>시차별 ESG ↔ 주가 산점도 격자 (M+0 ~ M+12)</div>
  <img src="data:image/png;base64,{fig3_b64}" alt="그림 3"/>
</div>
<div class="table-block">
  <div class="table-caption"><span class="tag">표 6</span>시차별 Pearson 상관계수</div>
  <table>
    <thead>
      <tr><th>시차</th><th class="num-col">r</th><th class="num-col">p-value</th><th class="num-col">n</th><th>유의성</th></tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
  <div class="table-note">
    {change_msg}<br>
    <strong>유의(p&lt;0.10) 양의 상관 시차:</strong> {pos_text} &nbsp;|&nbsp;
    <strong>유의(p&lt;0.10) 음의 상관 시차:</strong> {neg_text}<br>
    <strong>최고 절댓값 상관 시차:</strong> M+{summary['best_lag']}
    (r={summary['best_r']:+.4f}, p={summary['best_p']:.4f})<br>
    *** p&lt;0.01, ** p&lt;0.05, * p&lt;0.10, n.s. = not significant
  </div>
</div>
<div class="interpret">
  ※ 본 분석(그림 3)은 <strong>재무요인을 통제하지 않은 단순 상관</strong>으로,
  Ⅲ.4의 회귀분석(그림 2)과 함께 보면 ESG의 추가 설명력과 시차별 패턴의 견고성을 교차 검증할 수 있음.
</div>
"""

print('✅ 11-6.5: _build_figure3() 정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-7. Ⅲ.5 주제 분석 빌더 (TOP3 + 완전형 + matplotlib PNG)
# ═══════════════════════════════════════════════════════════════════
def _build_topic_analysis(all_results, top_n=3):
    """주제 분석 섹션 — TOP3 이슈 × 긍·부정 × 4 클러스터"""
    issues = list(all_results.keys())[:top_n]
    fig_num = 4  # 그림 1~4는 이미 사용됨

    parts = ['<section><h2><span class="num">Ⅲ.5</span>주제 분석 — 의미 연결망</h2>']
    parts.append(
        f'<p style="font-size:13.5px; margin-bottom:24px;">'
        f'매칭 건수 상위 {len(issues)}개 ESG 이슈를 대상으로 의미 연결망 분석을 수행한다. '
        f'각 이슈에 대해 긍정·부정 감성 기사를 분리하여 키워드 동시 출현 빈도 기반 네트워크를 구성하고, '
        f'K-means 군집화를 통해 주제 클러스터를 도출한다.</p>'
    )

    for rank, issue in enumerate(issues, 1):
        sents = all_results[issue]
        if not sents:
            continue

        pos_data = sents.get('긍정', {})
        neg_data = sents.get('부정', {})
        total = pos_data.get('total', 0) + neg_data.get('total', 0)

        parts.append(f'''
<div class="issue-block">
  <div class="issue-header">
    <div class="issue-rank">{rank}</div>
    <div class="issue-info">
      <div class="issue-name">{_esc(issue)}</div>
      <div class="issue-meta">
        <span><strong>{total:,}건</strong> 매칭</span>
        <span>긍정 <strong>{pos_data.get('total', 0):,}건</strong> · 부정 <strong>{neg_data.get('total', 0):,}건</strong></span>
        <span>SASB 카테고리</span>
      </div>
    </div>
  </div>
  <div class="issue-body">''')

        pos_narr_text = ''
        neg_narr_text = ''
        for sn, data, css_cls in [('긍정', pos_data, 'pos'), ('부정', neg_data, 'neg')]:
            if not data:
                continue
            fig_num += 1
            narrative = data.get('narrative', '')
            if sn == '긍정':
                pos_narr_text = narrative
            else:
                neg_narr_text = narrative

            cluster_html = ''
            clusters = data.get('clusters', [])
            for ci, c in enumerate(clusters[:4], 1):
                size = c.get('size', 0)
                ratio = round(size / max(data.get('total', 1), 1) * 100, 1)
                theme = c.get('theme', '')
                content = c.get('content', '')

                keywords = c.get('keywords', [])
                if not keywords:
                    keywords = [k.strip() for k in theme.split(',')[:15]]
                top_kws = keywords[:5]
                rest_kws = keywords[5:15]
                kw_chips = ''.join([f'<span class="kw-chip top">{_esc(k)}</span>' for k in top_kws])
                kw_chips += ''.join([f'<span class="kw-chip">{_esc(k)}</span>' for k in rest_kws])

                cluster_html += f'''
<div class="cluster-card c{ci}">
  <div class="cluster-head"><span class="cluster-id">CLUSTER {ci}</span><span class="cluster-size">{size:,}건 ({ratio}%)</span></div>
  <div class="cluster-name">{_esc(theme)}</div>
  <div class="cluster-keywords">{kw_chips}</div>
  <div class="cluster-desc">{_esc(content)}</div>
</div>'''

            img_html = ''
            if data.get('img'):
                img_html = f'''
<div class="figure-block" style="background:#fff;">
  <div class="figure-caption"><span class="tag">그림 {fig_num}</span>{sn} 감성 의미 연결망 ({_esc(issue[:40])})</div>
  <div class="network-figure"><img src="data:image/png;base64,{data['img']}" alt="그림 {fig_num}"/></div>
</div>'''

            parts.append(f'''
<div class="sentiment-section {css_cls}">
  <div class="sentiment-header">
    <span class="sentiment-badge {css_cls}">{sn} 감성</span>
    <span class="sentiment-count">{sn} 기사 <strong>{data.get('total', 0):,}건</strong> · {len(clusters)}개 클러스터</span>
  </div>
  {img_html}
  <div class="narrative-box">
    <div class="narr-label">학술체 종합 서술 (narrative)</div>
    {_esc(narrative)}
  </div>
  <div class="clusters-grid">{cluster_html}</div>
</div>''')

        if pos_narr_text and neg_narr_text:
            print(f'  [Ⅲ.5] {issue} 비교 단락 생성 중...')
            comparison = gen_comparison(issue, pos_narr_text, neg_narr_text)
            parts.append(f'''
<div class="comparison-box">
  <div class="comparison-label">긍·부정 담론 구조 비교</div>
  <div class="comparison-text">{_esc(comparison)}</div>
</div>''')

        parts.append('</div></div>')

    parts.append('</section>')
    return '\n'.join(parts)


print('✅ 11-7: _build_topic_analysis() 정의 완료')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-8. Ⅳ. 정책적·실무적 제언 + 한계 빌더
# ═══════════════════════════════════════════════════════════════════
def _build_conclusion(t2):
    top1 = t2[0]['issue'] if t2 else 'N/A'
    return f"""
<section><h2><span class="num">Ⅳ</span>정책적·실무적 제언</h2>
<div class="insights">
<div class="insight-card"><div class="insight-num">1</div>
<div class="insight-title">규제·법무 영역 선제 대응</div>
<div class="insight-body">감성지수가 낮은 규제 관련 이슈에 대해 사후 해명이 아닌 <strong>분기별 IR에 법무·규제 대응</strong> 정기 포함 필요.</div></div>
<div class="insight-card"><div class="insight-num">2</div>
<div class="insight-title">상위 이슈 평판 관리</div>
<div class="insight-body"><strong>{_esc(top1[:40])}</strong> 이슈가 매칭 1위. 긍정 담론 강화와 부정 담론 조기 진화 양면 전략 필요.</div></div>
<div class="insight-card"><div class="insight-num">3</div>
<div class="insight-title">감성 모니터링 상시화</div>
<div class="insight-body">ESG 감성은 <strong>수개월 선행</strong>해 주가에 영향. 월 단위 대시보드 구축으로 조기 포착 및 IR·경영전략 반영.</div></div>
</div></section>

<section><div class="limits"><div class="limits-title">Limitations &amp; Future Work</div><ul>
<li>단일 기업 분석으로 일반화에 제약 — 동일 산업군 확장 필요</li>
<li>이분산성·잔차 자기상관 가능성 — Newey-West 또는 GARCH 적용 검토</li>
<li>SASB 단일 프레임워크 기준 — ISSB·GRI 등 비교 분석 확장 가능</li>
</ul></div></section>
"""


print('✅ 11-8: _build_conclusion() 정의 완료')

In [ ]:
def gen_paper_html(paper_data, all_results, top_n_for_topic=3):
    """논문 형식 HTML — 4단 구조 (KPI·FAQ·Discoveries 완전 제거)."""
    css = build_paper_css()
    meta = paper_data['meta']
    t1   = paper_data['table1']
    t2   = paper_data['table2']
    t3   = paper_data['table3']
    t4   = paper_data['table4']
    t5   = paper_data['table5']
    kf   = paper_data['key_findings']
    sig_count = sum(1 for r in t4 if r['sig'] != 'n.s.')

    body = (
        _build_hero(meta, kf, t1) +
        '<div class="container">' +

        # 초록
        _build_abstract(paper_data, sig_count) +

        # Ⅰ. 연구문제의 중요성
        _build_importance(paper_data) +

        # Ⅱ. 방법의 독창성
        _build_originality() +

        # Ⅲ. 결과의 가치
        '<section><h2><span class="num">Ⅲ</span>결과의 가치</h2>'
        '<p style="margin-bottom:24px;">'
        '데이터 수집 결과와 상위 이슈 식별에서 시작하여, 이슈별 감성지수의 시계열 추이, '
        'Model 1 대비 Model 2 의 증분 설명력, 시차 회귀 구조, 시차별 단순 상관관계, '
        '주제 분석을 통한 담론 분해, 회귀 진단까지 차례로 제시한다.'
        '</p>' +

        _build_table1(t1) +
        _build_table2(t2) +
        _build_figure1(paper_data['figure1_b64']) +
        _build_table3(t3) +
        _build_figure2_table4(paper_data['figure2_b64'], t4) +
        _build_figure3(paper_data['figure3_b64'],                # ★ 추가
                       paper_data['figure3_summary']) +
        _build_topic_analysis(all_results, top_n=top_n_for_topic) +
        _build_table5(t5) +

        '</section>' +  # ← Ⅲ 섹션 닫기

        # Ⅳ 정책 제언 + 한계
        _build_conclusion(t2) +

        # ❌ _build_discoveries 제거됨
        # _build_discoveries(paper_data, all_results) +

        '</div>' +

        f'<footer><div class="container">'
        f'ESG News Impact Analysis · {_esc(meta["company"])} ({_esc(meta["stock_code"])}) '
        f'· Analysis period {_esc(meta["period"])}</div></footer>'
    )

    html = (
        '<!DOCTYPE html><html lang="ko"><head>'
        '<meta charset="UTF-8">'
        '<meta name="viewport" content="width=device-width,initial-scale=1">'
        f'<title>{_esc(meta["company"])} ESG 분석 리포트</title>'
        f'<style>{css}</style></head><body>'
        + body +
        '</body></html>'
    )
    return html


print('✅ gen_paper_html 최종 (그림 3 추가, KPI·FAQ·Discoveries 모두 제거)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11-10. HTML 생성 + 파일 저장 + 미리보기
# ═══════════════════════════════════════════════════════════════════
print('=' * 60)
print('논문 형식 HTML v2 생성 시작')
print('=' * 60)

# 사전 점검
assert 'paper_data' in dir(), '❌ paper_data 없음. 섹션 9-11을 먼저 실행하세요.'
assert 'all_results' in dir(), '❌ all_results 없음. 섹션 10을 먼저 실행하세요.'
print(f'✅ 입력 데이터 확인 완료')
print(f'   paper_data : 표 {sum(1 for k in paper_data if k.startswith("table"))}개 + 그림 2개')
print(f'   all_results: {len(all_results)}개 이슈')

# HTML 생성
html_output = gen_paper_html(paper_data, all_results, top_n_for_topic=3)

# 파일명
fname = f"{SEARCH_KEYWORD}_ESG_v2_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.html"
with open(fname, 'w', encoding='utf-8') as f:
    f.write(html_output)

print(f'\n✅ HTML 저장 완료: {fname}')
print(f'   크기: {len(html_output):,} chars ({len(html_output)/1024:.1f} KB)')

# 노트북 내 미리보기
display(HTML(html_output))

In [ ]:
# ─── 셀 식별자 (자동 추가) ─────────────────────────
print()
print('=' * 70)
print('[셀 38]  섹션 11-2 : GitHub 업로드')
print('=' * 70)

token = GITHUB_TOKEN  # ✅ 변수명 연결

# GitHub 업로드
from github import Github, Auth, GithubException
from urllib.parse import quote

token = str(GITHUB_TOKEN).strip()
owner = GITHUB_USERNAME
repo_name = GITHUB_REPO

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

api_base = "https://api.github.com"

if not token or not owner or not repo_name:
    print("실패: GITHUB_TOKEN, GITHUB_USERNAME, GITHUB_REPO 값을 확인하세요.")

else:
    try:
        # ------------------------------------------------------------
        # 1. 저장소 확인
        # ------------------------------------------------------------
        repo_url = f"{api_base}/repos/{owner}/{repo_name}"
        r = requests.get(repo_url, headers=headers)

        # ------------------------------------------------------------
        # 2. 저장소가 없으면 생성
        # ------------------------------------------------------------
        if r.status_code == 404:
            print("저장소가 없어 새로 생성합니다.")

            create_repo_url = f"{api_base}/user/repos"
            payload = {
                "name": repo_name,
                "private": False,
                "auto_init": True
            }

            cr = requests.post(create_repo_url, headers=headers, json=payload)

            if cr.status_code not in [201, 202]:
                raise Exception(f"저장소 생성 실패: {cr.status_code} {cr.text}")

            time.sleep(5)
            r = requests.get(repo_url, headers=headers)

        if r.status_code != 200:
            raise Exception(f"저장소 접근 실패: {r.status_code} {r.text}")

        repo_info = r.json()
        branch = repo_info.get("default_branch") or "main"

        print(f"저장소 확인 완료: {owner}/{repo_name}")
        print(f"기본 브랜치: {branch}")

        # ------------------------------------------------------------
        # 3. 긴 파일명 HTML 하나만 업로드
        # ------------------------------------------------------------
        upload_path = f"{GITHUB_FOLDER}/{fname}"
        encoded_path = quote(upload_path, safe="/")
        contents_url = f"{api_base}/repos/{owner}/{repo_name}/contents/{encoded_path}"

        gr = requests.get(
            contents_url,
            headers=headers,
            params={"ref": branch}
        )

        payload = {
            "message": f"Update {fname}",
            "content": base64.b64encode(html_output.encode("utf-8")).decode("utf-8"),
            "branch": branch
        }

        if gr.status_code == 200:
            payload["sha"] = gr.json()["sha"]
            action = "Update"
        elif gr.status_code == 404:
            action = "Add"
        else:
            raise Exception(f"{upload_path} 조회 실패: {gr.status_code} {gr.text}")

        pr = requests.put(contents_url, headers=headers, json=payload)

        if pr.status_code not in [200, 201]:
            raise Exception(f"{upload_path} 업로드 실패: {pr.status_code} {pr.text}")

        print(f"{action} 완료: {upload_path}")

        # ------------------------------------------------------------
        # 4. GitHub Pages 활성화 또는 업데이트
        # ------------------------------------------------------------
        pages_url_api = f"{api_base}/repos/{owner}/{repo_name}/pages"

        pages_payload = {
            "source": {
                "branch": branch,
                "path": "/"
            }
        }

        pg = requests.get(pages_url_api, headers=headers)

        if pg.status_code == 200:
            up = requests.put(pages_url_api, headers=headers, json=pages_payload)

            if up.status_code in [200, 204]:
                print("GitHub Pages 설정 업데이트 완료")
            else:
                print(f"Pages 업데이트 응답: {up.status_code} {up.text}")

        elif pg.status_code == 404:
            cp = requests.post(pages_url_api, headers=headers, json=pages_payload)

            if cp.status_code in [201, 202]:
                print("GitHub Pages 활성화 완료")
            else:
                print(f"Pages 활성화 응답: {cp.status_code} {cp.text}")

        else:
            print(f"Pages 상태 확인 응답: {pg.status_code} {pg.text}")

        # ------------------------------------------------------------
        # 5. 최종 긴 URL 출력
        # ------------------------------------------------------------
        final_url = f"https://{owner}.github.io/{repo_name}/{quote(upload_path, safe='/')}"

        print()
        print("✅ 완료")
        print(f"접속 URL: {final_url}")
        print()
        print("※ GitHub Pages 반영에는 보통 1~3분 정도 걸릴 수 있습니다.")

    except Exception as e:
        print(f"실패: {e}")